[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/protosome/convergent_overlapping_gene_prediction/blob/main/codon_overlap.ipynb)

In [ ]:
#@title ###**Load dependencies**.


import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import torch
import torch.nn as nn
import tensorflow as tf
import numpy as np
import pandas as pd
import json
from Bio import pairwise2
from Bio.Seq import Seq
from Bio.Seq import CodonTable
import itertools as it
import pandas as pd
import math
import random
import pickle
from protsub_matrix import prot_sub_matrix, calculate_protsub_similarity
from blosum62_matrix import blosum62_matrix, calculate_blosum62_similarity
from running_s4pred import predict_secondary_structure # This loads the s4pred function to run as a subprocess, outputting only the structure prediction sequence
from running_s4pred_batch import predict_secondary_structure_batch # This loads the s4pred function to run as a subprocess, outputting only the structure prediction sequence
from transformer_encoder_model import TransformerModel, SinusoidalPositionalEncoding, TransformerBlock
import time
import openpyxl
import time, math
import torch.nn as nn
import torch.nn.functional as F
from typing import List, Tuple, Optional
from skimage.metrics import structural_similarity as ssim

In [ ]:
#@title ###**Additional loading**.


# Identify available GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)
torch.cuda.get_device_name(0)

# Tokenization and text vectorization
max_length = 315  # max len of the overlap
vocab_size = 27

# Define a function to load the tokenizer
def load_tokenizer(filename):
    with open(filename, 'rb') as file:
        tokenizer = pickle.load(file)
    return tokenizer

concat_tokenizer = load_tokenizer('/home/jason/outputdir/python_projects/concat_tokenizer.pkl')
overlap_tokenizer = load_tokenizer('/home/jason/outputdir/python_projects/overlap_tokenizer.pkl')

# Function to tokenize sentences using TensorFlow's tokenizer
def tokenize(sentences):
    tokenizer = tf.keras.preprocessing.text.Tokenizer(num_words=vocab_size, filters='')
    tokenizer.fit_on_texts(sentences)  # Fit the tokenizer on the combined texts
    return tokenizer

# Function to vectorize sentences using the fitted tokenizer
def vectorize(tokenizer, sentences):
    seqs = tokenizer.texts_to_sequences(sentences)  # Convert texts to sequences of integers
    return tf.keras.preprocessing.sequence.pad_sequences(seqs, maxlen=max_length, padding='post')  # Pad sequences


In [ ]:
#@title ###**Load helper functions**.

####################################################################################
### Function to predict the overlapping dna sequence for two aa's
####################################################################################

# Codons and their frequencies for each amino acid based on the E. coli table
back_translation_code_with_all_options = {
    'A': [('GCG', 0.27), ('GCT', 0.26), ('GCC', 0.26), ('GCA', 0.21)],
    'C': [('TGC', 0.53), ('TGT', 0.47)],
    'D': [('GAT', 0.63), ('GAC', 0.37)],
    'E': [('GAA', 0.68), ('GAG', 0.32)],
    'F': [('TTT', 0.58), ('TTC', 0.42)],
    'G': [('GGC', 0.35), ('GGT', 0.32), ('GGG', 0.25), ('GGA', 0.08)],
    'H': [('CAT', 0.56), ('CAC', 0.44)],
    'I': [('ATT', 0.48), ('ATC', 0.39), ('ATA', 0.14)],
    'K': [('AAA', 0.74), ('AAG', 0.26)],
    'L': [('CTG', 0.43), ('CTT', 0.13), ('CTC', 0.13), ('TTA', 0.14), ('CTA', 0.07), ('TTG', 0.13)],
    'M': [('ATG', 1.00)],
    'N': [('AAC', 0.60), ('AAT', 0.40)],
    'P': [('CCG', 0.52), ('CCA', 0.19), ('CCT', 0.16), ('CCC', 0.13)],
    'Q': [('CAG', 0.66), ('CAA', 0.34)],
    'R': [('CGT', 0.36), ('CGC', 0.36), ('CGG', 0.11), ('AGA', 0.08), ('AGG', 0.05), ('CGA', 0.04)],
    'S': [('AGC', 0.24), ('TCC', 0.24), ('TCT', 0.17), ('TCG', 0.15), ('TCA', 0.14), ('AGT', 0.15)],
    'T': [('ACC', 0.36), ('ACA', 0.28), ('ACG', 0.25), ('ACT', 0.11)],
    'V': [('GTG', 0.46), ('GTT', 0.28), ('GTC', 0.15), ('GTA', 0.11)],
    'W': [('TGG', 1.00)],
    'Y': [('TAT', 0.59), ('TAC', 0.41)],
    '*': [('TAA', 0.61), ('TGA', 0.30), ('TAG', 0.09)]
}

def predict_overlapping_sequence(trained_model, aa_sequences, concat_tokenizer, overlap_tokenizer):
    
    def translate_to_dna_with_all_options(aa_sequence: str) -> list:
    
        def choose_codon_based_on_frequency(codons):

            # Extract codon names and their frequencies
            codon_names = [codon for codon, _ in codons]
            codon_freqs = [freq for _, freq in codons]
            
            # Normalize the frequencies to ensure they sum up to 1
            total_frequency = sum(codon_freqs)
            normalized_freqs = [freq/total_frequency for freq in codon_freqs]
            
            # Randomly select a codon based on the frequency distribution
            chosen_codon = np.random.choice(codon_names, p = normalized_freqs)
            
            return [chosen_codon]
    
        # For each amino acid in the sequence, choose a codon based on its frequency
        list_of_list_of_codons = [choose_codon_based_on_frequency(back_translation_code_with_all_options[aa]) for aa in aa_sequence]
        
        # Combine the codons to form the nucleotide sequence
        list_of_combinations = [''.join(combination) for combination in it.product(*list_of_list_of_codons)]
        
        return list_of_combinations

    def translate(model, concat_tokenizer, overlap_tokenizer, text, device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')):
        # Set the model to evaluation mode
        #model.train()

        #model.eval()
    
        vectorized_input = vectorize(concat_tokenizer, [text])
        vectorized_input = torch.tensor(vectorized_input, dtype=torch.long).to(device)
    
        # Perform inference
        # with torch.no_grad():
        #     output = model(vectorized_input)

        # Perform inference
        with torch.inference_mode():
            output = model(vectorized_input)
    
        # Get the predicted classes
        _, predicted_classes = torch.max(output, dim=-1)
        predicted_classes = predicted_classes.cpu().numpy()
    
        # Function to reverse the predicted output back into text
        def decode_predictions(predicted_classes, tokenizer):
            # Get the index to word mapping from the tokenizer
            index_word = tokenizer.index_word
        
            # Convert indices to words
            predicted_text = ' '.join([index_word.get(index, '') for index in predicted_classes[0]])
        
            return predicted_text
    
        # Decode the predicted classes into text
        predicted_text = decode_predictions(predicted_classes, overlap_tokenizer)
    
        return predicted_text


    def convert_chars_for_translated_overlap(string):
        string = string.replace("no overlap", "#", 1)  # Replace first occurrence of "no overlap" with "#"
        string = string.replace("overlap", "#", 1)  # Replace first occurrence of "overlap" with "#"
        string = string.replace("no", "#", 1)  # Replace first occurrence of "no" with "#"
        if string == "#":
            return string
        new_string = ""
        for char in string:
            if char.lower() == "b":
                new_string += "A"
            elif char.lower() == "j":
                new_string += "G"
            elif char.lower() == "o":
                new_string += "T"
            elif char.lower() == "u":
                new_string += "C"
            else:
                new_string += char
        return new_string.replace(" ", "")
    

    # Running the above sequence through the model, using the tokenizer run on the data from the original model training process
    translated_overlap = translate(trained_model, concat_tokenizer, overlap_tokenizer, aa_sequences)
    
    overlap_output_from_model = convert_chars_for_translated_overlap(translated_overlap)
    
    # Generate the rc to include in the next small df
    reverse_complement_overlap = Seq(overlap_output_from_model)
    reverse_complement_overlap = str(reverse_complement_overlap.reverse_complement())
    
    # Create a DataFrame with two rows: original and reversed sequences
    # Note, the rc sequence actually comes first here, followed by the model output overlap sequence
    output_forward_reverse = pd.DataFrame({"overlap_sequence": [reverse_complement_overlap, overlap_output_from_model]})
    
    # Splitting and collapsing the sequences
    sequence_list = aa_sequences.split()
    collapsed_sequence = ''.join(sequence_list)
    
    # Splitting the sequence at the asterisk and keeping the asterisk
    split_sequences_with_asterisk = [seq + '*' for seq in collapsed_sequence.split('*') if seq]
    
    # Creating a dataframe from the sequences
    df_sequences = pd.DataFrame(split_sequences_with_asterisk, columns=["amino_acid_sequence"])
    
    # Adding the back translated dna sequences to the data frame.
    df_sequences['nt_sequence'] = df_sequences['amino_acid_sequence'].apply(lambda aa_seq: translate_to_dna_with_all_options(aa_seq)[0])
    
    df_sequences["overlap_seq"] = output_forward_reverse
    
    #here, we are attaching the overlap sequence to the known coding sequence, generated above. It is
    #added at the location based on the length of the overlap_seq
    def modify_sequence(df):
        # Check if 'overlap_seq' column exists in the DataFrame
        if 'overlap_seq' not in df.columns:
            raise ValueError("DataFrame must contain 'overlap_seq' column.")
    
        # Calculate the length of the overlap sequence
        df['overlap_length'] = df['overlap_seq'].apply(len)
    
        # Modify the sequences
        df['modified_sequence'] = df.apply(lambda row: row['nt_sequence'][:-row['overlap_length']] + row['overlap_seq'], axis=1)
    
        return df
    
    modify_sequence(df_sequences)
    
    # Custom function to translate using Seq
    def translate_with_seq(sequence):
        coding_dna = Seq(sequence)
        return str(coding_dna.translate())
    
    # Apply the custom function to the 'nt_sequence' column
    df_sequences['translated_sequence'] = df_sequences['modified_sequence'].apply(translate_with_seq)
    
    #print(compare_aa)
    return(df_sequences)


#defining this align_sequences_identity function to check pariwise identity matching, since the globalxx approach allows sequence shifting, 
#which is problematic in the partial-matching predictions.
def align_sequences_identity(seq1, seq2):
    # Ensure sequences are of the same length
    if len(seq1) != len(seq2):
        raise ValueError("Sequences must be of the same length for 1:1 alignment")

    # Calculate the alignment score by comparing each character
    score = sum(1 for a, b in zip(seq1, seq2) if a == b)
    
    return score

################################
# Function to generate partially matching convergent overlap sequences
################################


def find_partially_matching_sequence(model_to_predict, formatted_aa_seq, max_attempts, alignment_threshold_1, alignment_threshold_2, blosum_threshold_1, blosum_threshold_2, _counter=[0]):
    _counter[0] += 1

    print(f"Parsing row {_counter[0]}...")

    attempts = 0
    match_found = False
    matching_dataframe = None
    
    while attempts < max_attempts:
        attempts += 1

        try:
            result_df = predict_overlapping_sequence(model_to_predict, formatted_aa_seq, concat_tokenizer, overlap_tokenizer)
        except CodonTable.TranslationError as e:
            print(f"Error: {e}")
             # Print the "overlap_length" at the end of each iteration
            print(f"Attempt {attempts}... Overlap Length: {matching_dataframe['overlap_length'].iloc[0] if matching_dataframe is not None else 'N/A'}")
            continue
        
        # Check if the translated sequences contain more than just the terminal asterisk
        if result_df['translated_sequence'].iloc[0].count('*') > 1 or result_df['translated_sequence'].iloc[1].count('*') > 1:
            continue  # Skip the rest of the loop and try again if there are multiple asterisks


        # Calculate the new overlap length based on the provided conditions
        overlap_length = result_df['overlap_length'].iloc[0]

        if overlap_length % 3 == 0:
            overlap_length_3 = overlap_length / 3
        else:
            overlap_length_3 = (overlap_length + 1) / 3

        # Round the result to the nearest whole number
        overlap_length_3 = int(round(overlap_length_3, 0))
        
        # Truncate the sequences based on overlap_length_3
        truncated_aa_seq_1 = result_df['amino_acid_sequence'].iloc[0][-overlap_length_3:]
        truncated_trans_seq_1 = result_df['translated_sequence'].iloc[0][-overlap_length_3:]
        
        truncated_aa_seq_2 = result_df['amino_acid_sequence'].iloc[1][-overlap_length_3:]
        truncated_trans_seq_2 = result_df['translated_sequence'].iloc[1][-overlap_length_3:]

        blosum62_sim_score_1 = calculate_blosum62_similarity(truncated_aa_seq_1, truncated_trans_seq_1)
        blosum62_sim_score_2 = calculate_blosum62_similarity(truncated_aa_seq_2, truncated_trans_seq_2)

        blosum62_sim_score_1_ratio = (calculate_blosum62_similarity(truncated_aa_seq_1, truncated_trans_seq_1)/calculate_blosum62_similarity(truncated_aa_seq_1, truncated_aa_seq_1))
        blosum62_sim_score_2_ratio = (calculate_blosum62_similarity(truncated_aa_seq_2, truncated_trans_seq_2)/calculate_blosum62_similarity(truncated_aa_seq_2, truncated_aa_seq_2))

        protsub_sim_score_1 = calculate_protsub_similarity(truncated_aa_seq_1, truncated_trans_seq_1)
        protsub_sim_score_2 = calculate_protsub_similarity(truncated_aa_seq_2, truncated_trans_seq_2)
        
        # Perform new alignment on truncated sequences
        truncated_align_score_1 = align_sequences_identity(truncated_aa_seq_1, truncated_trans_seq_1) / overlap_length_3
        truncated_align_score_2 = align_sequences_identity(truncated_aa_seq_2, truncated_trans_seq_2) / overlap_length_3
        
        # Add new alignment scores to the DataFrame
        result_df['truncated_norm_align_score_1'] = round(truncated_align_score_1, 2)
        result_df['truncated_norm_align_score_2'] = round(truncated_align_score_2, 2)

        # Add blosum62 similarity scores to the DataFrame
        # Add new alignment scores to the DataFrame
        result_df['blosum62_sim_score_1'] = round(blosum62_sim_score_1, 2)
        result_df['blosum62_sim_score_1_len_norm'] = round(blosum62_sim_score_1 / overlap_length_3, 2)

        result_df['blosum62_sim_score_2'] = round(blosum62_sim_score_2, 2)
        result_df['blosum62_sim_score_2_len_norm'] = round(blosum62_sim_score_2 / overlap_length_3, 2)

        result_df['blosum62_sim_score_1_ratio'] = round(blosum62_sim_score_1_ratio, 2)
        result_df['blosum62_sim_score_2_ratio'] = round(blosum62_sim_score_2_ratio, 2)

        # Combine the two truncated alignment scores into one column
        result_df['blosum62_combined_scores'] = result_df.apply(
            lambda row: f"{row['blosum62_sim_score_1']} / {row['blosum62_sim_score_2']}",
            axis=1
        )

        # Combine the two truncated alignment scores into one column
        result_df['blosum62_combined_normalized_scores'] = result_df.apply(
            lambda row: f"{row['blosum62_sim_score_1_len_norm']} / {row['blosum62_sim_score_2_len_norm']}",
            axis=1
        )

        # Combine the two truncated alignment scores into one column
        result_df['blosum62_combined_ratio_scores'] = result_df.apply(
            lambda row: f"{row['blosum62_sim_score_1_ratio']} / {row['blosum62_sim_score_2_ratio']}",
            axis=1
        )

        # Combine the two truncated alignment scores into one column
        result_df['truncated_combined_scores'] = result_df.apply(
            lambda row: f"{row['truncated_norm_align_score_1']} / {row['truncated_norm_align_score_2']}",
            axis=1
        )

        # Add protsub similarity scores to the DataFrame
        # Add new alignment scores to the DataFrame
        result_df['protsub_sim_score_1'] = round(protsub_sim_score_1, 2)
        result_df['protsub_sim_score_1_len_norm'] = round(protsub_sim_score_1 / overlap_length_3, 2)

        result_df['protsub_sim_score_2'] = round(protsub_sim_score_2, 2)
        result_df['protsub_sim_score_2_len_norm'] = round(protsub_sim_score_2 / overlap_length_3, 2)

        # Combine the two truncated alignment scores into one column
        result_df['protsub_combined_scores'] = result_df.apply(
            lambda row: f"{row['protsub_sim_score_1']} / {row['protsub_sim_score_2']}",
            axis=1
        )

        # Combine the two truncated alignment scores into one column
        result_df['protsub_combined_normalized_scores'] = result_df.apply(
            lambda row: f"{row['protsub_sim_score_1_len_norm']} / {row['protsub_sim_score_2_len_norm']}",
            axis=1
        )

        #update me to set the alignment lower boundary desired
        align_limit_1 = alignment_threshold_1
        align_limit_2 = alignment_threshold_2
        blosum_limit_1 = blosum_threshold_1
        blosum_limit_2 = blosum_threshold_2

        if (truncated_align_score_1 >= align_limit_1 and
            truncated_align_score_2 >= align_limit_2 and
            result_df['blosum62_sim_score_1_len_norm'].iloc[0] >= blosum_limit_1 and
            result_df['blosum62_sim_score_2_len_norm'].iloc[0] >= blosum_limit_2 and
            result_df['translated_sequence'].iloc[0].count('*') == 1 and
            result_df['translated_sequence'].iloc[1].count('*') == 1 and
            result_df['translated_sequence'].iloc[0].endswith('*') and
            result_df['translated_sequence'].iloc[1].endswith('*')):
            match_found = True
            matching_dataframe = result_df
            break

    if match_found:
        # Print the combined truncated alignment scores
        combined_scores = matching_dataframe['truncated_combined_scores'].iloc[0]
        #combined_blosum_scores = matching_dataframe['blosum62_combined_scores'].iloc[0]
        combined_blosum_len_norm_scores = matching_dataframe['blosum62_combined_normalized_scores'].iloc[0]
        #combined_protsub_scores = matching_dataframe['protsub_combined_scores'].iloc[0]
        combined_protsub_len_norm_scores = matching_dataframe['protsub_combined_normalized_scores'].iloc[0]
        blosum62_combined_ratio_scores = matching_dataframe['blosum62_combined_ratio_scores'].iloc[0]
        print(f"Attempt {attempts}... Overlap Length: {matching_dataframe['overlap_length'].iloc[0] if matching_dataframe is not None else 'N/A'}")
        print(f"Truncated Combined Scores: {combined_scores}")
        #print(f"Blosum62 Similarity Scores: {combined_blosum_scores}")
        #print(f"Blosum62 Length Normalized Similarity Scores: {combined_blosum_len_norm_scores}")
        #print(f"Blosum62 Length Normalized Similarity Ratio Scores: {blosum62_combined_ratio_scores}")
        #print(f"ProtSub Similarity Scores: {combined_protsub_scores}")
        #print(f"ProtSub Length Normalized Similarity Scores: {combined_protsub_len_norm_scores}")
        return matching_dataframe
    
    else:
        print("There is no predicted significant overlap")
        return None


def process_sequences(aa_seq_1, aa_seq_2):

    if len(aa_seq_1) < 103:
        return "Error: sequence_1 is not long enough to process. The minimum length is 103 amino acids."
    if len(aa_seq_2) < 103:
        return "Error: sequence_2 is not long enough to process. The minimum length is 103 amino acids."

    # Keep only the final 104 amino acids of each sequence
    aa_seq_1_trimmed = aa_seq_1[-104:]
    aa_seq_2_trimmed = aa_seq_2[-104:]

    # Replace the first amino acid with an 'M'
    aa_seq_1_processed = 'M' + aa_seq_1_trimmed[1:]
    aa_seq_2_processed = 'M' + aa_seq_2_trimmed[1:]

    # Add an asterisk after each sequence
    aa_seq_1_processed += '*'
    aa_seq_2_processed += '*'

    # Concatenate the two sequences
    concatenated_seq = aa_seq_1_processed + aa_seq_2_processed

    # Add a space between every character
    final_seq = ' '.join(concatenated_seq)

    # Return the processed sequence
    return final_seq

def translate_to_dna_with_all_options(aa_sequence: str) -> str:
    
    # Codons and their frequencies for each amino acid based on the E. coli table
    back_translation_code_with_all_options = {
        'A': [('GCG', 0.27), ('GCT', 0.26), ('GCC', 0.26), ('GCA', 0.21)],
        'C': [('TGC', 0.53), ('TGT', 0.47)],
        'D': [('GAT', 0.63), ('GAC', 0.37)],
        'E': [('GAA', 0.68), ('GAG', 0.32)],
        'F': [('TTT', 0.58), ('TTC', 0.42)],
        'G': [('GGC', 0.35), ('GGT', 0.32), ('GGG', 0.25), ('GGA', 0.08)],
        'H': [('CAT', 0.56), ('CAC', 0.44)],
        'I': [('ATT', 0.48), ('ATC', 0.39), ('ATA', 0.14)],
        'K': [('AAA', 0.74), ('AAG', 0.26)],
        'L': [('CTG', 0.43), ('CTT', 0.13), ('CTC', 0.13), ('TTA', 0.14), ('CTA', 0.07), ('TTG', 0.13)],
        'M': [('ATG', 1.00)],
        'N': [('AAC', 0.60), ('AAT', 0.40)],
        'P': [('CCG', 0.52), ('CCA', 0.19), ('CCT', 0.16), ('CCC', 0.13)],
        'Q': [('CAG', 0.66), ('CAA', 0.34)],
        'R': [('CGT', 0.36), ('CGC', 0.36), ('CGG', 0.11), ('AGA', 0.08), ('AGG', 0.05), ('CGA', 0.04)],
        'S': [('AGC', 0.24), ('TCC', 0.24), ('TCT', 0.17), ('TCG', 0.15), ('TCA', 0.14), ('AGT', 0.15)],
        'T': [('ACC', 0.36), ('ACA', 0.28), ('ACG', 0.25), ('ACT', 0.11)],
        'V': [('GTG', 0.46), ('GTT', 0.28), ('GTC', 0.15), ('GTA', 0.11)],
        'W': [('TGG', 1.00)],
        'Y': [('TAT', 0.59), ('TAC', 0.41)],
        '*': [('TAA', 0.61), ('TGA', 0.30), ('TAG', 0.09)]
    }

    def choose_codon_based_on_frequency(codons):
  
        # Extract codon names and their frequencies
        codon_names = [codon for codon, _ in codons]
        codon_freqs = [freq for _, freq in codons]

        # Normalize the frequencies to ensure they sum up to 1
        total_frequency = sum(codon_freqs)
        normalized_freqs = [freq / total_frequency for freq in codon_freqs]

        # Randomly select a codon based on the frequency distribution
        chosen_codon = np.random.choice(codon_names, p=normalized_freqs)

        return chosen_codon

    # For each amino acid in the sequence, choose the most probable codon
    chosen_codons = [choose_codon_based_on_frequency(back_translation_code_with_all_options[aa]) for aa in aa_sequence]

    # Combine the chosen codons to form the nucleotide sequence
    nucleotide_sequence = ''.join(chosen_codons)

    return nucleotide_sequence

# Function to integrate modified sequence into original sequence
def integrate_modified_sequence(original_dna, modified_dna):
    # Remove the terminal xx nucleotides from the original sequence
    trimmed_original_dna = original_dna[:-309] # this should be max nt length, minus 6, since the backtranslated sequence does not have stop codon DNA seqs
    # Remove the first three nucleotides from the modified sequence
    trimmed_modified_dna = modified_dna[3:]
    # Integrate the modified sequence
    integrated_sequence = trimmed_original_dna + trimmed_modified_dna
    return integrated_sequence

def is_dna_sequence(sequence: str) -> bool:
    """
    Determine if a sequence is a DNA sequence.
    A DNA sequence should only contain A, T, C, G (and sometimes N).
    """
    return all(char in 'ATCGNatcgn' for char in sequence)

def load_model(model_path, vocab_size, embedding_dim, num_blocks, num_heads, ffn_dim, max_length, dropout_rate):
    # Define and load model architecture
    model = TransformerModel(
        vocab_size=vocab_size, embedding_dim=embedding_dim, num_blocks=num_blocks,
        num_heads=num_heads, ffn_dim=ffn_dim, max_length=max_length, dropout_rate=dropout_rate
    )
    model.load_state_dict(torch.load(model_path))
    model.train()
    return model.to(torch.device('cuda' if torch.cuda.is_available() else 'cpu'))


############################

def get_bracket_positions(sequence_with_brackets):
    """
    Extract positions of amino acids within brackets in sequence1 and map them to sequence2.

    Args:
        sequence_with_brackets (str): The amino acid sequence with bracketed sections.

    Returns:
        list of tuples: Each tuple contains the start position in sequence2 and the bracketed amino acids.
    """
    bracketed_amino_acids = []
    pos_with_brackets = 0  # Position in sequence1 (with brackets)
    pos_without_brackets = 0  # Position in sequence2 (without brackets)

    while pos_with_brackets < len(sequence_with_brackets):
        if sequence_with_brackets[pos_with_brackets] == '[':
            pos_with_brackets += 1  # Skip '['
            bracket_start_seq2 = pos_without_brackets  # Record start position in sequence2
            bracket_content = []

            # Collect all amino acids within the brackets
            while pos_with_brackets < len(sequence_with_brackets) and sequence_with_brackets[pos_with_brackets] != ']':
                bracket_content.append(sequence_with_brackets[pos_with_brackets])
                pos_with_brackets += 1
                pos_without_brackets += 1  # Advance sequence2 position as brackets are not present

            if pos_with_brackets >= len(sequence_with_brackets):
                raise ValueError("Unmatched '[' in sequence.")

            pos_with_brackets += 1  # Skip ']'
            bracketed_amino_acids.append((bracket_start_seq2, ''.join(bracket_content)))
        else:
            # Regular amino acid, advance both positions
            pos_with_brackets += 1
            pos_without_brackets += 1

    return bracketed_amino_acids

def compare_sequences_aa_selected(sequence1_with_brackets, sequence2):
    """
    Compare specific bracketed sections in sequence1 with sequence2.

    Args:
        sequence1_with_brackets (str): First amino acid sequence with brackets.
        sequence2 (str): Second amino acid sequence without brackets.

    Returns:
        dict: A dictionary with start positions in sequence2 as keys and True/False for match/mismatch.
    """
    bracketed_amino_acids = get_bracket_positions(sequence1_with_brackets)
    results = {}

    for start, amino_acids in bracketed_amino_acids:
        # Extract amino acids from sequence2 for the corresponding positions
        seq2_amino_acids = sequence2[start:start + len(amino_acids)]
        results[start] = (seq2_amino_acids == amino_acids)  # True if match, False if not

    return results

# Function to get comparison results between sequence1 and sequence2
def get_comparison_results(sequence1, sequence2):
    comparison_results = compare_sequences_aa_selected(sequence1, sequence2)
    return comparison_results

def match_status(comparison_results):
    """
    Determines the match status based on the values in comparison_results.

    Args:
        comparison_results (dict): A dictionary with match results (True/False values).

    Returns:
        str: A message indicating whether all, some, or none of the comparisons matched.
    """
    if all(comparison_results.values()):
        return "Match"
    elif any(comparison_results.values()):
        return "Partial match"
    else:
        return "No match"


# New version that only accounts for overlap secondary structure predictions for the comparison values
def compare_sequences(seq1, seq2, pred1, pred2, overlap_length):
    """
    Compare two sequences with their respective predictions and score the matches.
    
    Args:
    - seq1: Original sequence 1 (string)
    - seq2: Original sequence 2 (string)
    - pred1: Predicted sequence 1 (string)
    - pred2: Predicted sequence 2 (string)
    - overlap_length: Overlap length (int) 
    
    Returns:
    - match_count: Number of positions where both sequences match their predictions
    - combined_score: Percentage score of combined matches relative to total positions
    - match_count_seq1: Number of positions where seq1 matches pred1
    - score_seq1: Percentage score of matches for seq1 relative to total positions
    - match_count_seq2: Number of positions where seq2 matches pred2
    - score_seq2: Percentage score of matches for seq2 relative to total positions
    - average_score_seq1_seq2: Mean average score of seq1 and seq2
    - abs_value: Absolute difference between score_seq1 and score_seq2
    """

    if overlap_length % 3 == 0:
        overlap_length_3 = overlap_length / 3
    else:
        overlap_length_3 = (overlap_length + 1) / 3

    # Round the result to the nearest whole number
    overlap_length_3 = int(round(overlap_length_3, 0))
    
    # Truncate the sequences based on overlap_length_3
    truncated_seq1 = seq1[-overlap_length_3:]
    truncated_seq2 = seq2[-overlap_length_3:]

    truncated_pred1 = pred1[-overlap_length_3:]
    truncated_pred2 = pred2[-overlap_length_3:]

    # Initialize counters
    total_positions = len(truncated_seq1)  # Assuming both sequences are the same length
    match_count = 0
    match_count_seq1 = 0
    match_count_seq2 = 0

    # Comparison loop
    for i in range(total_positions):
        if truncated_seq1[i] == truncated_pred1[i]:
            match_count_seq1 += 1
        if truncated_seq2[i] == truncated_pred2[i]:
            match_count_seq2 += 1
        if truncated_seq1[i] == truncated_pred2[i] and truncated_seq2[i] == truncated_pred2[i]:
            match_count += 1

    # Calculate scores
    combined_score = round(match_count / total_positions * 100, 2)
    score_seq1 = round(match_count_seq1 / total_positions * 100, 2)
    score_seq2 = round(match_count_seq2 / total_positions * 100, 2)
    average_score_seq1_seq2 = round((score_seq1 + score_seq2) / 2, 2)
    abs_value = round(abs(score_seq1 - score_seq2), 2)

    return (match_count, combined_score, match_count_seq1, score_seq1, 
            match_count_seq2, score_seq2, average_score_seq1_seq2, abs_value)



###############################################################################
# Helper – gather bracketed regions and compute per‑region identity
###############################################################################
def bracket_alignment_stats(seq_with_brackets: str,
                            candidate_seq: str) -> tuple[int, float]:
    """
    Return (match_count, match_fraction) for the AA positions enclosed by [].

    Parameters
    ----------
    seq_with_brackets : str
        Reference sequence containing one or more bracketed regions,
        e.g. "...TT[T]L[TYG]V...".
    candidate_seq : str
        Sequence to compare against (no brackets).

    Returns
    -------
    match_count : int
        Number of positions in bracketed regions whose residues match `candidate_seq`.
    match_fraction : float
        match_count / total_bracket_length, rounded to 4 decimals.
        Returns 0.0 if there are zero bracketed positions (shouldn’t happen).
    """
    bracketed_regions = get_bracket_positions(seq_with_brackets)  # [(start, "AAA"), ...]
    total_len, match_count = 0, 0

    for start, ref_subseq in bracketed_regions:
        total_len += len(ref_subseq)
        # Compare residue‑by‑residue within this bracketed block
        for offset, ref_aa in enumerate(ref_subseq):
            if start + offset >= len(candidate_seq):
                raise IndexError(
                    f"Candidate sequence too short for bracket at pos {start}"
                )
            if candidate_seq[start + offset] == ref_aa:
                match_count += 1

    match_fraction = round(match_count / total_len, 4) if total_len else 0.0
    return match_count, match_fraction

###########################
# Functions to predict secondary structures using S4Pred, batched
###########################

output_dir = "/home/jason/outputdir/python_projects/s4pred/outputs"

# This is used for single AA sequence secondary structure predictions
def structure_prediction_wrapper(sequence):
    output_dir = "/home/jason/outputdir/python_projects/s4pred/outputs"
    
    sequence = sequence.replace('*', '')

    # Call the prediction function
    prediction = predict_secondary_structure(sequence, output_dir)
    
    # Print the sequence and its prediction
    print(f"Sequence: {sequence}")
    print(f"Predicted Structure: {prediction}")
    print("-" * 50)  # Separator for readability
    
    return prediction

# This is used for batched AA sequence secondary structure predictions
def batch_structure_prediction_wrapper(sequences, output_dir):
    # Ensure all sequences in the batch are cleaned of asterisks
    cleaned_sequences = [seq.replace('*', '') for seq in sequences]
    
    # Call the batch prediction function
    predictions = predict_secondary_structure_batch(cleaned_sequences, output_dir)
    
    # Return the list of predictions
    return predictions

#####################
# Function to re-calculate an alignment score based on the original sequence, using the predicted sequence from a re-run of inference (used to fine-tune the sequence)
#####################

# Function to define the alignment score if rerun is True.
def rerun_alignment_score(original_sequence, predicted_sequence, overlap_length):
    # Remove terminal asterisk if it exists
    if original_sequence.endswith('*'):
        original_sequence = original_sequence[:-1]
    if predicted_sequence.endswith('*'):
        predicted_sequence = predicted_sequence[:-1]

    # Calculate overlap length in amino acids
    if overlap_length % 3 == 0:
        overlap_length_3 = overlap_length / 3
    else:
        overlap_length_3 = (overlap_length + 1) / 3

    # Round the result to the nearest whole number
    overlap_length_3 = int(round(overlap_length_3, 0))
    
    # Truncate the sequences based on overlap_length_3
    truncated_aa_seq = original_sequence[-overlap_length_3:]
    truncated_trans_seq = predicted_sequence[-overlap_length_3:]

    # Perform new alignment on truncated sequences
    truncated_align_score = align_sequences_identity(truncated_aa_seq, truncated_trans_seq) / overlap_length_3

    return truncated_align_score

#################
# Function to split the input sequence into two sequences without an asterisk
#################

def split_sequence(sequence):
    # Remove spaces from the sequence
    cleaned_sequence = sequence.replace(" ", "")
    
    # Split the sequence at asterisks and remove any empty strings in case of multiple asterisks
    split_seqs = [seq for seq in cleaned_sequence.split('*') if seq]
    
    # Ensure only two sequences are returned
    if len(split_seqs) == 2:
        return split_seqs[0], split_seqs[1]
    else:
        raise ValueError("The sequence does not split cleanly into two parts with a single asterisk.")

# AutoUpdate for selected amino acids
###########

def update_bracketed_sequence(seq, reference_bracketed_seq):
    """
    Replace the amino acids in 'seq' at the positions defined by 
    the bracketed regions in 'reference_bracketed_seq' with the bracketed amino acids.
    
    Args:
        seq (str): The candidate sequence to be updated.
        reference_bracketed_seq (str): The reference sequence that includes bracketed amino acids.
        
    Returns:
        str: The updated sequence.
    """
    # Get the positions and the bracketed amino acids from the reference
    bracket_positions = get_bracket_positions(reference_bracketed_seq)
    # Convert the sequence into a mutable list
    seq_list = list(seq)
    for pos, aa in bracket_positions:
        # Replace the substring with the bracketed amino acids.
        # (Assumes that the positions match the intended residues.)
        seq_list[pos: pos + len(aa)] = list(aa)
    return "".join(seq_list)



###########
## Model Data File
###########

# This is the 199 to 312 models set, with aa changes.
model_data = pd.read_excel('/mnt/e/RStuff/codon_overlap/trained_model_refs/overlap_length_models_aa_change_random_pairs_315nt_length_199_312_20250326_zero_and_non_zero_probs.xlsx')



In [ ]:
# Automatically test aa pairs in order, to identify those with matches
########################################################################################################################################

def select_and_update_sequences(
    final_results_df,
    sequence_1_aa_brackets,
    sequence_2_aa_brackets,
    score_threshold=None,
    avg_score_value=None,
    return_ranked=False
):
    # ---- filter ------------------------------------------------------
    filtered = final_results_df.copy()

    required = ['score_seq1', 'score_seq2', 'average_score_seq1_seq2']
    missing = [c for c in required if c not in final_results_df.columns]
    if missing:
        raise KeyError(f"select_and_update_sequences expected columns {missing} but they are not present.")

    if score_threshold is not None:
        mask = (
            filtered['score_seq1'].gt(score_threshold) &
            filtered['score_seq2'].gt(score_threshold)
        )
        filtered = filtered[mask]

    if avg_score_value is not None:
        filtered = filtered[
            filtered['average_score_seq1_seq2'] == avg_score_value
        ]

    if filtered.empty:                        # fall back to all rows
        filtered = final_results_df

    # ---- priority order ---------------------------------------------
    sorted_by_score = filtered.sort_values(
        by='average_score_seq1_seq2', ascending=False
    ).reset_index(drop=True)

    if return_ranked:
        return sorted_by_score

    # ---- legacy single-row return (unchanged) -----------------------
    best_row = sorted_by_score.iloc[0]
    upd1 = update_bracketed_sequence(
        best_row['translated_integrated_seq_1'], sequence_1_aa_brackets
    )
    upd2 = update_bracketed_sequence(
        best_row['translated_integrated_seq_2'], sequence_2_aa_brackets
    )
    return upd1, upd2


In [ ]:
# ================================================================================================
# ----------------------------- RANGE MERGER (deduplicates / coalesces) --------------------------
# ================================================================================================
from typing import List, Tuple

def merge_ranges(ranges: List[Tuple[int, int]]) -> List[Tuple[int, int]]:
    """
    Merge overlapping or immediately adjacent (start, end) index ranges.

    Parameters
    ----------
    ranges : List[Tuple[int, int]]
        Inclusive index pairs, e.g. [(12, 37), (35, 50)].

    Returns
    -------
    List[Tuple[int, int]]
        Sorted, non‑overlapping index pairs.
    """
    if not ranges:
        return []

    # sort by start index, then sweep‑merge
    ranges = sorted(ranges, key=lambda r: r[0])
    merged = [list(ranges[0])]

    for s, e in ranges[1:]:
        # if current range overlaps or kisses the previous one, extend it
        if s <= merged[-1][1] + 1:
            merged[-1][1] = max(merged[-1][1], e)
        else:
            merged.append([s, e])

    # cast inner lists back to tuples
    return [tuple(r) for r in merged]


In [ ]:
# ================================================================================================
# ----------------------------- FC‑DROPOUT RANGE BUILDER -----------------------------------------
# ================================================================================================
from typing import List, Tuple

def build_fc_dropout_ranges(seq_bracketed: str,
                            window_aa: int = 105,
                            tok_len: int = 315,
                            invert: bool = False) -> List[Tuple[int, int]]:
    """
    Convert bracketed amino‑acid notation into nucleotide‑token index ranges to be
    used for FC‑layer token‑dropout.

    Parameters
    ----------
    seq_bracketed : str
        Full amino‑acid sequence with regions to protect / measure enclosed in '[]'.
    window_aa     : int
        Length (in amino acids) of the C‑terminal slice that is fed to the model.
    tok_len       : int
        Length of the nucleotide token axis (== window_aa * 3).
    invert        : bool, default False
        Set True for sequences that are used in *reverse* orientation (seq‑2),
        producing indices relative to the non‑flipped mask.

    Returns
    -------
    List[Tuple[int, int]]
        Sorted, non‑overlapping (start, end) pairs **inclusive** on the token axis.
    """
    # ------------------------------------------------------------------
    # 1) locate every AA index that appears inside brackets
    # ------------------------------------------------------------------
    aa_idx = -1            # running index across *true* amino‑acid letters
    in_bracket = False
    current = []
    groups: List[List[int]] = []

    for ch in seq_bracketed:
        if ch == '[':
            in_bracket = True
            continue
        if ch == ']':
            in_bracket = False
            if current:
                groups.append(current)
                current = []
            continue
        if not ch.isalpha():   # ignore spaces / punctuation
            continue

        aa_idx += 1
        if in_bracket:
            current.append(aa_idx)

    if current:                 # handles a bracket that reaches the last char
        groups.append(current)

    aa_len = aa_idx + 1
    if aa_len < window_aa:
        raise ValueError("window_aa exceeds available AA length")

    # ------------------------------------------------------------------
    # 2) map AA indices -> nucleotide‑token indices (0 … tok_len‑1)
    #    token 0  == last nucleotide (3' end)
    # ------------------------------------------------------------------
    ranges: List[Tuple[int, int]] = []
    for g in groups:
        # discard any AA that lies *outside* the final window_aa slice
        valid = [pos for pos in g if (aa_len - 1 - pos) < window_aa]
        if not valid:
            continue

        # contiguous by construction – only need first & last
        first, last = valid[0], valid[-1]

        # distance from C‑terminus (k = 0 is very last AA)
        k_start = aa_len - 1 - last      # closest to 3' end
        k_end   = aa_len - 1 - first     # farthest within window

        tok_start = k_start * 3
        tok_end   = k_end   * 3 + 2      # inclusive

        if invert:                       # reverse orientation (seq‑2)
            tok_start, tok_end = tok_len - 1 - tok_end, tok_len - 1 - tok_start

        ranges.append((tok_start, tok_end))

    # ------------------------------------------------------------------
    # 3) merge any overlaps and sort
    # ------------------------------------------------------------------
    ranges.sort()
    merged: List[Tuple[int, int]] = []
    for s, e in ranges:
        if not merged or s > merged[-1][1] + 1:
            merged.append([s, e])
        else:
            merged[-1][1] = max(merged[-1][1], e)

    return [tuple(r) for r in merged]


In [ ]:
# Automatically test aa pairs in order, to identify those with matches
########################################################################################################################################

def select_and_update_sequences(
    final_results_df,
    sequence_1_aa_brackets,
    sequence_2_aa_brackets,
    score_threshold=None,
    avg_score_value=None,
    return_ranked=False
):
    # ---- filter ------------------------------------------------------
    filtered = final_results_df.copy()

    required = ['score_seq1', 'score_seq2', 'average_score_seq1_seq2']
    missing = [c for c in required if c not in final_results_df.columns]
    if missing:
        raise KeyError(f"select_and_update_sequences expected columns {missing} but they are not present.")

    if score_threshold is not None:
        mask = (
            filtered['score_seq1'].gt(score_threshold) &
            filtered['score_seq2'].gt(score_threshold)
        )
        filtered = filtered[mask]

    if avg_score_value is not None:
        filtered = filtered[
            filtered['average_score_seq1_seq2'] == avg_score_value
        ]

    if filtered.empty:                        # fall back to all rows
        filtered = final_results_df

    # ---- priority order ---------------------------------------------
    sorted_by_score = filtered.sort_values(
        by='average_score_seq1_seq2', ascending=False
    ).reset_index(drop=True)

    if return_ranked:
        return sorted_by_score

    # ---- legacy single-row return (unchanged) -----------------------
    best_row = sorted_by_score.iloc[0]
    upd1 = update_bracketed_sequence(
        best_row['translated_integrated_seq_1'], sequence_1_aa_brackets
    )
    upd2 = update_bracketed_sequence(
        best_row['translated_integrated_seq_2'], sequence_2_aa_brackets
    )
    return upd1, upd2


In [ ]:
# ================================================================================================
# ----------------------------- RANGE MERGER (deduplicates / coalesces) --------------------------
# ================================================================================================
from typing import List, Tuple

def merge_ranges(ranges: List[Tuple[int, int]]) -> List[Tuple[int, int]]:
    """
    Merge overlapping or immediately adjacent (start, end) index ranges.

    Parameters
    ----------
    ranges : List[Tuple[int, int]]
        Inclusive index pairs, e.g. [(12, 37), (35, 50)].

    Returns
    -------
    List[Tuple[int, int]]
        Sorted, non‑overlapping index pairs.
    """
    if not ranges:
        return []

    # sort by start index, then sweep‑merge
    ranges = sorted(ranges, key=lambda r: r[0])
    merged = [list(ranges[0])]

    for s, e in ranges[1:]:
        # if current range overlaps or kisses the previous one, extend it
        if s <= merged[-1][1] + 1:
            merged[-1][1] = max(merged[-1][1], e)
        else:
            merged.append([s, e])

    # cast inner lists back to tuples
    return [tuple(r) for r in merged]


In [ ]:
# ================================================================================================
# ----------------------------- FC‑DROPOUT RANGE BUILDER -----------------------------------------
# ================================================================================================
from typing import List, Tuple

def build_fc_dropout_ranges(seq_bracketed: str,
                            window_aa: int = 105,
                            tok_len: int = 315,
                            invert: bool = False) -> List[Tuple[int, int]]:
    """
    Convert bracketed amino‑acid notation into nucleotide‑token index ranges to be
    used for FC‑layer token‑dropout.

    Parameters
    ----------
    seq_bracketed : str
        Full amino‑acid sequence with regions to protect / measure enclosed in '[]'.
    window_aa     : int
        Length (in amino acids) of the C‑terminal slice that is fed to the model.
    tok_len       : int
        Length of the nucleotide token axis (== window_aa * 3).
    invert        : bool, default False
        Set True for sequences that are used in *reverse* orientation (seq‑2),
        producing indices relative to the non‑flipped mask.

    Returns
    -------
    List[Tuple[int, int]]
        Sorted, non‑overlapping (start, end) pairs **inclusive** on the token axis.
    """
    # ------------------------------------------------------------------
    # 1) locate every AA index that appears inside brackets
    # ------------------------------------------------------------------
    aa_idx = -1            # running index across *true* amino‑acid letters
    in_bracket = False
    current = []
    groups: List[List[int]] = []

    for ch in seq_bracketed:
        if ch == '[':
            in_bracket = True
            continue
        if ch == ']':
            in_bracket = False
            if current:
                groups.append(current)
                current = []
            continue
        if not ch.isalpha():   # ignore spaces / punctuation
            continue

        aa_idx += 1
        if in_bracket:
            current.append(aa_idx)

    if current:                 # handles a bracket that reaches the last char
        groups.append(current)

    aa_len = aa_idx + 1
    if aa_len < window_aa:
        raise ValueError("window_aa exceeds available AA length")

    # ------------------------------------------------------------------
    # 2) map AA indices -> nucleotide‑token indices (0 … tok_len‑1)
    #    token 0  == last nucleotide (3' end)
    # ------------------------------------------------------------------
    ranges: List[Tuple[int, int]] = []
    for g in groups:
        # discard any AA that lies *outside* the final window_aa slice
        valid = [pos for pos in g if (aa_len - 1 - pos) < window_aa]
        if not valid:
            continue

        # contiguous by construction – only need first & last
        first, last = valid[0], valid[-1]

        # distance from C‑terminus (k = 0 is very last AA)
        k_start = aa_len - 1 - last      # closest to 3' end
        k_end   = aa_len - 1 - first     # farthest within window

        tok_start = k_start * 3
        tok_end   = k_end   * 3 + 2      # inclusive

        if invert:                       # reverse orientation (seq‑2)
            tok_start, tok_end = tok_len - 1 - tok_end, tok_len - 1 - tok_start

        ranges.append((tok_start, tok_end))

    # ------------------------------------------------------------------
    # 3) merge any overlaps and sort
    # ------------------------------------------------------------------
    ranges.sort()
    merged: List[Tuple[int, int]] = []
    for s, e in ranges:
        if not merged or s > merged[-1][1] + 1:
            merged.append([s, e])
        else:
            merged[-1][1] = max(merged[-1][1], e)

    return [tuple(r) for r in merged]


In [ ]:
###################################################################################################
# FINAL FC-LAYER TOKEN-DROPOUT with FIXED-LOGIT CONSTRAINTS (no MHA masking). Mirroring fixed.
###################################################################################################
from typing import Optional, List, Tuple, Dict

import torch
import torch.nn as nn
import math
import time
import pandas as pd
from Bio.Seq import Seq

# ================================================================================================
# ----------------------------- FC DROPOUT / CONSTRAINT UTILITIES --------------------------------
# ================================================================================================
def _build_bool_mask_from_ranges(L: int,
                                 ranges: Optional[List[Tuple[int, int]]]) -> Optional[torch.Tensor]:
    if not ranges:
        return None
    m = torch.zeros(L, dtype=torch.bool)
    for s, e in ranges:
        if s < 0 or e < 0 or s >= L or e >= L or e < s:
            raise ValueError(f"Bad range ({s},{e}) for length {L}")
        m[s:e+1] = True
    return m


# Mapping derived from your dictionary and char→nucleotide rules:
#   dict: {'o':1,'b':2,'u':3,'j':4,'no':5,'overlap':6}
#   char→nuc in your conversion: b→A, j→G, o→T, u→C  ⇒  T:1, A:2, C:3, G:4
NUC_TO_CLASS: Dict[str, int] = {'T': 1, 'A': 2, 'C': 3, 'G': 4}
CLASS_TO_NUC: Dict[int, str] = {v: k for k, v in NUC_TO_CLASS.items()}

def classes_from_nuc_string(nuc_string: str) -> List[int]:
    """Map a nucleotide string like 'TACG' to class indices [1,2,3,4]."""
    out: List[int] = []
    for ch in nuc_string:
        ch = ch.upper()
        if ch in NUC_TO_CLASS:
            out.append(NUC_TO_CLASS[ch])
    return out

def build_fixed_logits_spec(positions: List[int],
                            classes: List[int],
                            L: int) -> Dict[int, int]:
    """Return {pos: class_idx} with bounds checks."""
    if len(positions) != len(classes):
        raise ValueError("positions and classes must be same length")
    spec: Dict[int, int] = {}
    for p, c in zip(positions, classes):
        if p < 0 or p >= L:
            raise ValueError(f"Bad fixed position {p} for length {L}")
        spec[p] = int(c)
    return spec

def fixed_spec_from_nuc_string(nuc_string: str,
                               start_pos: int,
                               tok_len: int) -> Dict[int, int]:
    """Convenience: 'TACG', start_pos=100 -> {100:1,101:2,102:3,103:4}."""
    cls = classes_from_nuc_string(nuc_string)
    positions = list(range(start_pos, start_pos + len(cls)))
    return build_fixed_logits_spec(positions, cls, L=tok_len)


def _apply_fixed_logits(out: torch.Tensor,
                        fixed_spec: Dict[int, int],
                        tok_ax: int,
                        fixed_value: float = 12.0,
                        blend_alpha: Optional[float] = None) -> torch.Tensor:
    """
    Enforce logits at positions in fixed_spec.

    out: logits tensor with class axis last; token axis = tok_ax
    fixed_value:
      - blend_alpha is None  -> hard one-hot: target=+fixed_value, others=-inf
      - blend_alpha in (0,1] -> soft blend: out=(1-α)·out + α·one_hot·fixed_value
    """
    # Move token axis to front for easy indexing
    perm = [tok_ax] + [i for i in range(out.dim()) if i != tok_ax]
    inv  = [perm.index(i) for i in range(len(perm))]
    x = out.permute(*perm)  # (T, ..., V)
    T = x.shape[0]

    for pos, cls in fixed_spec.items():
        if pos < 0 or pos >= T:
            continue
        sl = x[pos]  # (..., V)
        if blend_alpha is None:
            sl.fill_(-1e9)
            sl[..., cls] = fixed_value
        else:
            one_hot = torch.zeros_like(sl)
            one_hot[..., cls] = fixed_value
            sl.mul_(1.0 - blend_alpha).add_(one_hot, alpha=blend_alpha)

    return x.permute(*inv)


def _masked_logit_dropout_with_constraints(
    logits: torch.Tensor,
    token_mask_1d: Optional[torch.Tensor],
    p: float,
    training_flag: bool,
    tok_len: int = 315,
    fixed_logits_spec: Optional[Dict[int, int]] = None,
    fixed_value: float = 12.0,
    blend_alpha: Optional[float] = None,
    debug: bool = False
) -> torch.Tensor:
    # Identify token axis exactly like before
    if logits.shape[0] == tok_len:
        tok_ax = 0
    elif logits.ndim > 1 and logits.shape[1] == tok_len:
        tok_ax = 1
    else:
        # No token axis found; still allow fixed-only path
        if fixed_logits_spec:
            return logits  # or raise, but keeping old behavior
        return logits

    # If no dropout to apply, still enforce fixed logits if present
    if (not training_flag) or p == 0.0 or token_mask_1d is None:
        if fixed_logits_spec:
            return _apply_fixed_logits(
                logits, fixed_spec=fixed_logits_spec, tok_ax=tok_ax,
                fixed_value=fixed_value, blend_alpha=blend_alpha
            )
        return logits

    # Permute to put token axis first
    perm = [tok_ax] + [i for i in range(logits.dim()) if i != tok_ax]
    inv  = [perm.index(i) for i in range(len(perm))]
    x = logits.permute(*perm)  # (T, ..., V)
    T = x.shape[0]

    # Build/broadcast mask EXACTLY like the old function (per-class broadcasting)
    mask = token_mask_1d.to(x.device)
    if mask.shape[0] != T:
        raise ValueError(f"token_mask_1d length {mask.shape[0]} != token axis {T}")
    mask_view = mask.view(T, *([1] * (x.dim() - 1))).expand_as(x)  # note: -1 (not -2), covers class axis too

    # Do-not-drop mask for fixed positions, also broadcast over classes
    if fixed_logits_spec:
        fixed_mask_t = torch.zeros(T, dtype=torch.bool, device=x.device)
        for pos in fixed_logits_spec.keys():
            if 0 <= pos < T:
                fixed_mask_t[pos] = True
        fixed_mask_view = fixed_mask_t.view(T, *([1] * (x.dim() - 1))).expand_as(x)
    else:
        fixed_mask_view = torch.zeros_like(x, dtype=torch.bool)

    # PER-CLASS dropout, like the old hook
    rand = torch.rand_like(x)
    drop = (rand < p) & mask_view & (~fixed_mask_view)
    keep = ~drop

    out = x * keep.to(x.dtype)

    # Inverted dropout scaling on all masked, non-fixed entries (matches old)
    scale = torch.ones_like(x)
    scale[mask_view & (~fixed_mask_view)] = 1.0 / (1.0 - p)
    out = out * scale

    # Now enforce fixed logits (hard/soft) at those positions
    if fixed_logits_spec:
        out = _apply_fixed_logits(out, fixed_spec=fixed_logits_spec, tok_ax=0,
                                  fixed_value=fixed_value, blend_alpha=blend_alpha)

    return out.permute(*inv)



def _find_output_linear_single(model: nn.Module,
                               vocab_size_guess: Optional[int] = None):
    if hasattr(model, "fc") and isinstance(model.fc, nn.Linear):
        return [("fc", model.fc)]
    cands = [(n, m) for n, m in model.named_modules() if isinstance(m, nn.Linear)]
    if not cands:
        return []
    if vocab_size_guess is not None:
        vs = [(n, m) for (n, m) in cands if m.out_features == vocab_size_guess]
        if vs:
            return [vs[-1]]
    return [cands[-1]]


def install_fc_token_dropout_and_constraints_hook(model: nn.Module,
                                                  token_mask_1d: Optional[torch.Tensor],
                                                  p: float,
                                                  force_eval: bool,
                                                  tok_len: int,
                                                  vocab_size_guess: Optional[int] = None,
                                                  fixed_logits_spec: Optional[Dict[int, int]] = None,
                                                  fixed_value: float = 12.0,
                                                  blend_alpha: Optional[float] = None,
                                                  debug: bool = False):
    """
    Registers a forward hook on the final Linear to apply:
      - masked token dropout (MC) in the selected region
      - fixed logits at user-specified token positions

    If you want only fixed logits (no dropout), pass p=0; token_mask_1d may be None.
    """
    # Still set up the hook if fixed logits are requested even without a mask/p>0
    if (p <= 0.0 and not fixed_logits_spec) and token_mask_1d is None:
        return []

    targets = _find_output_linear_single(model, vocab_size_guess)
    if not targets:
        return []

    def make_hook():
        def hook(_module, _inp, out):
            training_flag = _module.training or force_eval
            return _masked_logit_dropout_with_constraints(
                out,
                token_mask_1d=token_mask_1d,
                p=p,
                training_flag=training_flag,
                tok_len=tok_len,
                fixed_logits_spec=fixed_logits_spec,
                fixed_value=fixed_value,
                blend_alpha=blend_alpha,
                debug=debug
            )
        return hook

    handles = []
    for _name, lin in targets:
        h = lin.register_forward_hook(make_hook())
        handles.append(h)
    return handles


def shift_and_mirror_fixed_logits_positions_forward(
    fixed_spec: Dict[int, int],
    tok_len: int,
    model_number: int
) -> Dict[int, int]:
    """
    Shift fixed logits positions forward, then mirror across tok_len to align with reverse predictions.
    """
    #shift = tok_len - model_number - 7
    shift = 0
    mirrored = {}
    for pos, cls in fixed_spec.items():
        shifted_pos = pos - shift
        mirrored_pos = (tok_len - 1) - shifted_pos  # global mirror
        mirrored[mirrored_pos] = cls
    return mirrored



def shift_and_mirror_fixed_logits_positions_forward(
    fixed_spec: Dict[int, int],
    tok_len: int,
    model_number: int
) -> Dict[int, int]:
    """
    Shift fixed logits positions forward, then mirror across tok_len to align with reverse predictions.
    Excludes any positions that fall outside [0, tok_len-1].
    """
    shift = 0  # change if you actually need to apply an offset

    mirrored = {}
    for pos, cls in fixed_spec.items():
        # Apply shift
        shifted_pos = pos - shift

        # Skip if shifted position is invalid
        if shifted_pos < 0 or shifted_pos >= tok_len:
            continue

        # Mirror and skip if mirrored position is invalid
        mirrored_pos = (tok_len - 1) - shifted_pos
        if mirrored_pos < 0 or mirrored_pos >= tok_len:
            continue

        mirrored[mirrored_pos] = cls

    return mirrored

def swap_fixed_logits_spec_classes_only_reverse(
    fixed_spec: Dict[int, int],
    tok_len: int,
    model_number: int
) -> Dict[int, int]:
    """
    Swap nucleotide class IDs only (no position mirroring) and shift positions:
      - Shift each position by subtracting (tok_len - model_number).
      - Swap classes: 1 <-> 2, 3 <-> 4.
      - Other IDs remain unchanged.
    """
    swap_map = {1: 2, 2: 1, 3: 4, 4: 3}
    shift = tok_len - model_number
    swapped = {}
    for pos, cls in fixed_spec.items():
        new_pos = pos - shift
        swapped_cls = swap_map.get(cls, cls)
        swapped[new_pos] = swapped_cls
    return swapped



# ================================================================================================
# ----------------------------- MAIN INFERENCE ----------------------------------------------------
# ================================================================================================
def run_inference_for_models(
        model_numbers,
        seq_1,
        seq_2,
        max_attempts_first_pass,
        max_attempts_second_pass,
        first_pass_alignment_threshold_1,
        first_pass_alignment_threshold_2,
        second_pass_alignment_threshold_1,
        second_pass_alignment_threshold_2,
        first_pass_blosum_threshold_1,
        first_pass_blosum_threshold_2,
        second_pass_blosum_threshold_1,
        second_pass_blosum_threshold_2,
        first_pass_iterations,
        second_pass_iterations,
        inference_mode,
        feedforward_dropout_rate,
        attention_dropout_rate,
        set_seed,
        # NEW ARGS
        fc_dropout_p: float = 0.0,
        fc_ranges_forward: Optional[List[Tuple[int,int]]] = None,
        fc_ranges_reverse: Optional[List[Tuple[int,int]]] = None,
        tok_len: int = 315,
        apply_fc_dropout_in_eval: bool = True,
        vocab_size_guess: Optional[int] = None,
        # NEW: fixed logits control
        fixed_logits_spec_forward: Optional[Dict[int, int]] = None,
        fixed_logits_spec_reverse: Optional[Dict[int, int]] = None,
        fixed_value: float = 12.0,
        blend_alpha: Optional[float] = None
    ):

    if set_seed is None:
        torch.manual_seed(int(time.time()))
    else:
        torch.manual_seed(set_seed)

    if is_dna_sequence(seq_1) and is_dna_sequence(seq_2):
        aa_seq_1 = str(Seq(seq_1).translate())[:-1]
        aa_seq_2 = str(Seq(seq_2).translate())[:-1]
    else:
        aa_seq_1 = seq_1
        aa_seq_2 = seq_2
        back_translated_aa_seq_1 = str(translate_to_dna_with_all_options(aa_seq_1))
        back_translated_aa_seq_2 = str(translate_to_dna_with_all_options(aa_seq_2))

    concatenated_sequence = process_sequences(aa_seq_1, aa_seq_2)
    if "Error" in concatenated_sequence:
        print(concatenated_sequence)
        return None

    print("The input amino acid sequences have been processed and concatenated:")
    print(concatenated_sequence)

    p = concatenated_sequence.split('*')
    formatted_aa_seq_1 = f"{p[0].strip()} * {p[1].strip()} *"
    formatted_aa_seq_2 = f"{p[1].strip()} * {p[0].strip()} *"

    # Build FC masks
    fc_mask_forward = None
    if fc_ranges_forward is not None:
        fc_mask_forward = _build_bool_mask_from_ranges(tok_len, fc_ranges_forward).flip(0)
    fc_mask_reverse = _build_bool_mask_from_ranges(tok_len, fc_ranges_reverse)
    
    if fixed_logits_spec_forward is not None:
        fixed_logits_spec_forward = shift_and_mirror_fixed_logits_positions_forward(fixed_logits_spec_forward, tok_len, model_numbers[0])
        print(fixed_logits_spec_forward)

    if fixed_logits_spec_reverse is not None:
        fixed_logits_spec_reverse = swap_fixed_logits_spec_classes_only_reverse(fixed_logits_spec_reverse, tok_len, model_numbers[0])
        print(fixed_logits_spec_reverse)


    all_results = []

    # --------------------------- FIRST PASS ---------------------------
    for model_num in model_numbers:
        model_path = model_data.loc[model_data['overlap_length'] == model_num, 'model_pth_location'].values[0]
        model = torch.load(model_path, map_location=device, weights_only=False).to(device)

        if inference_mode == "train":
            model.train()
        elif inference_mode == "eval":
            model.eval()
        else:
            raise ValueError("Invalid inference_mode. Choose 'train' or 'eval'.")

        for m in model.modules():
            if isinstance(m, nn.MultiheadAttention):
                m.dropout = attention_dropout_rate
            if isinstance(m, nn.Dropout):
                m.p = feedforward_dropout_rate
            if isinstance(m, nn.LayerNorm):
                m.eval()

        print("################################################################################################")
        print(f'Attempting to predict overlaps of length: {model_num}')
        print(f'This will be run {first_pass_iterations} time(s), then reversed.')
        print("################################################################################################")

        # Forward orientation
        fwd_handles = install_fc_token_dropout_and_constraints_hook(
            model,
            token_mask_1d=fc_mask_forward,
            p=fc_dropout_p,
            force_eval=apply_fc_dropout_in_eval,
            tok_len=tok_len,
            vocab_size_guess=vocab_size_guess,
            fixed_logits_spec=fixed_logits_spec_forward,
            fixed_value=fixed_value,
            blend_alpha=blend_alpha,
            debug=False
        )

        for i in range(first_pass_iterations):
            df = find_partially_matching_sequence(
                model,
                formatted_aa_seq_1,
                max_attempts_first_pass,
                first_pass_alignment_threshold_1,
                first_pass_alignment_threshold_2,
                first_pass_blosum_threshold_1,
                first_pass_blosum_threshold_2
            )
            if df is not None:
                all_results.append({
                    'model_number': model_num,
                    'pass': 'first',
                    'iteration': i + 1,
                    'modified_sequence_1': df['modified_sequence'].iloc[0],
                    'modified_sequence_2': df['modified_sequence'].iloc[1],
                    'overlap_length': df['overlap_length'].iloc[0],
                    'translated_aa_seq_1': df['translated_sequence'].iloc[0],
                    'translated_aa_seq_2': df['translated_sequence'].iloc[1],
                    'truncated_norm_align_score_1': round(df['truncated_norm_align_score_1'].iloc[0], 2),
                    'truncated_norm_align_score_2': round(df['truncated_norm_align_score_2'].iloc[0], 2),
                    'truncated_norm_align_avg_score': round(
                        (df['truncated_norm_align_score_1'].iloc[0] + df['truncated_norm_align_score_2'].iloc[0]) / 2, 2
                    ),
                    'blosum62_sim_score_1': round(df['blosum62_sim_score_1'].iloc[0], 2),
                    'blosum62_sim_score_2': round(df['blosum62_sim_score_2'].iloc[0], 2),
                    'blosum_sim_score_len_norm_1': round(df['blosum62_sim_score_1_len_norm'].iloc[0], 2),
                    'blosum_sim_score_len_norm_2': round(df['blosum62_sim_score_2_len_norm'].iloc[0], 2),
                    'blosum62_sum_score': round(df['blosum62_sim_score_1'].iloc[0] + df['blosum62_sim_score_2'].iloc[0], 2),
                    'protsub_sim_score_1': round(df['protsub_sim_score_1'].iloc[0], 2),
                    'protsub_sim_score_2': round(df['protsub_sim_score_2'].iloc[0], 2),
                    'protsub_sim_score_len_norm_1': round(df['protsub_sim_score_1_len_norm'].iloc[0], 2),
                    'protsub_sim_score_len_norm_2': round(df['protsub_sim_score_2_len_norm'].iloc[0], 2),
                    'protsub_sum_score': round(df['protsub_sim_score_1'].iloc[0] + df['protsub_sim_score_2'].iloc[0], 2)
                })

        for h in fwd_handles: h.remove()

        # Reverse orientation (NO MIRRORING if mask is None)
        rev_handles = install_fc_token_dropout_and_constraints_hook(
            model,
            token_mask_1d=fc_mask_reverse,
            p=fc_dropout_p,
            force_eval=apply_fc_dropout_in_eval,
            tok_len=tok_len,
            vocab_size_guess=vocab_size_guess,
            fixed_logits_spec=fixed_logits_spec_reverse,
            fixed_value=fixed_value,
            blend_alpha=blend_alpha,
            debug=False
        )

        for i in range(first_pass_iterations):
            df = find_partially_matching_sequence(
                model,
                formatted_aa_seq_2,
                max_attempts_first_pass,
                first_pass_alignment_threshold_2,
                first_pass_alignment_threshold_1,
                first_pass_blosum_threshold_2,
                first_pass_blosum_threshold_1
            )
            if df is not None:
                all_results.append({
                    'model_number': model_num,
                    'pass': 'first',
                    'iteration': i + 1,
                    'modified_sequence_1': df['modified_sequence'].iloc[1],
                    'modified_sequence_2': df['modified_sequence'].iloc[0],
                    'overlap_length': df['overlap_length'].iloc[0],
                    'translated_aa_seq_1': df['translated_sequence'].iloc[1],
                    'translated_aa_seq_2': df['translated_sequence'].iloc[0],
                    'truncated_norm_align_score_1': round(df['truncated_norm_align_score_2'].iloc[0], 2),
                    'truncated_norm_align_score_2': round(df['truncated_norm_align_score_1'].iloc[0], 2),
                    'truncated_norm_align_avg_score': round(
                        (df['truncated_norm_align_score_1'].iloc[0] + df['truncated_norm_align_score_2'].iloc[0]) / 2, 2
                    ),
                    'blosum62_sim_score_1': round(df['blosum62_sim_score_2'].iloc[0], 2),
                    'blosum62_sim_score_2': round(df['blosum62_sim_score_1'].iloc[0], 2),
                    'blosum_sim_score_len_norm_1': round(df['blosum62_sim_score_2_len_norm'].iloc[0], 2),
                    'blosum_sim_score_len_norm_2': round(df['blosum62_sim_score_1_len_norm'].iloc[0], 2),
                    'blosum62_sum_score': round(df['blosum62_sim_score_1'].iloc[0] + df['blosum62_sim_score_2'].iloc[0], 2),
                    'protsub_sim_score_1': round(df['protsub_sim_score_2'].iloc[0], 2),
                    'protsub_sim_score_2': round(df['protsub_sim_score_1'].iloc[0], 2),
                    'protsub_sim_score_len_norm_1': round(df['protsub_sim_score_2_len_norm'].iloc[0], 2),
                    'protsub_sim_score_len_norm_2': round(df['protsub_sim_score_1_len_norm'].iloc[0], 2),
                    'protsub_sum_score': round(df['protsub_sim_score_1'].iloc[0] + df['protsub_sim_score_2'].iloc[0], 2)
                })

        for h in rev_handles: h.remove()

    # --------------------------- FILTER ---------------------------
    results_df = pd.DataFrame(all_results)
    filtered = results_df[
        (results_df['truncated_norm_align_score_1'] >= first_pass_alignment_threshold_1) &
        (results_df['truncated_norm_align_score_2'] >= first_pass_alignment_threshold_2) &
        (results_df['overlap_length'] >= results_df['model_number'].apply(math.floor))
    ].drop_duplicates(subset=['model_number'])

    if filtered.empty:
        print("No models meet the alignment threshold.")
        return None

    final_results = []

    # --------------------------- SECOND PASS ---------------------------
    for _, row in filtered.iterrows():
        model_num = row['model_number']
        model_path = model_data.loc[model_data['overlap_length'] == model_num, 'model_pth_location'].values[0]
        model = torch.load(model_path, map_location=device, weights_only=False).to(device)

        if inference_mode == "train":
            model.train()
        else:
            model.eval()

        for m in model.modules():
            if isinstance(m, nn.MultiheadAttention):
                m.dropout = attention_dropout_rate
            if isinstance(m, nn.Dropout):
                m.p = feedforward_dropout_rate

        print("################################################################################################")
        print(f'Attempting to predict overlaps of length: {model_num}')
        print(f'This will be run {second_pass_iterations} time(s), then reversed.')
        print("################################################################################################")

        # Forward
        fwd_handles = install_fc_token_dropout_and_constraints_hook(
            model,
            token_mask_1d=fc_mask_forward,
            p=fc_dropout_p,
            force_eval=apply_fc_dropout_in_eval,
            tok_len=tok_len,
            vocab_size_guess=vocab_size_guess,
            fixed_logits_spec=fixed_logits_spec_forward,
            fixed_value=fixed_value,
            blend_alpha=blend_alpha,
            debug=False
        )

        for i in range(second_pass_iterations):
            df = find_partially_matching_sequence(
                model,
                formatted_aa_seq_1,
                max_attempts_second_pass,
                second_pass_alignment_threshold_1,
                second_pass_alignment_threshold_2,
                second_pass_blosum_threshold_1,
                second_pass_blosum_threshold_2
            )
            if df is not None:
                final_results.append({
                    'model_number': model_num,
                    'pass': 'second',
                    'iteration': i + 1,
                    'modified_sequence_1': df['modified_sequence'].iloc[0],
                    'modified_sequence_2': df['modified_sequence'].iloc[1],
                    'overlap_length': df['overlap_length'].iloc[0],
                    'translated_aa_seq_1': df['translated_sequence'].iloc[0],
                    'translated_aa_seq_2': df['translated_sequence'].iloc[1],
                    'truncated_norm_align_score_1': round(df['truncated_norm_align_score_1'].iloc[0], 2),
                    'truncated_norm_align_score_2': round(df['truncated_norm_align_score_2'].iloc[0], 2),
                    'truncated_norm_align_avg_score': round(
                        (df['truncated_norm_align_score_1'].iloc[0] + df['truncated_norm_align_score_2'].iloc[0]) / 2, 2
                    ),
                    'blosum62_sim_score_1': round(df['blosum62_sim_score_1'].iloc[0], 2),
                    'blosum62_sim_score_2': round(df['blosum62_sim_score_2'].iloc[0], 2),
                    'blosum_sim_score_len_norm_1': round(df['blosum62_sim_score_1_len_norm'].iloc[0], 2),
                    'blosum_sim_score_len_norm_2': round(df['blosum62_sim_score_2_len_norm'].iloc[0], 2),
                    'blosum62_sum_score': round(df['blosum62_sim_score_1'].iloc[0] + df['blosum62_sim_score_2'].iloc[0], 2),
                    'protsub_sim_score_1': round(df['protsub_sim_score_1'].iloc[0], 2),
                    'protsub_sim_score_2': round(df['protsub_sim_score_2'].iloc[0], 2),
                    'protsub_sim_score_len_norm_1': round(df['protsub_sim_score_1_len_norm'].iloc[0], 2),
                    'protsub_sim_score_len_norm_2': round(df['protsub_sim_score_2_len_norm'].iloc[0], 2),
                    'protsub_sum_score': round(df['protsub_sim_score_1'].iloc[0] + df['protsub_sim_score_2'].iloc[0], 2)
                })

        for h in fwd_handles: h.remove()

        # Reverse
        rev_handles = install_fc_token_dropout_and_constraints_hook(
            model,
            token_mask_1d=fc_mask_reverse,
            p=fc_dropout_p,
            force_eval=apply_fc_dropout_in_eval,
            tok_len=tok_len,
            vocab_size_guess=vocab_size_guess,
            fixed_logits_spec=fixed_logits_spec_reverse,
            fixed_value=fixed_value,
            blend_alpha=blend_alpha,
            debug=False
        )

        for i in range(second_pass_iterations):
            df = find_partially_matching_sequence(
                model,
                formatted_aa_seq_2,
                max_attempts_second_pass,
                second_pass_alignment_threshold_2,
                second_pass_alignment_threshold_1,
                second_pass_blosum_threshold_2,
                second_pass_blosum_threshold_1
            )
            if df is not None:
                final_results.append({
                    'model_number': model_num,
                    'pass': 'second',
                    'iteration': i + 1,
                    'modified_sequence_1': df['modified_sequence'].iloc[1],
                    'modified_sequence_2': df['modified_sequence'].iloc[0],
                    'overlap_length': df['overlap_length'].iloc[0],
                    'translated_aa_seq_1': df['translated_sequence'].iloc[1],
                    'translated_aa_seq_2': df['translated_sequence'].iloc[0],
                    'truncated_norm_align_score_1': round(df['truncated_norm_align_score_2'].iloc[0], 2),
                    'truncated_norm_align_score_2': round(df['truncated_norm_align_score_1'].iloc[0], 2),
                    'truncated_norm_align_avg_score': round(
                        (df['truncated_norm_align_score_1'].iloc[0] + df['truncated_norm_align_score_2'].iloc[0]) / 2, 2
                    ),
                    'blosum62_sim_score_1': round(df['blosum62_sim_score_2'].iloc[0], 2),
                    'blosum62_sim_score_2': round(df['blosum62_sim_score_1'].iloc[0], 2),
                    'blosum_sim_score_len_norm_1': round(df['blosum62_sim_score_2_len_norm'].iloc[0], 2),
                    'blosum_sim_score_len_norm_2': round(df['blosum62_sim_score_1_len_norm'].iloc[0], 2),
                    'blosum62_sum_score': round(df['blosum62_sim_score_1'].iloc[0] + df['blosum62_sim_score_2'].iloc[0], 2),
                    'protsub_sim_score_1': round(df['protsub_sim_score_2'].iloc[0], 2),
                    'protsub_sim_score_2': round(df['protsub_sim_score_1'].iloc[0], 2),
                    'protsub_sim_score_len_norm_1': round(df['protsub_sim_score_2_len_norm'].iloc[0], 2),
                    'protsub_sim_score_len_norm_2': round(df['protsub_sim_score_1_len_norm'].iloc[0], 2),
                    'protsub_sum_score': round(df['protsub_sim_score_1'].iloc[0] + df['protsub_sim_score_2'].iloc[0], 2)
                })

        for h in rev_handles: h.remove()

    final_results_df = pd.DataFrame(final_results + all_results)

    # post-processing
    if is_dna_sequence(seq_1) and is_dna_sequence(seq_2):
        final_results_df['integrated_seq_1'] = final_results_df.apply(
            lambda r: integrate_modified_sequence(seq_1, r['modified_sequence_1']), axis=1)
        final_results_df['integrated_seq_2'] = final_results_df.apply(
            lambda r: integrate_modified_sequence(seq_2, r['modified_sequence_2']), axis=1)
        final_results_df['translated_integrated_seq_1'] = final_results_df['integrated_seq_1'].apply(
            lambda s: str(Seq(s).translate()))
        final_results_df['translated_integrated_seq_2'] = final_results_df['integrated_seq_2'].apply(
            lambda s: str(Seq(s).translate()))
    else:
        final_results_df['integrated_seq_1'] = final_results_df.apply(
            lambda r: integrate_modified_sequence(back_translated_aa_seq_1, r['modified_sequence_1']), axis=1)
        final_results_df['integrated_seq_2'] = final_results_df.apply(
            lambda r: integrate_modified_sequence(back_translated_aa_seq_2, r['modified_sequence_2']), axis=1)
        final_results_df['translated_integrated_seq_1'] = final_results_df['integrated_seq_1'].apply(
            lambda s: str(Seq(s).translate()))
        final_results_df['translated_integrated_seq_2'] = final_results_df['integrated_seq_2'].apply(
            lambda s: str(Seq(s).translate()))

    print("\nFinal Results DataFrame:")
    print(final_results_df)
    return final_results_df


In [ ]:
from typing import Dict, List, Optional
import random

# Standard genetic code (NCBI #1) mapping AA -> possible codons
CODONS: Dict[str, List[str]] = {
    'A': ["GCT", "GCC", "GCA", "GCG"],
    'R': ["CGT", "CGC", "CGA", "CGG", "AGA", "AGG"],
    'N': ["AAT", "AAC"],
    'D': ["GAT", "GAC"],
    'C': ["TGT", "TGC"],
    'Q': ["CAA", "CAG"],
    'E': ["GAA", "GAG"],
    'G': ["GGT", "GGC", "GGA", "GGG"],
    'H': ["CAT", "CAC"],
    'I': ["ATT", "ATC", "ATA"],
    'L': ["TTA", "TTG", "CTT", "CTC", "CTA", "CTG"],
    'K': ["AAA", "AAG"],
    'M': ["ATG"],  # start Met
    'F': ["TTT", "TTC"],
    'P': ["CCT", "CCC", "CCA", "CCG"],
    'S': ["TCT", "TCC", "TCA", "TCG", "AGT", "AGC"],
    'T': ["ACT", "ACC", "ACA", "ACG"],
    'W': ["TGG"],
    'Y': ["TAT", "TAC"],
    'V': ["GTT", "GTC", "GTA", "GTG"],
    '*': ["TAA", "TAG", "TGA"],  # stop
    # Non-standard/ambiguous inputs handled below
}

# Very simple default codon usage (weights sum to 1 per amino acid). Replace with species-specific if desired.
DEFAULT_USAGE: Dict[str, Dict[str, float]] = {
    aa: {c: 1.0 / len(cs) for c in cs} for aa, cs in CODONS.items()
}
# Example: bias a few toward common bacterial codons (optional tweak)
DEFAULT_USAGE['L'] = {'CTG': 0.45, 'TTG': 0.15, 'TTA': 0.05, 'CTC': 0.1, 'CTA': 0.05, 'CTT': 0.2}
DEFAULT_USAGE['A'] = {'GCT': 0.18, 'GCC': 0.4, 'GCA': 0.23, 'GCG': 0.19}
DEFAULT_USAGE['G'] = {'GGT': 0.16, 'GGC': 0.41, 'GGA': 0.23, 'GGG': 0.20}
DEFAULT_USAGE['R'] = {'CGT': 0.2, 'CGC': 0.35, 'CGA': 0.07, 'CGG': 0.07, 'AGA': 0.16, 'AGG': 0.15}

def _gc_fraction(dna: str) -> float:
    if not dna: return 0.0
    g = dna.count('G'); c = dna.count('C')
    return (g + c) / len(dna)

def _validate_usage(codon_usage: Dict[str, Dict[str, float]]) -> None:
    for aa, weights in codon_usage.items():
        if aa not in CODONS:  # allow only known keys
            continue
        allowed = set(CODONS[aa])
        if set(weights.keys()) - allowed:
            bad = set(weights.keys()) - allowed
            raise ValueError(f"Codon usage for {aa} contains invalid codons: {bad}")
        s = sum(weights.values())
        if s <= 0:
            raise ValueError(f"Codon usage weights for {aa} must sum to > 0")
        # Normalize in-place
        for k in weights:
            weights[k] /= s

def reverse_translate(
    aa_seq: str,
    *,
    strategy: str = "most_frequent",         # "most_frequent" | "random_weighted" | "gc_balanced"
    codon_usage: Optional[Dict[str, Dict[str, float]]] = None,
    target_gc: Optional[float] = None,       # used only if strategy == "gc_balanced"
    gc_tolerance: float = 0.01,              # how aggressively to steer GC
    add_start: bool = False,                 # prepend ATG (overrides first AA only if you want to force ATG at start)
    end_with_stop: bool = False,             # append a stop codon
    start_codon: str = "ATG",                # if add_start=True
    stop_codons_priority: Optional[List[str]] = None,  # order for '*' or end stop; e.g., ["TAA","TGA","TAG"]
    rng: Optional[random.Random] = None,
    case: str = "upper"                      # "upper" or "lower"
) -> str:
    """
    Reverse-translate an amino-acid sequence to a DNA sequence.

    Ambiguity handling:
      - 'X': choose among all sense codons (excludes stops) via strategy/usage
      - 'B' (D/N): choose from D or N codons; 'Z' (E/Q): from E or Q codons; 'J' (L/I): from L or I codons
      - '*': choose a stop (or use stop_codons_priority)
      - 'U' treated as 'C' (selenocysteine not supported here)
    """
    if rng is None:
        rng = random.Random()

    seq = aa_seq.strip().upper()
    # Map a few ambiguous cases to allowed sets
    ambiguous_map: Dict[str, List[str]] = {
        'X': sum([v for k, v in CODONS.items() if k not in ('*')], []),
        'B': CODONS['D'] + CODONS['N'],
        'Z': CODONS['E'] + CODONS['Q'],
        'J': CODONS['L'] + CODONS['I'],
    }

    # Prepare codon usage
    usage = {aa: w.copy() for aa, w in (codon_usage or DEFAULT_USAGE).items()}
    # Ensure all keys exist
    for aa, cods in CODONS.items():
        usage.setdefault(aa, {c: 1.0 / len(cods) for c in cods})
    _validate_usage(usage)

    def pick_most_frequent(candidates: List[str], aa_for_usage: Optional[str]) -> str:
        if aa_for_usage and aa_for_usage in usage:
            weights = usage[aa_for_usage]
            # Filter to candidate set
            best = max(candidates, key=lambda c: weights.get(c, 0.0))
            return best
        # Fallback: just first candidate
        return candidates[0]

    def pick_weighted(candidates: List[str], aa_for_usage: Optional[str]) -> str:
        if aa_for_usage and aa_for_usage in usage:
            weights = [usage[aa_for_usage].get(c, 0.0) for c in candidates]
            s = sum(weights)
            if s <= 0:
                # Equal weights fallback
                idx = rng.randrange(len(candidates))
                return candidates[idx]
            # Normalize
            weights = [w / s for w in weights]
            r = rng.random()
            cum = 0.0
            for c, w in zip(candidates, weights):
                cum += w
                if r <= cum:
                    return c
            return candidates[-1]
        # Equal weight fallback
        return candidates[rng.randrange(len(candidates))]

    def pick_gc_balanced(candidates: List[str], aa_for_usage: Optional[str], built: str) -> str:
        # If no target, fallback to most_frequent
        if target_gc is None:
            return pick_most_frequent(candidates, aa_for_usage)
        # Score each candidate by how close the resulting GC would be to target;
        # break ties using usage weight (prefer more common codons).
        best_codon, best_score, best_usage = None, float("inf"), -1.0
        for c in candidates:
            gc_next = _gc_fraction(built + c)
            score = abs(gc_next - target_gc)
            u = usage.get(aa_for_usage, {}).get(c, 0.0) if aa_for_usage else 0.0
            # Prefer closer to target; on ties, higher usage
            if (score + gc_tolerance) < best_score or (abs(score - best_score) <= gc_tolerance and u > best_usage):
                best_codon, best_score, best_usage = c, score, u
        return best_codon or pick_most_frequent(candidates, aa_for_usage)

    def codon_for_symbol(sym: str, built: str) -> str:
        if sym == 'U':  # treat selenocysteine as cysteine here (no SEC machinery modeled)
            sym = 'C'
        if sym == '*':
            candidates = stop_codons_priority or CODONS['*']
            # For stops, just take priority order
            return candidates[0]
        if sym in CODONS:
            candidates = CODONS[sym]
            aa_key = sym
        elif sym in ambiguous_map:
            candidates = ambiguous_map[sym]
            aa_key = None  # usage not meaningful across mixed AAs
        else:
            raise ValueError(f"Unsupported amino-acid symbol: '{sym}'")

        if strategy == "most_frequent":
            return pick_most_frequent(candidates, aa_key)
        elif strategy == "random_weighted":
            return pick_weighted(candidates, aa_key)
        elif strategy == "gc_balanced":
            return pick_gc_balanced(candidates, aa_key, built)
        else:
            raise ValueError("strategy must be one of {'most_frequent','random_weighted','gc_balanced'}")

    dna_chunks: List[str] = []

    # Optional forced start codon
    if add_start:
        dna_chunks.append(start_codon.upper())

    for i, aa in enumerate(seq):
        # If add_start and the first AA is M, you may want to skip picking another codon for position 1.
        if i == 0 and add_start and aa == 'M':
            continue
        dna_chunks.append(codon_for_symbol(aa, "".join(dna_chunks)))

    if end_with_stop and (not seq or seq[-1] != '*'):
        stop_choice = (stop_codons_priority or CODONS['*'])[0]
        dna_chunks.append(stop_choice)

    dna = "".join(dna_chunks)
    if case == "lower":
        dna = dna.lower()
    return dna


def get_bracket_positions_logits(sequence_with_brackets: str) -> List[Tuple[int, str]]:
    """
    Extract (start_index, bracketed_aa) pairs where index 0 is the amino acid
    that is 104 residues in from the C-terminus. Indices increase toward the N-terminus.

    Keeps any bracket that overlaps the region within min_allowed AA from the C-terminal,
    and trims the N-terminal side if part of the bracket is outside that zone.
    """
    seq2_len = sum(1 for ch in sequence_with_brackets if ch not in "[]")
    anchor_i = seq2_len - 104
    if anchor_i < 0:
        raise ValueError(f"Sequence too short ({seq2_len} aa) to be 104 in from tail.")

    def map_index(i: int) -> int:
        return i - anchor_i

    min_allowed = (model_numbers[0] // 3) - 2  # AA from C-terminal

    bracketed_amino_acids: List[Tuple[int, str]] = []
    pos_with_brackets = 0
    i_seq2 = 0  # AA index in unbracketed seq

    while pos_with_brackets < len(sequence_with_brackets):
        ch = sequence_with_brackets[pos_with_brackets]

        if ch == '[':
            start_index_unbr = i_seq2
            pos_with_brackets += 1
            content = []

            while (pos_with_brackets < len(sequence_with_brackets) and
                   sequence_with_brackets[pos_with_brackets] != ']'):
                content.append(sequence_with_brackets[pos_with_brackets])
                pos_with_brackets += 1
                i_seq2 += 1

            if pos_with_brackets >= len(sequence_with_brackets):
                raise ValueError("Unmatched '[' in sequence.")

            pos_with_brackets += 1
            end_index_unbr = i_seq2 - 1

            # Distances from C-terminal for each residue in this bracket
            distances = [seq2_len - (start_index_unbr + k) for k in range(len(content))]

            # Find first AA within allowed zone
            keep_start_idx = None
            for idx, dist in enumerate(distances):
                if dist <= min_allowed:
                    keep_start_idx = idx
                    break

            if keep_start_idx is not None:
                trimmed_content = ''.join(content[keep_start_idx:])
                start_index_mapped = map_index(start_index_unbr + keep_start_idx)
                bracketed_amino_acids.append((start_index_mapped, trimmed_content))

        elif ch == ']':
            raise ValueError("Unmatched ']' in sequence.")
        else:
            pos_with_brackets += 1
            i_seq2 += 1

    return bracketed_amino_acids


NUC_TO_NUM = {"A": 1, "T": 2, "G": 3, "C": 4}

## Testing modified approach
def bracket_aas_to_logits_spec(
    sequence_with_brackets: str,
    *,
    strategy: str = "random_weighted",
    codon_usage: Optional[Dict[str, Dict[str, float]]] = None,
    target_gc: Optional[float] = None,
    rng: Optional[random.Random] = None
) -> Dict[int, int]:
    """
    Convert bracketed AA segments to a fixed_logits_spec dict where keys are
    nucleotide positions (0-based) and values are numeric codes:
        T=1, A=2, G=3, C=4
    """
    positions: List[Tuple[int, str]] = get_bracket_positions_logits(sequence_with_brackets)
    fixed_logits_spec: Dict[int, int] = {}

    for start_idx, aa_seg in positions:
        dna_seq = reverse_translate(
            aa_seg,
            strategy=strategy,
            codon_usage=codon_usage,
            target_gc=target_gc,
            rng=rng,
            case="upper"
        )
        # Map AA start to nucleotide start
        nuc_start = start_idx * 3
        for offset, base in enumerate(dna_seq):
            fixed_logits_spec[nuc_start + offset] = NUC_TO_NUM[base]

    return fixed_logits_spec



COMPLEMENT = str.maketrans('ATGC', 'TACG')

def reverse_complement(seq: str) -> str:
    """Return reverse complement of DNA sequence."""
    return seq.translate(COMPLEMENT)[::-1]



def bracket_aas_to_logits_spec_flipped(
    sequence_with_brackets: str,
    *,
    strategy: str = "random_weighted",
    codon_usage: Optional[Dict[str, Dict[str, float]]] = None,
    target_gc: Optional[float] = None,
    rng: Optional[random.Random] = None,
    tok_len: int = 315,
    frame_shift: Optional[int] = None   # 0..2; if None we derive from model_index
) -> Dict[int, int]:
    """
    Map bracketed AA segments from the SECOND sequence to fixed logits on the FORWARD token axis,
    using reverse complement AND mirrored positions.

    For a segment starting at nt_start with length L, the flipped start is:
        start_flipped = tok_len - (nt_start + L)
    Then we write rc(dna) left-to-right starting at start_flipped.
    """
    positions: List[Tuple[int, str]] = get_bracket_positions_logits(sequence_with_brackets)
    fixed_logits_spec: Dict[int, int] = {}

    # Optional small shift (0..2) to keep codon frames aligned between strands.
    if frame_shift is None:
        if model_numbers[0] is not None:

            frame_shift = ((tok_len) - model_numbers[0])
        else:
            frame_shift = 0

    for aa_start, aa_seg in positions:
        # 1) translate, 2) RC
        dna = reverse_translate(
            aa_seg,
            strategy=strategy,
            codon_usage=codon_usage,
            target_gc=target_gc,
            rng=rng,
            case="upper",
        )
        rc = reverse_complement(dna)
        Lnt = len(rc)

        # 3) original nt start (left-indexed AA positions)
        nt_start = aa_start * 3

        # 4) mirror about the right edge + optional frame shift; NO modulo wrap
        start_flipped = tok_len - (nt_start + Lnt) + frame_shift

        # 5) bounds check (fail fast instead of silently wrapping)
        if start_flipped < 0 or start_flipped + Lnt > tok_len:
            raise ValueError(
                f"Flipped segment out of bounds: start={start_flipped}, len={Lnt}, tok_len={tok_len} "
                f"(aa_start={aa_start}, aa_len={len(aa_seg)}, frame_shift={frame_shift})"
            )

        for o, base in enumerate(rc):
            fixed_logits_spec[start_flipped + o] = NUC_TO_NUM[base]

    return fixed_logits_spec




In [ ]:
#This code is used to generate an ESM substitution matrix. It then allows direct comparison of two aa sequences given its matrix.

import pandas as pd
import subprocess
import pandas as pd
import os
import tempfile
import shutil


class ESMSubstitutionMatrix:
    def __init__(self, filepath):
        # Load CSV
        df = pd.read_csv(filepath)
        df = df.rename(columns={"Unnamed: 0": "PosRes"})
        
        # Extract position and wild-type residue
        df["Position"] = df["PosRes"].str.extract(r"(\d+)$").astype(int)
        df["Residue"] = df["PosRes"].str.extract(r"^([A-Z])")
        
        # Store for lookup
        self.df = df.set_index(["Position", "Residue"])
        self.amino_acids = [c for c in df.columns if c not in ["PosRes", "Position", "Residue"]]
        
        # Also reconstruct the wild-type sequence from the row labels
        wt = df.sort_values("Position")["Residue"].tolist()
        self.wt_sequence = "".join(wt)

    def score_position(self, position, wt_residue, candidate_residue):
        """
        Score a substitution at a single position.
        Returns 0.0 if wild-type preserved, negative otherwise.
        """
        return self.df.loc[(position, wt_residue), candidate_residue]

    def score_sequence(self, candidate_sequence):
        """
        Score an entire protein sequence relative to the wild-type.
        Length must match.
        """
        if len(candidate_sequence) != len(self.wt_sequence):
            raise ValueError("Candidate sequence length must match wild-type sequence length.")
        
        total_score = 0.0
        scores = []
        
        for pos, (wt_res, cand_res) in enumerate(zip(self.wt_sequence, candidate_sequence), start=1):
            score = self.score_position(pos, wt_res, cand_res)
            scores.append(score)
            total_score += score
        
        return total_score, scores
    

def generate_esmscan_matrix(seq,
                            model_path="/home/jason/outputdir/python_projects/esm1v_t33_650M_UR90S_1.pt",
                            esmscan_path="/home/jason/outputdir/python_projects/ESM-Scan/esmscan.py"):
    """
    Run ESM-scan once on a sequence and return the substitution matrix as a DataFrame.
    """
    tmpdir = tempfile.mkdtemp()
    prefix = os.path.join(tmpdir, "run")

    try:
        cmd = [
            "python", esmscan_path,
            "--model-location", model_path,
            "--sequence", seq,
            "--output-prefix", prefix
        ]
        result = subprocess.run(cmd, capture_output=True, text=True)

        if result.returncode != 0:
            print("=== ESM-scan STDERR ===")
            print(result.stderr)
            print("=== ESM-scan STDOUT ===")
            print(result.stdout)
            raise RuntimeError(f"ESM-scan failed with code {result.returncode}")

        # Load CSV into a DataFrame
        csv_path = prefix + "-res-in-matrix.csv"
        df = pd.read_csv(csv_path)
        df = df.rename(columns={"Unnamed: 0": "PosRes"})
        df["Position"] = df["PosRes"].str.extract(r"(\d+)$").astype(int)
        df["Residue"] = df["PosRes"].str.extract(r"^([A-Z])")
        df = df.set_index(["Position", "Residue"])

        return df.copy()  # return as reusable DataFrame

    finally:
        shutil.rmtree(tmpdir)


def esm_scan_score_sequence(seq1, seq2, matrix_df):
    """
    Given a reference sequence (seq1), a candidate (seq2), and
    the ESM-scan substitution matrix DataFrame for seq1,
    return the total substitution score and per-residue scores.
    """
    if len(seq1) != len(seq2):
        raise ValueError("Sequences must be the same length for position-specific scoring.")

    total_score = 0.0
    per_residue_scores = []
    for pos, (wt, cand) in enumerate(zip(seq1, seq2), start=1):
        try:
            score = matrix_df.loc[(pos, wt), cand]
        except KeyError:
            raise KeyError(f"Substitution {wt}->{cand} at position {pos} not found in matrix.")
        per_residue_scores.append(score)
        total_score += score

    return total_score, per_residue_scores

def esm_percentage_score(seq1, seq2, matrix_df):
    total = 0.0
    norm_total = 0.0
    aa_cols = [c for c in matrix_df.columns if len(c) == 1 and c.isalpha()]  # only single-letter AA columns

    for pos, (wt, cand) in enumerate(zip(seq1, seq2), start=1):
        # actual score for this substitution
        obs = matrix_df.loc[(pos, wt), cand]

        # worst score at this position (numeric only)
        worst = matrix_df.loc[(pos, wt), aa_cols].min()

        # normalize: map [worst .. 0] → [0 .. 1]
        norm = (obs - worst) / (0 - worst)
        total += norm
        norm_total += 1

    return float(round(100 * (total / norm_total), 2))



In [ ]:
#!/usr/bin/env python3

# This is the most current version, updated ss score variable name

"""
Batch runner for amino-acid pair table (CSV/XLSX) with ESM integration and
configurable secondary-structure (SS) evaluation behavior.

This script was designed to support overlapping gene design experiments, where
two amino acid sequences (potentially overlapping in different reading frames)
are optimized and scored on multiple metrics:
    - Secondary structure (SS) similarity
    - Amino acid alignment identity / substitution scores
    - Embedding similarity from ESM-2 (mean pooled embeddings)
    - Contact map similarity (via SSIM, Structural Similarity Index)

The workflow allows for "deferred" computation of heavy structural predictions,
so users can control runtime costs. Caching is heavily used to prevent
recomputing SS or embeddings for the same sequences across rows.

"""

import os
import re
import io
import math
import hashlib
import sys
from datetime import datetime
import pandas as pd
from contextlib import redirect_stdout
from typing import List, Dict, Any, Iterable, Tuple, Optional

# Numerical / ML deps
import numpy as np
import torch
from skimage.metrics import structural_similarity as ssim

# ===== NEW: global flag controlling INITIAL vs SUBSEQUENT behaviour =====
# Global flag to track if any prior window had success
had_successful_prediction = False


# ESM (facebookresearch/esm)
try:
    import esm
    _ESM_AVAILABLE = True
except Exception as _e:
    esm = None
    _ESM_AVAILABLE = False
    print(f"[WARN] ESM not available: {_e}")

model_numbers = [311]

# ==============================
# Config
# ==============================
# Secondary-structure control:
DO_SS = True                    # If True: run SS for all candidates (original heavy behaviour)
EVAL_SS_ON_EARLY = True          # If DO_SS==False: evaluate SS for the early-stop candidate (if found)
EVAL_SS_AT_ATTEMPT_END = True    # If DO_SS==False: evaluate SS for the chosen final candidate at attempt end

# ESM thresholds (0..100)
ESM_EARLY_THRESHOLD = 92.0
ESM_SUCCESS_THRESHOLD = 92.0

# Whether to reset heavy caches per-row (default False to avoid recomputing across many rows)
RESET_CACHES_PER_ROW = True

# ==============================
# IO / Utility functions
# ==============================

def windows_to_wsl_path(p: str) -> str:
    p = p.strip().strip('"').strip("'")
    if re.match(r'^[A-Za-z]:\\', p):
        drive = p[0].lower()
        rest = p[2:].replace('\\', '/')
        return f"/mnt/{drive}{rest}"
    return p.replace('\\', '/')

def load_pairs_table(path: str) -> pd.DataFrame:

    """
    Load an input table containing pair rows.
    Accepts .xlsx/.xls and .csv. Verifies required columns exist:
        - aa_seq_1
        - aa_seq_2
        - aa_seq_1_brackets ###If no brackets are defind, this can be the same as aa_seq_1
        - aa_seq_2_brackets ###If no brackets are defind, this can be the same as aa_seq_2
        
    Returns:
        pandas.DataFrame loaded from file.
    Raises:
        ValueError for unsupported extensions.
        KeyError if required columns missing.
    """

    ext = os.path.splitext(path)[1].lower()
    if ext in (".xlsx", ".xls"):
        df = pd.read_excel(path)
    elif ext in (".csv",):
        df = pd.read_csv(path)
    else:
        raise ValueError(f"Unsupported file type: {ext}")
    required = {"aa_seq_1", "aa_seq_2", "aa_seq_1_brackets", "aa_seq_2_brackets"}
    missing = required - set(df.columns)
    if missing:
        raise KeyError(f"Missing required columns: {sorted(missing)}")
    return df

# ───────────────────────────────────────────────
# 1. Global caches and metrics (persist across rows)
# ───────────────────────────────────────────────
SSCACHE: Dict[str, Any] = {}          # sequence -> secondary structure prediction (raw)
METRICS = {
    "ss_total_requests": 0,
    "ss_unique_requests": 0,
    "ss_cache_hits": 0,
    "pairs_requested": 0,
    "pairs_unique": 0,
}
all_pairs_records: List[Dict[str, Any]] = []

# Helper to update records with deferred SS results
def update_all_pairs_records_with_ss(target_seq1: str, target_seq2: str,
                                     attempt: int, window: int,
                                     s1_pct: float, s2_pct: float,
                                     ss_pred_a: Optional[str] = None,
                                     ss_pred_b: Optional[str] = None,
                                     avg_pct: Optional[float] = None) -> int:
    """
    Update all_pairs_records in-place for records matching:
      - translated_integrated_seq_1 == target_seq1 (trimmed, trailing '*' removed)
      - translated_integrated_seq_2 == target_seq2
      - attempt == attempt
      - window == window

    Writes:
      ss_score_1, ss_score_2, ss_score_avg, ss_pred_1, ss_pred_2

    Returns:
      number of records updated (int)

    Note:
      Matches on normalized strings to tolerate trailing '*' and whitespace.
    """
    updated = 0
    t1 = str(target_seq1).strip().rstrip("*")
    t2 = str(target_seq2).strip().rstrip("*")
    for rec in all_pairs_records:
        if (str(rec.get("translated_integrated_seq_1", "")).strip().rstrip("*") == t1
                and str(rec.get("translated_integrated_seq_2", "")).strip().rstrip("*") == t2
                and int(rec.get("attempt", -1)) == int(attempt)
                and int(rec.get("window", -999)) == int(window)):
            rec["ss_score_1"] = float(s1_pct)
            rec["ss_score_2"] = float(s2_pct)
            rec["ss_score_avg"] = float(avg_pct) if avg_pct is not None else (float(s1_pct) + float(s2_pct)) / 2.0
            rec["ss_pred_1"] = ss_pred_a
            rec["ss_pred_2"] = ss_pred_b
            updated += 1
    return updated

# ───────────────────────────────────────────────
# 2. Secondary structure caching (user must supply batch_structure_prediction_wrapper)
# ───────────────────────────────────────────────

def ss_predict_cached(seq_list: Iterable[str]) -> List[Any]:
    """
    Deduplicate and cache SS predictions for a batch of sequences.

    Workflow:
      - Count requests in METRICS
      - Find which sequences are cache misses
      - Call batch_structure_prediction_wrapper(misses, output_dir) for misses
      - Store returned preds in SSCACHE
      - Return predictions in input order (pulling from cache)

    Requirements:
      - The function batch_structure_prediction_wrapper(misses, output_dir) must exist
        in the user's environment and return predictions in the same order as misses.
      - `output_dir` must be defined in the calling context (this module uses it as a global)

    Returns:
      List of predictions aligned to seq_list order. Predictions may be None for failures.
    """
    global METRICS, SSCACHE, output_dir
    seq_list = list(seq_list)
    METRICS["ss_total_requests"] += len(seq_list)
    misses = []
    for s in seq_list:
        if s in SSCACHE:
            METRICS["ss_cache_hits"] += 1
        elif s not in misses:
            # ensure we only request each distinct miss once
            misses.append(s)
    if misses:
        METRICS["ss_unique_requests"] += len(misses)
        preds = batch_structure_prediction_wrapper(misses, output_dir)
        for s, p in zip(misses, preds):
            SSCACHE[s] = p
    # Return cached preds in the same order as the requested list        
    return [SSCACHE[s] for s in seq_list]

# ───────────────────────────────────────────────
# 3. ESM integration (embeddings + contact maps) and helpers
# ───────────────────────────────────────────────
ESM_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ESM_MODEL_NAME = "esm2_t33_650M_UR50D"  # pick smaller if memory limited
ESM_LAYER = 33
ESM_BATCH_SIZE = 8

# Performance knobs
USE_AUTOCast_FP16_IF_CUDA = True
USE_MODEL_FP16 = False

# Global ESM caches (persist across rows by default)
ESM_EMB_CACHE: Dict[str, np.ndarray] = {}   # sequence -> embedding (mean-pooled)
ESM_CONT_CACHE: Dict[str, np.ndarray] = {}  # sequence -> contact map (L_res x L_res)

def _load_esm_model():
    """
    Lazily loads the selected ESM model and returns (model, alphabet, batch_converter)
    - If esm.pretrained has a direct attribute for the chosen name, use it, otherwise call known loader.
    - Places model on ESM_DEVICE and sets eval() mode.
    - Optionally converts model to half precision if USE_MODEL_FP16 is True and a CUDA device is present.

    Raises:
      RuntimeError if ESM not available.

    Returns:
      (model, alphabet, batch_converter)
    """
    if not _ESM_AVAILABLE:
        raise RuntimeError("ESM not installed; pip install fair-esm")
    loader = getattr(esm.pretrained, ESM_MODEL_NAME, None)
    if loader is None:
        model, alphabet = esm.pretrained.esm2_t33_650M_UR50D()
    else:
        model, alphabet = loader()
    batch_converter = alphabet.get_batch_converter()
    model = model.eval().to(ESM_DEVICE)
    if USE_MODEL_FP16 and ESM_DEVICE.type == "cuda":
        model = model.half()
    return model, alphabet, batch_converter

def _chunked(lst, n):
    """
    Simple generator to yield chunks (sublists) of size up to `n`.
    Keeps memory use bounded for batched processing.
    """
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

def _meanpool_token_representations(token_reps: torch.Tensor, tokens: torch.Tensor, padding_idx: int) -> torch.Tensor:
    """
    Mean-pool token representations while ignoring padding and special tokens.
    - token_reps: (B, L, C) tensor of token-level representations
    - tokens: (B, L) long tensor with token indices (to detect padding)
    - padding_idx: alphabet.padding_idx value

    Behavior:
      - Excludes first and last positions (often BOS/EOS in ESM)
      - Excludes any padding positions
      - Returns (B, C) pooled tensor as sums / (counts) per sequence
      - clamps count to at least 1 to avoid divide-by-zero
    """
    B, L, C = token_reps.shape
    nonpad = (tokens != padding_idx)
    idx = torch.arange(L, device=tokens.device)
    valid_pos = nonpad & (idx[None, :] > 0) & (idx[None, :] < (L - 1))
    sums = (token_reps * valid_pos.unsqueeze(-1)).sum(dim=1)
    counts = valid_pos.sum(dim=1).clamp(min=1)
    return sums / counts.unsqueeze(-1)

def _contacts_to_numpy(contact_batch: torch.Tensor, lens: torch.Tensor) -> list:
    """
    Convert model contacts output into numpy 2D contact matrices for each sequence.

    The model returns contact tensors with padding included; this extracts
    the residue-residue slice corresponding to the real sequence length (excluding special tokens).
    Returns:
      list of numpy arrays (L_res x L_res), one per sequence in batch
    """
    outs = []
    for cm, le in zip(contact_batch, lens):
        L = int(le.item()) - 2
        L = max(L, 1)
        cm_res = cm[1:1+L, 1:1+L]
        outs.append(cm_res.detach().float().cpu().numpy())
    return outs

def esm_features_cached(seq_list: Iterable[str], model_alphabet_converter=None):
    """
    Ensure ESM_EMB_CACHE and ESM_CONT_CACHE entries exist for the sequences in seq_list.

    Behavior:
      - Normalizes sequences (strip spaces, trailing '*', skip empty)
      - Finds cache misses
      - Loads model via _load_esm_model() unless model_alphabet_converter passed
      - Batches inference, extracts:
          - mean-pooled token representations (embedding)
          - contact maps (2D)
      - Stores results in ESM_EMB_CACHE and ESM_CONT_CACHE keyed by sequence string.

    Note:
      - This function intentionally populates caches and returns None.
      - On failure for a chunk, it will raise (caller may catch).
    """
    global ESM_EMB_CACHE, ESM_CONT_CACHE
    seq_list = [s.strip().replace(" ", "").rstrip("*") for s in seq_list if isinstance(s, str) and len(s)]
    misses = [s for s in seq_list if (s not in ESM_EMB_CACHE) or (s not in ESM_CONT_CACHE)]
    if not misses:
        return

    if model_alphabet_converter is None:
        mac = _load_esm_model()
    else:
        mac = model_alphabet_converter
    model, alphabet, batch_converter = mac

    with torch.no_grad():
        for chunk in _chunked(misses, ESM_BATCH_SIZE):
            data = [(f"seq_{i}", s) for i, s in enumerate(chunk)]
            labels, batch_strs, batch_tokens = batch_converter(data)
            batch_tokens = batch_tokens.to(ESM_DEVICE)
            lens = (batch_tokens != alphabet.padding_idx).sum(1)

            if USE_AUTOCast_FP16_IF_CUDA and ESM_DEVICE.type == "cuda":
                # modern API — avoids the FutureWarning and is forward compatible
                with torch.amp.autocast(device_type="cuda", dtype=torch.float16):
                    out = model(batch_tokens, repr_layers=[ESM_LAYER], return_contacts=True)
            else:
                out = model(batch_tokens, repr_layers=[ESM_LAYER], return_contacts=True)

            reps = out["representations"][ESM_LAYER]
            contacts = out["contacts"]

            pooled = _meanpool_token_representations(reps, batch_tokens, alphabet.padding_idx)
            pooled = pooled.detach().to(torch.float32).cpu().numpy()
            c_maps = _contacts_to_numpy(contacts, lens)

            for s, emb, cm in zip(chunk, pooled, c_maps):
                ESM_EMB_CACHE[s] = emb
                ESM_CONT_CACHE[s] = cm

def _crop_to_min(a: np.ndarray, b: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """
    Crop two square matrices to the same minimal square size using their first dimension.
    Useful because SSIM and other matrix comparisons require shapes to align.
    """
    m = min(a.shape[0], b.shape[0])
    return a[:m, :m], b[:m, :m]

def _safe_ssim(cm_o: np.ndarray, cm_c: np.ndarray) -> float:
    """
    Compute SSIM between two contact maps with basic safety guards:
      - Crops to minimum common size
      - If too small (min_dim < 3) returns a neutral 0.5
      - Chooses win_size based on smallest dimension to avoid invalid window sizes
      - Catches exceptions and returns neutral 0.5 on failure

    Returns:
      SSIM float in [0,1]
    """
    try:
        cm_o, cm_c = _crop_to_min(cm_o, cm_c)
        min_dim = min(cm_o.shape[0], cm_c.shape[0])
        if min_dim < 3:
            return 0.5
        win_size = 7 if min_dim >= 7 else (5 if min_dim >= 5 else 3)
        return float(ssim(cm_o, cm_c, data_range=1.0, win_size=win_size))
    except Exception:
        return 0.5

def clamp01(x, eps=1e-9):
    return float(np.clip(x, eps, 1.0 - eps))

def transform_ssim(x, method="power", gamma=1.0, temp=1.0, batch=None):
    """
    Map a single SSIM x (0..1) to a transformed 0..1 value.
    If x is nan, return nan.
    Methods: "power","logit","linear_minmax","quantile","quantile_norm","tanh_z"
    - batch required for quantile/linear_minmax/tanh_z/quantile_norm
    """
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return np.nan

    x = clamp01(x)

    if method == "power":
        return x ** gamma

    if method == "logit":
        # amplify differences near 0/1 via logit/temperature
        logit = np.log(x / (1.0 - x))
        scaled = logit / float(max(1e-9, temp))
        return float(expit(scaled))

    if method == "linear_minmax":
        if batch is None:
            raise ValueError("batch required for linear_minmax")
        arr = np.asarray([v for v in batch if not (np.isnan(v))], dtype=float)
        if arr.size == 0:
            return x  # nothing to scale
        mn, mx = arr.min(), arr.max()
        if mx <= mn:
            return 0.0 if x <= mn else 1.0
        return float(np.clip((x - mn) / (mx - mn), 0.0, 1.0))

    if method == "quantile":
        if batch is None:
            raise ValueError("batch required for quantile")
        arr = np.asarray(batch, dtype=float)
        pct = float((arr < x).sum()) / max(1, len(arr))
        return pct

    if method == "quantile_norm":
        if batch is None:
            raise ValueError("batch required for quantile_norm")
        arr = np.asarray(batch, dtype=float)
        pct = float((arr < x).sum()) / max(1, len(arr))
        z = norm.ppf(np.clip(pct, 1e-6, 1-1e-6))
        return float(0.5 * (np.tanh(z / (temp if temp > 0 else 1.0)) + 1.0))

    if method == "tanh_z":
        if batch is None:
            return x ** gamma
        arr = np.asarray(batch, dtype=float)
        arr = arr[~np.isnan(arr)]
        if arr.size < 2:
            return x ** gamma
        mu, sigma = arr.mean(), arr.std() if arr.std() > 0 else 1.0
        z = (x - mu) / (sigma * (temp if temp>0 else 1.0))
        return float(0.5 * (np.tanh(z) + 1.0))

    return x

def esm_pair_metrics(orig_seq: str, cand_seq: str) -> Dict[str, float]:
    cm_o = ESM_CONT_CACHE.get(orig_seq, None)
    cm_c = ESM_CONT_CACHE.get(cand_seq, None)

    if cm_o is None or cm_c is None:
        print(f"[DEBUG] Missing contact map(s): "
              f"{'orig_seq missing' if cm_o is None else ''} "
              f"{'cand_seq missing' if cm_c is None else ''}")
        return {"cmap_ssim_100": 50.0, "esm_score_100": 50.0}

    try:
        # Ensure float32 for consistency
        cm_o = np.asarray(cm_o, dtype=np.float32)
        cm_c = np.asarray(cm_c, dtype=np.float32)

        # Fixed range: contact maps assumed normalized to [0,1]
        ssim_raw, _ = ssim(cm_o, cm_c, full=True, data_range=1.0)
        ssim_val = transform_ssim(ssim_raw, method="power", gamma=1.0)

        return {
            "cmap_ssim_100": round(ssim_val * 100.0, 2),
            "esm_score_100": round(ssim_val * 100.0, 2),
        }

    except Exception as e:
        print(f"[ERROR] SSIM computation failed for pair "
              f"({orig_seq[:8]}..., {cand_seq[:8]}...): {e}")
        return {"cmap_ssim_100": 50.0, "esm_score_100": 50.0}

# def esm_pair_metrics(orig_seq: str, cand_seq: str) -> Dict[str, float]:
#     """
#     Compute pairwise ESM-derived metrics for two sequences using cached contact maps.

#     Primary metric: contact-map SSIM scaled to 0..100 and returned under keys:
#       - cmap_ssim_100
#       - esm_score_100 (same as cmap_ssim_100 in this design)

#     If either contact map missing, return neutral 50.0 for both fields.
#     """
#     cm_o = ESM_CONT_CACHE.get(orig_seq, None)
#     cm_c = ESM_CONT_CACHE.get(cand_seq, None)
#     if cm_o is None or cm_c is None:
#         return {"cmap_ssim_100": 50.0, "esm_score_100": 50.0}

#     ssim_val = transform_ssim(cm_o, cm_c)
#     return {
#         "cmap_ssim_100": round(ssim_val * 100.0, 2),
#         "esm_score_100": round(ssim_val * 100.0, 2),
#     }

# -------------------------
# Helper: compute SS for a chosen pair (uses cached ss_predict_cached)
# -------------------------
def _compute_ss_for_pair_and_scores(ori_1_predicted_structure, ori_2_predicted_structure,
                                    seq_a: str, seq_b: str, model_id: int):
    """
    For a chosen candidate pair (seq_a, seq_b):
      - Ensure SS predictions via ss_predict_cached for seq_a and seq_b.
      - Call compare_sequences(...) which performs the SS comparison of candidate vs original predicted structures.
      - Unpack and return the percentage scores and raw SS predictions.

    Returns:
      (s1_pct, s2_pct, avg_pct, ss_pred_a, ss_pred_b)

    Note:
      - compare_sequences must exist in the environment and is expected to return a tuple whose
        positions are unpacked here. This wrapper isolates the caching and common unpack logic.
    """
    seqs = [seq_a, seq_b]
    preds = ss_predict_cached(seqs)
    ss_a, ss_b = preds[0], preds[1]
    *_, s1_pct, _, s2_pct, avg_pct, _ = compare_sequences(
        ori_1_predicted_structure, ori_2_predicted_structure,
        ss_a, ss_b, model_id
    )
    return float(s1_pct), float(s2_pct), float(avg_pct), ss_a, ss_b

# ======================================================================================
# 4. Core single-pair optimization wrapper (with ESM scoring + optional deferred SS eval)
# ======================================================================================
def optimize_pair_and_save(seq1: str, seq2: str, seq1_bracket:str, seq2_bracket:str, row_index_1based: int) -> str:
    """
    Main pipeline for processing a single input row (pair of AA sequences).

    Responsibilities:
      - Prepare sequences and optionally compute original SS predictions.
      - Warm-up/load ESM and populate caches for originals.
      - Iterate over windows and run the inference engine (user-provided run_inference_for_models)
      - Score candidates using a weighted composite (SS, alignment, substitution, ESM)
      - Optionally perform deferred SS computation (if DO_SS==False)
      - Persist all candidate rows to an Excel file named by date/model/row index

    Inputs:
      - seq1, seq2: raw AA sequences from the table (may contain whitespace or trailing '*')
      - seq1_bracket, seq2_bracket: bracket annotations used to set fixed_logits specs
      - row_index_1based: integer for output filename hygiene

    Returns:
      - path to saved .xlsx result output for this row
    """
    # --- local/default params (kept similar to your original script) ---
    
    # --- Declare globals before any assignment ---
    global all_pairs_records, output_dir, METRICS

    had_successful_prediction = False

    # --- Reset per-row records so nothing leaks across rows ---
    all_pairs_records = []

    #model_numbers = [311]
    SUB_MATRIX = "blosum62"
    LOGIT_DROPOUT_RATE = 0.25
    TOK_LEN = 315

    first_pass_iterations = 1
    second_pass_iterations = 75
    first_pass_alignment_threshold_1 = 0.36
    first_pass_alignment_threshold_2 = 0.36
    second_pass_alignment_threshold_1 = 0.36
    second_pass_alignment_threshold_2 = 0.36
    first_pass_blosum_threshold_1 = 0
    first_pass_blosum_threshold_2 = 0
    second_pass_blosum_threshold_1 = 0
    second_pass_blosum_threshold_2 = 0

    max_attempts_first_pass = 325
    max_attempts_second_pass = 200
    inference_mode = "train"
    set_seed = None

    MAX_RETRIES_PER_WINDOW = 3
    APPLY_LOGIT_DROPOUT_IN_EVAL = True

    WINDOW_AA, STRIDE_AA, MODEL_AA = 10, 8, 105
    num_w = math.ceil(MODEL_AA / STRIDE_AA)

    SUCCESS_SS_THRESHOLD = 100.0
    EARLY_SS_THRESHOLD = 100.0
    MAX_TOTAL_RETRIES = 2

    FEEDFORWARD_DROPOUT_INITIAL = 0.1
    ATTENTION_DROPOUT_INITIAL = 0.1
    FEEDFORWARD_DROPOUT_SUBSEQ = 0.0
    ATTENTION_DROPOUT_SUBSEQ = 0.0

    output_dir = working_directory

    # use originals (no brackets here)
    sequence_1_original = seq1.strip().replace(" ", "").strip("*")
    sequence_2_original = seq2.strip().replace(" ", "").strip("*")
    sequence_1_aa_brackets = seq1_bracket.strip().replace(" ", "").strip("*")
    sequence_2_aa_brackets = seq2_bracket.strip().replace(" ", "").strip("*")

    def _preview(s: str, n: int = 120) -> str:
        if not s:
            return "<EMPTY>"
        return s if len(s) <= n else f"{s[:n]}... (len={len(s)})"

    # Print diagnostics to debug delimiter/format problems quickly
    print("[INPUT DIAG] sequence_1_original preview:", _preview(sequence_1_original))
    print("           length:", len(sequence_1_original), "  '*' count:", sequence_1_original.count("*"))
    print("[INPUT DIAG] sequence_2_original preview:", _preview(sequence_2_original))
    print("           length:", len(sequence_2_original), "  '*' count:", sequence_2_original.count("*"))

    print("[INPUT DIAG] sequence_1_aa_brackets preview:", _preview(sequence_1_aa_brackets))
    print("           length:", len(sequence_1_aa_brackets), "  '*' count:", sequence_1_aa_brackets.count("*"))
    print("[INPUT DIAG] sequence_2_aa_brackets preview:", _preview(sequence_2_aa_brackets))
    print("           length:", len(sequence_2_aa_brackets), "  '*' count:", sequence_2_aa_brackets.count("*"))

    # Optionally reset caches per-row (default False)
    global SSCACHE, ESM_EMB_CACHE, ESM_CONT_CACHE
    if RESET_CACHES_PER_ROW:
        SSCACHE = {}
        ESM_EMB_CACHE = {}
        ESM_CONT_CACHE = {}

    print("\n[INFO] Processing sequences:")
    # Precompute originals and optionally SS for originals (only if DO_SS True or if deferred eval will need them later)
    concatenated_sequence_original = process_sequences(sequence_1_original, sequence_2_original)
    print("\n[INFO] Concatenated original sequence (nucleotides):")
    print(concatenated_sequence_original)

    ori_seq_1, ori_seq_2 = split_sequence(concatenated_sequence_original)
    print("\n[INFO] Split sequences (AA):")
    print("  Seq1:", ori_seq_1)
    print("  Seq2:", ori_seq_2)

    ori_1_predicted_structure = None
    ori_2_predicted_structure = None
    if DO_SS:
        ori_1_predicted_structure = predict_secondary_structure(ori_seq_1)
        ori_2_predicted_structure = predict_secondary_structure(ori_seq_2)
        print("\n[INFO] Secondary structure predictions (orig computed):")
        print("  Seq1 secondary structure:", ori_1_predicted_structure)
        print("  Seq2 secondary structure:", ori_2_predicted_structure)
    else:
        print("\n[INFO] Secondary-structure scoring deferred (DO_SS is False).")

    # ESM warm-up for originals (cache them)
    if _ESM_AVAILABLE:
        try:
            mac = _load_esm_model()
            esm_features_cached([ori_seq_1, ori_seq_2], model_alphabet_converter=mac)
        except Exception as _e:
            mac = None
            print(f"[WARN] ESM loading/feature extraction failed; continuing without ESM. Error: {_e}")
    else:
        mac = None
        print("[WARN] ESM not available; ESM-derived scores will be neutral.")

    ### Let's add the ESM scan matrix generation here 

    print("\n[INFO] Generating ESM-scan matrices for original sequences...")
    esm_scan_matrix_1 = generate_esmscan_matrix(ori_seq_1)
    esm_scan_matrix_2 = generate_esmscan_matrix(ori_seq_2)
    print("[INFO] ESM-scan matrices generated successfully.")
    
    #print("\n[INFO] Generating ESM-fold pLDDT values for original sequences...")
    #esm_fold_ori_seq_1 = esm_fold_plddt_output(ori_seq_1, num_recycles=3)
    #esm_fold_ori_seq_2 = esm_fold_plddt_output(ori_seq_2, num_recycles=3)
    #print("[INFO] ESM-fold pLDDT values generated successfully.")

    def _has_real_seq(seq_list):
        """
        Return True if seq_list contains at least one non-empty, non-whitespace, non-'*' sequence.
        Filters out None, NaN, empty strings, whitespace-only strings, and strings consisting only of '*'s.
        """
        for s in seq_list:
            if s is None:
                continue
            # guard against pandas NaN (float)
            if isinstance(s, float) and np.isnan(s):
                continue
            if not isinstance(s, str):
                s = str(s)
            cleaned = s.strip().strip("*").strip()
            if cleaned:
                return True
        return False

    # helper - safe_run_inference uses the inference function you already have
    def safe_run_inference(fwd_range, cur_seq1_input, cur_seq2_input):
        """
        Wraps run_inference_for_models with:
          - retry loop up to MAX_RETRIES_PER_WINDOW
          - dynamic dropout selection depending on whether previous windows succeeded
          - combined fixed logits spec built from bracket annotations

        Returns:
          DataFrame from run_inference_for_models on success, or None if all retries failed.

        Note:
          - run_inference_for_models is expected to be defined in the environment and accept
            the long list of parameters passed here.
        """

        if not had_successful_prediction:
            print("[INFO] Using INITIAL dropout/alignment thresholds.", file=sys.__stdout__)
            ff_drop = FEEDFORWARD_DROPOUT_INITIAL
            att_drop = ATTENTION_DROPOUT_INITIAL
            phase = "INITIAL"
        else:
            print("[INFO] Prior window success detected; using SUBSEQUENT dropout/alignment thresholds for all remaining runs.", file=sys.__stdout__)
            ff_drop = FEEDFORWARD_DROPOUT_SUBSEQ
            att_drop = ATTENTION_DROPOUT_SUBSEQ
            phase = "SUBSEQUENT"

            print(f"[Dropout State: {phase}] Feedforward={ff_drop}, Attention={att_drop}, Sub Matrix={SUB_MATRIX}", file=sys.__stdout__)

        
        # Build combined fixed logits spec once per attempt (keeps same semantics as older code)
        fixed_logits_spec_seq1 = bracket_aas_to_logits_spec(sequence_1_aa_brackets, rng=None)
        fixed_logits_spec_seq2 = bracket_aas_to_logits_spec_flipped(sequence_2_aa_brackets, rng=None)
        combined_logits = {**fixed_logits_spec_seq1, **fixed_logits_spec_seq2}

        for attempt in range(1, MAX_RETRIES_PER_WINDOW + 1):
            buf = io.StringIO()
            try:
                with redirect_stdout(buf):
                    df = run_inference_for_models(
                        model_numbers,
                        cur_seq1_input, cur_seq2_input,
                        max_attempts_first_pass, max_attempts_second_pass,
                        first_pass_alignment_threshold_1, first_pass_alignment_threshold_2,
                        second_pass_alignment_threshold_1, second_pass_alignment_threshold_2,
                        first_pass_blosum_threshold_1, first_pass_blosum_threshold_2,
                        second_pass_blosum_threshold_1, second_pass_blosum_threshold_2,
                        first_pass_iterations, second_pass_iterations,
                        inference_mode,
                        ff_drop, att_drop,
                        set_seed,
                        fc_dropout_p=LOGIT_DROPOUT_RATE,
                        fc_ranges_forward=fwd_range,
                        fc_ranges_reverse=fwd_range,
                        tok_len=TOK_LEN,
                        apply_fc_dropout_in_eval=APPLY_LOGIT_DROPOUT_IN_EVAL,
                        fixed_logits_spec_forward=combined_logits,
                        fixed_logits_spec_reverse=combined_logits,
                        fixed_value=12.0,
                        blend_alpha=None
                    )
                return df
            except Exception as e:
                # print error and optionally the last lines of the buffered stdout for debugging
                debug_out = buf.getvalue()
                print(f"    [{attempt}/{MAX_RETRIES_PER_WINDOW}] Inference attempt failed: {e}")
                if debug_out:
                    # show a short preview of the last buffered line to help debugging
                    last_line = debug_out.strip().splitlines()[-1] if debug_out.strip() else ""
                    print(f"    [debug stdout last line] {last_line}")
                # if we have more attempts, continue to next loop iteration and retry
        # exhausted attempts
        print(f"    All {MAX_RETRIES_PER_WINDOW} inference attempts failed for window {fwd_range}; returning None.")
        return None

    # Main optimisation loop across retry attempts (restarts entire window sweep up to MAX_TOTAL_RETRIES)
    retry_count = 0
    success = False

    start_seq_aa_1 = sequence_1_original
    start_seq_aa_2 = sequence_2_original

    best_overall_scores = (0.0, 0.0)
    best_overall_seq1 = start_seq_aa_1
    best_overall_seq2 = start_seq_aa_2
    best_overall_summaries = []

    while retry_count < MAX_TOTAL_RETRIES:
        retry_count += 1
        print(f"\n=== Optimisation attempt {retry_count}/{MAX_TOTAL_RETRIES} ===")
        summaries = []
        early_halt = False

        for w in range(num_w):
            print(f"Entering window {w+1}/{num_w}")
            start_aa, end_aa = w * STRIDE_AA, min((w * STRIDE_AA) + WINDOW_AA, MODEL_AA)
            fwd_range = [(start_aa * 3, end_aa * 3 - 1)]

            try:
                buf = io.StringIO()
                with redirect_stdout(buf):
                    df = safe_run_inference(fwd_range, start_seq_aa_1, start_seq_aa_2)
            except Exception as e:
                print(f"    Inference failed for window {w+1}: {e}")
                df = None

            if df is None or df.empty:
                print("    All retries failed or no candidates; skipping this window.")
                continue

            seqs1_all = df["translated_aa_seq_1"].astype(str).tolist()
            seqs2_all = df["translated_aa_seq_2"].astype(str).tolist()

            # deduplicate pairs
            unique_pairs, seen_pair_keys = [], set()
            for s1, s2 in zip(seqs1_all, seqs2_all):
                if (s1, s2) not in seen_pair_keys:
                    seen_pair_keys.add((s1, s2))
                    unique_pairs.append((s1, s2))

            METRICS["pairs_requested"] += len(seqs1_all)
            METRICS["pairs_unique"] += len(unique_pairs)

            seqs1 = [p[0] for p in unique_pairs]
            seqs2 = [p[1] for p in unique_pairs]

            # SS predictions: either full (DO_SS True) or deferred (placeholders)
            if DO_SS:
                ss1_list = ss_predict_cached(seqs1)
                ss2_list = ss_predict_cached(seqs2)
            else:
                ss1_list = [None] * len(seqs1)
                ss2_list = [None] * len(seqs2)

            # ESM batch compute features for candidates (contacts + embeddings)
            if _ESM_AVAILABLE and mac is not None:
                try:
                    esm_features_cached(set([ori_seq_1, ori_seq_2] + seqs1 + seqs2), model_alphabet_converter=mac)
                except Exception as _e:
                    print(f"[WARN] ESM feature extraction failed this window: {_e}")
            
            # If we got *any* valid sequence predictions, flip the flag and adjust thresholds for subsequent runs
            if not had_successful_prediction and _has_real_seq(seqs1_all) and _has_real_seq(seqs2_all):

                had_successful_prediction = True
                print("[STATE] Initial sequence prediction detected — future windows will use SUBSEQUENT alignment thresholds.")

                # Adjust thresholds for subsequent passes (example values; adjust as required)
                first_pass_alignment_threshold_1  = 0.80
                first_pass_alignment_threshold_2  = 0.80
                second_pass_alignment_threshold_1 = 0.80
                second_pass_alignment_threshold_2 = 0.80

                print(
                    f"FirstPass=({first_pass_alignment_threshold_1}, {first_pass_alignment_threshold_2}), "
                    f"SecondPass=({second_pass_alignment_threshold_1}, {second_pass_alignment_threshold_2})"
                )

            scores = []
            best_idx = None
            best_combined = -1e9
            early_idx = None

            # weight schedule
            if not had_successful_prediction:
                ss_weight = 0.1 if DO_SS else 0.0
                sub_weight = 0.1
                align_weight = 0.1
                esm_weight = 0.6
                esm_scan_weight = 0.1
                #esm_fold_weight = 0.5
            else:
                # if SS deferred, prefer esm more later
                if DO_SS:
                    ss_weight = 0.1
                    sub_weight = 0.1
                    align_weight = 0.1
                    esm_weight = 0.6
                    esm_scan_weight = 0.1
                    #esm_fold_weight = 0.5
                else:
                    ss_weight = 0.1
                    sub_weight = 0.1
                    align_weight = 0.1
                    esm_weight = 0.6
                    esm_scan_weight = 0.1
                    #esm_fold_weight = 0.8

            matrix_key = SUB_MATRIX.strip().lower()
            if matrix_key in ("blosum", "blosum62", "blosum_62"):
                sim_func = calculate_blosum62_similarity
            elif matrix_key in ("protsub", "prot-sub", "prot_sub"):
                sim_func = calculate_protsub_similarity
            else:
                raise ValueError(f"Unsupported SUB_MATRIX: {SUB_MATRIX}")

            model_id = model_numbers[0]
            for i, (s1, s2, ss1_pred, ss2_pred) in enumerate(zip(seqs1, seqs2, ss1_list, ss2_list)):
                # Secondary-structure comparison (only when DO_SS True)
                if DO_SS and (ss1_pred is not None) and (ss2_pred is not None):
                    *_, s1_pct, _, s2_pct, avg_pct, _ = compare_sequences(
                        ori_1_predicted_structure, ori_2_predicted_structure,
                        ss1_pred, ss2_pred, model_id
                    )
                else:
                    s1_pct = 0.0
                    s2_pct = 0.0
                    avg_pct = 0.0

                s1_clean = s1.rstrip("*")
                s2_clean = s2.rstrip("*")
                align1_pct = (align_sequences_identity(ori_seq_1, s1_clean) / len(s1_clean)) * 100 if s1_clean else 0.0
                align2_pct = (align_sequences_identity(ori_seq_2, s2_clean) / len(s2_clean)) * 100 if s2_clean else 0.0

                try:
                    den1 = float(sim_func(ori_seq_1, ori_seq_1))
                except Exception:
                    den1 = 0.0
                try:
                    den2 = float(sim_func(ori_seq_2, ori_seq_2))
                except Exception:
                    den2 = 0.0

                sub1_pct = round((float(sim_func(ori_seq_1, s1_clean)) / den1) * 100, 2) if (s1_clean and den1 != 0.0) else 0.0
                sub2_pct = round((float(sim_func(ori_seq_2, s2_clean)) / den2) * 100, 2) if (s2_clean and den2 != 0.0) else 0.0

                # ESM metrics
                if _ESM_AVAILABLE and mac is not None:
                    try:
                        m1 = esm_pair_metrics(ori_seq_1, s1_clean)
                        m2 = esm_pair_metrics(ori_seq_2, s2_clean)
                        esm1 = m1["esm_score_100"]
                        esm2 = m2["esm_score_100"]
                        esm_avg = round((esm1 + esm2) / 2.0, 2)
                    except Exception:
                        esm1 = esm2 = esm_avg = 50.0
                else:
                    esm1 = esm2 = esm_avg = 50.0

                # ESM Scan metrics
                esm_scan_score_1 = esm_percentage_score(ori_seq_1, s1_clean, esm_scan_matrix_1)
                esm_scan_score_2 = esm_percentage_score(ori_seq_2, s2_clean, esm_scan_matrix_2)
                esm_scan_avg = (esm_scan_score_1 + esm_scan_score_2) / 2.0
                
                # esm_fold_score_1 = round((esm_fold_plddt_output(s1_clean, 3) / esm_fold_ori_seq_1) * 100, 2)
                # esm_fold_score_2 = round((esm_fold_plddt_output(s2_clean, 3) / esm_fold_ori_seq_2) * 100, 2)
                # esm_fold_avg = (esm_fold_score_1 + esm_fold_score_2) / 2.0

                # combined_score = (
                #     (avg_pct * ss_weight) +
                #     (((align1_pct + align2_pct) / 2.0) * align_weight) +
                #     (((sub1_pct + sub2_pct) / 2.0) * sub_weight) +
                #     (esm_avg * esm_weight) +
                #     (esm_scan_avg * esm_scan_weight) +
                #     (esm_fold_avg * esm_fold_weight)  # ESM-fold score contribution

                # )

                # scores.append((
                #     s1_pct, s2_pct, avg_pct,
                #     align1_pct, align2_pct,
                #     sub1_pct, sub2_pct,
                #     esm1, esm2, esm_avg, 
                #     esm_scan_score_1, esm_scan_score_2, esm_scan_avg,
                #     esm_fold_score_1, esm_fold_score_2, esm_fold_avg,
                #     combined_score
                # ))

                combined_score = (
                    (avg_pct * ss_weight) +
                    (((align1_pct + align2_pct) / 2.0) * align_weight) +
                    (((sub1_pct + sub2_pct) / 2.0) * sub_weight) +
                    (esm_avg * esm_weight) +
                    (esm_scan_avg * esm_scan_weight)

                )

                scores.append((
                    s1_pct, s2_pct, avg_pct,
                    align1_pct, align2_pct,
                    sub1_pct, sub2_pct,
                    esm1, esm2, esm_avg, 
                    esm_scan_score_1, esm_scan_score_2, esm_scan_avg,
                    combined_score
                ))

                # --- Debug printing for first few candidates ---
                if i < 2:
                    print(
                        f"      Candidate {i}: "
                        f"SS_avg={avg_pct:.3f}, "
                        f"Align1={align1_pct:.2f}%, Align2={align2_pct:.2f}%, "
                        f"Sub_avg={(sub1_pct+sub2_pct)/2.0:.2f}%, "
                        f"ESM_avg={esm_avg:.2f}, "
                        f"ESMscan_avg={esm_scan_avg:.2f}, "
                        #f"ESM_fold_avg={esm_fold_avg:.2f}, "
                        f"Combined={combined_score:.3f}"
                    )

                if combined_score > best_combined:
                    best_combined = combined_score
                    best_idx = i

                # Early-stop logic: prefer SS if DO_SS True; otherwise ESM threshold
                if DO_SS:
                    if s1_pct >= EARLY_SS_THRESHOLD and s2_pct >= EARLY_SS_THRESHOLD and early_idx is None:
                        early_idx = i
                        print(f"      Early-stop candidate at i={i}: SS1={s1_pct:.2f}%, SS2={s2_pct:.2f}%")
                        break
                else:
                    if (esm1 >= ESM_EARLY_THRESHOLD) and (esm2 >= ESM_EARLY_THRESHOLD) and early_idx is None:
                        early_idx = i
                        print(f"      Early-stop candidate at i={i}: ESM1={esm1:.2f}, ESM2={esm2:.2f}")
                        break

            # Build df_unique and attach metrics
            processed_n = len(scores)
            df_unique = pd.DataFrame({
                "translated_aa_seq_1": seqs1[:processed_n],
                "translated_aa_seq_2": seqs2[:processed_n],
            })

            if scores:
                (df_unique["ss_score_1"], df_unique["ss_score_2"], df_unique["ss_score_avg"],
                 df_unique["align1"], df_unique["align2"],
                 df_unique["sub1"], df_unique["sub2"],
                 df_unique["esm1"], df_unique["esm2"], df_unique["esm_avg"],
                 df_unique["esm_scan_score_1"], df_unique["esm_scan_score_2"], df_unique["esm_scan_avg"],
                 #df_unique["esm_fold_score_1"], df_unique["esm_fold_score_2"], df_unique["esm_fold_avg"],   
                 df_unique["combined_score"]) = zip(*scores)

            # Map integrated sequences if present
            if "translated_integrated_seq_1" in df.columns and "translated_integrated_seq_2" in df.columns:
                pair_to_integrated = {}
                for r in df.itertuples(index=False):
                    key = (r.translated_aa_seq_1, r.translated_aa_seq_2)
                    if key not in pair_to_integrated:
                        pair_to_integrated[key] = (r.translated_integrated_seq_1, r.translated_integrated_seq_2)
                df_unique["translated_integrated_seq_1"] = [
                    pair_to_integrated.get((a, b), (a, b))[0].rstrip("*")
                    for a, b in zip(df_unique["translated_aa_seq_1"], df_unique["translated_aa_seq_2"])
                ]
                df_unique["translated_integrated_seq_2"] = [
                    pair_to_integrated.get((a, b), (a, b))[1].rstrip("*")
                    for a, b in zip(df_unique["translated_aa_seq_1"], df_unique["translated_aa_seq_2"])
                ]
            else:
                df_unique["translated_integrated_seq_1"] = [s.rstrip("*") for s in df_unique["translated_aa_seq_1"]]
                df_unique["translated_integrated_seq_2"] = [s.rstrip("*") for s in df_unique["translated_aa_seq_2"]]

            # Cleaned originals (no '*')
            seq1_clean = sequence_1_original.replace("*", "")
            seq2_clean = sequence_2_original.replace("*", "")

            # Take terminal 104 aa if available
            seq1_terminal_104 = seq1_clean[-104:] if len(seq1_clean) > 104 else seq1_clean
            seq2_terminal_104 = seq2_clean[-104:] if len(seq2_clean) > 104 else seq2_clean

            # Append records with attempt/window metadata (these may have SS placeholders if DO_SS==False)
            for i, row in enumerate(df_unique.itertuples(index=False)):
                all_pairs_records.append({
                    "aa_seq_1_original": sequence_1_original,
                    "aa_seq_2_original": sequence_2_original,

                    # NEW: terminal subsequences
                    "aa_seq_1_terminal_104": seq1_terminal_104,
                    "aa_seq_2_terminal_104": seq2_terminal_104,

                    "translated_aa_seq_1": row.translated_aa_seq_1,
                    "translated_aa_seq_2": row.translated_aa_seq_2,
                    "translated_integrated_seq_1": row.translated_integrated_seq_1,
                    "translated_integrated_seq_2": row.translated_integrated_seq_2,
                    # scores
                    "ss_score_1": float(getattr(row, "ss_score_1", 0.0)),
                    "ss_score_2": float(getattr(row, "ss_score_2", 0.0)),
                    "ss_score_avg": float(getattr(row, "ss_score_avg", 0.0)),
                    "ss_pred_1": ss1_list[i] if DO_SS else None,
                    "ss_pred_2": ss2_list[i] if DO_SS else None,
                    "align1": float(getattr(row, "align1", 0.0)),
                    "align2": float(getattr(row, "align2", 0.0)),
                    "sub1": float(getattr(row, "sub1", 0.0)),
                    "sub2": float(getattr(row, "sub2", 0.0)),
                    "esm1": float(getattr(row, "esm1", np.nan)),
                    "esm2": float(getattr(row, "esm2", np.nan)),
                    "esm_avg": float(getattr(row, "esm_avg", np.nan)),
                    "esm_scan_score_1": float(getattr(row, "esm_scan_score_1", 0.0)),
                    "esm_scan_score_2": float(getattr(row, "esm_scan_score_2", 0.0)),
                    "esm_scan_avg": float(getattr(row, "esm_scan_avg", 0.0)),
                    "combined_score": float(getattr(row, "combined_score", 0.0)),
                    "attempt": retry_count,
                    "window": w,
                    "attempt_window": f"{retry_count}-{w+1}"
                })

            # --- Ensure parent is considered without recomputing heavy metrics ---
            # config (near top of script)
            INCLUDE_PARENT_AS_CANDIDATE = False   # set False to NOT include parent as a candidate

            # --- Ensure parent is considered without recomputing heavy metrics (optional) ---
            parent_key1 = str(start_seq_aa_1).rstrip("*").strip()
            parent_key2 = str(start_seq_aa_2).rstrip("*").strip()

            parent_mask = (
                df_unique["translated_integrated_seq_1"].astype(str).str.rstrip("*").str.strip() == parent_key1
            ) & (
                df_unique["translated_integrated_seq_2"].astype(str).str.rstrip("*").str.strip() == parent_key2
            )

            # Try to find cached parent record (cheap in-memory lookup)
            parent_rec = None
            for rec in all_pairs_records:
                if (str(rec.get("translated_integrated_seq_1", "")).strip().rstrip("*") == parent_key1
                        and str(rec.get("translated_integrated_seq_2", "")).strip().rstrip("*") == parent_key2):
                    parent_rec = rec
                    break

            if INCLUDE_PARENT_AS_CANDIDATE:
                if parent_mask.any():
                    # Parent already present in df_unique: update its recorded heavy metrics from cache (if available)
                    if parent_rec is not None:
                        # update relevant columns for rows matching parent_key
                        mask = parent_mask
                        # ensure columns exist
                        for col in ["ss_score_1","ss_score_2","ss_score_avg","ss_pred_1","ss_pred_2",
                                    "align1","align2","sub1","sub2",
                                    "esm1","esm2","esm_avg",
                                    "esm_scan_score_1","esm_scan_score_2","esm_scan_avg",
                                    "combined_score"]:
                            if col not in df_unique.columns:
                                df_unique[col] = 0.0 if col != "ss_pred_1" and col != "ss_pred_2" else None
                        # assign from cached parent_rec (use get with defaults)
                        df_unique.loc[mask, "ss_score_1"] = parent_rec.get("ss_score_1", df_unique.loc[mask, "ss_score_1"])
                        df_unique.loc[mask, "ss_score_2"] = parent_rec.get("ss_score_2", df_unique.loc[mask, "ss_score_2"])
                        df_unique.loc[mask, "ss_score_avg"] = parent_rec.get("ss_score_avg", df_unique.loc[mask, "ss_score_avg"])
                        df_unique.loc[mask, "ss_pred_1"] = parent_rec.get("ss_pred_1", df_unique.loc[mask, "ss_pred_1"])
                        df_unique.loc[mask, "ss_pred_2"] = parent_rec.get("ss_pred_2", df_unique.loc[mask, "ss_pred_2"])
                        df_unique.loc[mask, "align1"] = parent_rec.get("align1", df_unique.loc[mask, "align1"])
                        df_unique.loc[mask, "align2"] = parent_rec.get("align2", df_unique.loc[mask, "align2"])
                        df_unique.loc[mask, "sub1"] = parent_rec.get("sub1", df_unique.loc[mask, "sub1"])
                        df_unique.loc[mask, "sub2"] = parent_rec.get("sub2", df_unique.loc[mask, "sub2"])
                        df_unique.loc[mask, "esm1"] = parent_rec.get("esm1", df_unique.loc[mask, "esm1"])
                        df_unique.loc[mask, "esm2"] = parent_rec.get("esm2", df_unique.loc[mask, "esm2"])
                        df_unique.loc[mask, "esm_avg"] = parent_rec.get("esm_avg", df_unique.loc[mask, "esm_avg"])
                        df_unique.loc[mask, "esm_scan_score_1"] = parent_rec.get("esm_scan_score_1", df_unique.loc[mask, "esm_scan_score_1"])
                        df_unique.loc[mask, "esm_scan_score_2"] = parent_rec.get("esm_scan_score_2", df_unique.loc[mask, "esm_scan_score_2"])
                        df_unique.loc[mask, "esm_scan_avg"] = parent_rec.get("esm_scan_avg", df_unique.loc[mask, "esm_scan_avg"])
                        # update combined_score only if cached parent has a numeric combined_score
                        try:
                            parent_comb = float(parent_rec.get("combined_score", np.nan))
                            if not np.isnan(parent_comb):
                                df_unique.loc[mask, "combined_score"] = parent_comb
                        except Exception:
                            pass

                else:
                    # Parent not present in df_unique: append cached parent row (if available)
                    if parent_rec is not None:
                        append_row = {
                            "translated_aa_seq_1": parent_rec.get("translated_aa_seq_1", parent_key1),
                            "translated_aa_seq_2": parent_rec.get("translated_aa_seq_2", parent_key2),
                            "translated_integrated_seq_1": parent_rec.get("translated_integrated_seq_1", parent_key1),
                            "translated_integrated_seq_2": parent_rec.get("translated_integrated_seq_2", parent_key2),
                            "ss_score_1": parent_rec.get("ss_score_1", 0.0),
                            "ss_score_2": parent_rec.get("ss_score_2", 0.0),
                            "ss_score_avg": parent_rec.get("ss_score_avg", 0.0),
                            "align1": parent_rec.get("align1", 0.0),
                            "align2": parent_rec.get("align2", 0.0),
                            "sub1": parent_rec.get("sub1", 0.0),
                            "sub2": parent_rec.get("sub2", 0.0),
                            "esm1": parent_rec.get("esm1", 50.0),
                            "esm2": parent_rec.get("esm2", 50.0),
                            "esm_avg": parent_rec.get("esm_avg", 50.0),
                            "esm_scan_score_1": parent_rec.get("esm_scan_score_1", 50.0),
                            "esm_scan_score_2": parent_rec.get("esm_scan_score_2", 50.0),
                            "esm_scan_avg": parent_rec.get("esm_scan_avg", 50.0),
                            # critical: use cached combined_score (no recompute)
                            "combined_score": parent_rec.get("combined_score", 0.0),
                            "ss_pred_1": parent_rec.get("ss_pred_1", None),
                            "ss_pred_2": parent_rec.get("ss_pred_2", None),
                            "attempt": parent_rec.get("attempt", retry_count),
                            "window": parent_rec.get("window", w),
                            "attempt_window": parent_rec.get("attempt_window", f"{retry_count}-{w+1}")
                        }
                        df_unique = pd.concat([df_unique, pd.DataFrame([append_row])], ignore_index=True, sort=False)

            # Make combined_score numeric & NaN-safe then pick best candidate (same logic as before)
            if "combined_score" not in df_unique.columns:
                df_unique["combined_score"] = 0.0
            df_unique["combined_score"] = pd.to_numeric(df_unique["combined_score"], errors="coerce").fillna(-1e12)

            # pick chosen_row: early_idx prioritized
            if early_idx is not None:
                chosen_row = df_unique.iloc[early_idx].copy()
                chosen_index = early_idx
                reason = "early stop"
            else:
                chosen_index = int(df_unique["combined_score"].idxmax())
                chosen_row = df_unique.loc[chosen_index].copy()
                reason = "best combined_score"

            # If DO_SS==False but EVAL_SS_ON_EARLY True and we had early candidate -> compute SS for that candidate now
            if (not DO_SS) and (early_idx is not None) and EVAL_SS_ON_EARLY:
                try:
                    # ensure original SS available
                    if ori_1_predicted_structure is None or ori_2_predicted_structure is None:
                        ori_1_predicted_structure = predict_secondary_structure(ori_seq_1)
                        ori_2_predicted_structure = predict_secondary_structure(ori_seq_2)

                    s1_pct, s2_pct, avg_pct, ss_a, ss_b = _compute_ss_for_pair_and_scores(
                        ori_1_predicted_structure, ori_2_predicted_structure,
                        str(chosen_row["translated_integrated_seq_1"]),
                        str(chosen_row["translated_integrated_seq_2"]),
                        model_id
                    )
                    chosen_row["ss_score_1"] = s1_pct
                    chosen_row["ss_score_2"] = s2_pct
                    chosen_row["ss_score_avg"] = avg_pct
                    # recompute combined_score
                    # chosen_row["combined_score"] = (
                    #     (avg_pct * ss_weight) +
                    #     (((chosen_row["align1"] + chosen_row["align2"]) / 2.0) * align_weight) +
                    #     (((chosen_row["sub1"] + chosen_row["sub2"]) / 2.0) * sub_weight) +
                    #     (chosen_row["esm_avg"] * esm_weight) +
                    #     (chosen_row["esm_scan_avg"] * esm_scan_weight) +
                    #     (chosen_row["esm_fold_avg"] * esm_fold_weight)  # ESM-fold score contribution
                    # )
                    chosen_row["combined_score"] = (
                        (avg_pct * ss_weight) +
                        (((chosen_row["align1"] + chosen_row["align2"]) / 2.0) * align_weight) +
                        (((chosen_row["sub1"] + chosen_row["sub2"]) / 2.0) * sub_weight) +
                        (chosen_row["esm_avg"] * esm_weight) +
                        (chosen_row["esm_scan_avg"] * esm_scan_weight)
                    )
                    # update df_unique so saved bookkeeping reflects computed SS
                    if chosen_index is not None:
                        df_unique.at[chosen_index, "ss_score_1"] = chosen_row["ss_score_1"]
                        df_unique.at[chosen_index, "ss_score_2"] = chosen_row["ss_score_2"]
                        df_unique.at[chosen_index, "ss_score_avg"] = chosen_row["ss_score_avg"]
                        df_unique.at[chosen_index, "combined_score"] = chosen_row["combined_score"]
                    else:
                        # parent selected but not present as an index in this df_unique — update any matching integrated pair rows
                        key1 = str(chosen_row["translated_integrated_seq_1"]).rstrip("*").strip()
                        key2 = str(chosen_row["translated_integrated_seq_2"]).rstrip("*").strip()
                        mask = (
                            df_unique["translated_integrated_seq_1"].astype(str).str.rstrip("*").str.strip() == key1
                        ) & (
                            df_unique["translated_integrated_seq_2"].astype(str).str.rstrip("*").str.strip() == key2
                        )
                        if mask.any():
                            df_unique.loc[mask, "ss_score_1"] = chosen_row["ss_score_1"]
                            df_unique.loc[mask, "ss_score_2"] = chosen_row["ss_score_2"]
                            df_unique.loc[mask, "ss_score_avg"] = chosen_row["ss_score_avg"]
                            df_unique.loc[mask, "combined_score"] = chosen_row["combined_score"]
                    print(f"    [EVAL SS ON EARLY] Computed SS for early candidate: SS1={s1_pct:.3f}, SS2={s2_pct:.3f}, new combined={chosen_row['combined_score']:.3f}")

                    # update the in-memory records so the saved Excel includes these SS values
                    updated = update_all_pairs_records_with_ss(
                        target_seq1=str(chosen_row["translated_integrated_seq_1"]),
                        target_seq2=str(chosen_row["translated_integrated_seq_2"]),
                        attempt=retry_count,
                        window=w,
                        s1_pct=s1_pct,
                        s2_pct=s2_pct,
                        ss_pred_a=ss_a,
                        ss_pred_b=ss_b,
                        avg_pct=avg_pct
                    )
                    print(f"    [EVAL SS ON EARLY] Updated {updated} entry(ies) in all_pairs_records with deferred SS.")
                except Exception as _e:
                    print(f"[WARN] SS evaluation for early candidate failed: {_e}")

            print(
                f"    Selected ({reason}): combined={chosen_row['combined_score']:.3f}, "
                f"Align1={chosen_row['align1']:.2f}%, Align2={chosen_row['align2']:.2f}%, "
                f"ESM_avg={chosen_row['esm_avg']:.2f}, ESMscan_avg={chosen_row['esm_scan_avg']:.2f}%"
            )

            start_seq_aa_1 = str(chosen_row["translated_integrated_seq_1"])
            start_seq_aa_2 = str(chosen_row["translated_integrated_seq_2"])

            summaries.append(dict(
                window = w,
                AA_range = f"{start_aa}-{end_aa}",
                SS_score1 = round(float(chosen_row.get("ss_score_1", 0.0)), 3),
                SS_score2 = round(float(chosen_row.get("ss_score_2", 0.0)), 3),
                SS_avg = round(float(chosen_row.get("ss_score_avg", 0.0)), 3),
                Align1 = round(float(chosen_row["align1"]), 2),
                Align2 = round(float(chosen_row["align2"]), 2),
                Sub1 = round(float(chosen_row["sub1"]), 2),
                Sub2 = round(float(chosen_row["sub2"]), 2),
                ESM1 = round(float(chosen_row["esm1"]), 2),
                ESM2 = round(float(chosen_row["esm2"]), 2),
                ESM_avg = round(float(chosen_row["esm_avg"]), 2),
                ESM_scan_score1 = round(float(chosen_row["esm_scan_score_1"]), 2),
                ESM_scan_score2 = round(float(chosen_row["esm_scan_score_2"]), 2),
                ESM_scan_avg = round(float(chosen_row["esm_scan_avg"]), 2),
                #ESM_fold_score1 = round(float(chosen_row["esm_fold_score_1"]), 2),
                #ESM_fold_score2 = round(float(chosen_row["esm_fold_score_2"]), 2),
                #ESM_fold_avg = round(float(chosen_row["esm_fold_avg"]), 2),
                Combined = round(float(chosen_row["combined_score"]), 3),
            ))
            print(f"  Summary so far: {summaries[-1]}")

            if early_idx is not None:
                print(f"Early success detected; halting this attempt.")
                early_halt = True
                break

        # end windows

        if early_halt:
            success = True
            # If we halted early, optionally evaluate SS for the final chosen pair at attempt end
            if (not DO_SS) and EVAL_SS_AT_ATTEMPT_END:
                try:
                    final_a = start_seq_aa_1
                    final_b = start_seq_aa_2
                    if ori_1_predicted_structure is None or ori_2_predicted_structure is None:
                        ori_1_predicted_structure = predict_secondary_structure(ori_seq_1)
                        ori_2_predicted_structure = predict_secondary_structure(ori_seq_2)
                    f_s1_pct, f_s2_pct, f_avg_pct, f_ss_a, f_ss_b = _compute_ss_for_pair_and_scores(
                        ori_1_predicted_structure, ori_2_predicted_structure,
                        final_a, final_b, model_id
                    )
                    print(f"[EVAL SS AT ATTEMPT END] Final pair SS: seq1={f_s1_pct:.3f}, seq2={f_s2_pct:.3f}, avg={f_avg_pct:.3f}")

                    # update summaries' last window entry if available
                    if summaries:
                        summaries[-1]["SS_score1"] = round(f_s1_pct, 3)
                        summaries[-1]["SS_score2"] = round(f_s2_pct, 3)
                        summaries[-1]["SS_avg"] = round(f_avg_pct, 3)

                    # update stored records for the final chosen pair - use last summary window if available
                    final_window_index = summaries[-1]["window"] if summaries else (w if 'w' in locals() else 0)
                    updated = update_all_pairs_records_with_ss(
                        target_seq1=final_a,
                        target_seq2=final_b,
                        attempt=retry_count,
                        window=final_window_index,
                        s1_pct=f_s1_pct,
                        s2_pct=f_s2_pct,
                        ss_pred_a=f_ss_a,
                        ss_pred_b=f_ss_b,
                        avg_pct=f_avg_pct
                    )
                    print(f"[EVAL SS AT ATTEMPT END] Updated {updated} entry(ies) in all_pairs_records with final SS.")
                except Exception as _e:
                    print(f"[WARN] Deferred SS computation at attempt end failed: {_e}")
            break

        if not summaries:
            print("\nNo summaries generated this attempt.")
            continue

        final_summary = summaries[-1]
        final_ss1, final_ss2 = final_summary["SS_score1"], final_summary["SS_score2"]
        final_esm1, final_esm2 = final_summary["ESM1"], final_summary["ESM2"]
        print(f"\nFinal metrics this attempt: SS (seq1,seq2)=({final_ss1},{final_ss2}), ESM (seq1,seq2)=({final_esm1},{final_esm2})")

        # If DO_SS False but we want SS at attempt end, compute SS for final chosen pair now
        if (not DO_SS) and EVAL_SS_AT_ATTEMPT_END:
            try:
                final_a = start_seq_aa_1
                final_b = start_seq_aa_2
                if ori_1_predicted_structure is None or ori_2_predicted_structure is None:
                    ori_1_predicted_structure = predict_secondary_structure(ori_seq_1)
                    ori_2_predicted_structure = predict_secondary_structure(ori_seq_2)
                f_s1_pct, f_s2_pct, f_avg_pct, f_ss_a, f_ss_b = _compute_ss_for_pair_and_scores(
                    ori_1_predicted_structure, ori_2_predicted_structure,
                    final_a, final_b, model_id
                )
                summaries[-1]["SS_score1"] = round(f_s1_pct, 3)
                summaries[-1]["SS_score2"] = round(f_s2_pct, 3)
                summaries[-1]["SS_avg"] = round(f_avg_pct, 3)
                print(f"[EVAL SS AT_ATTEMPT_END] Computed SS for attempt-final pair: SS1={f_s1_pct:.3f}, SS2={f_s2_pct:.3f}")

                # Update the stored records for the final chosen pair:
                final_window_index = final_summary["window"]
                updated = update_all_pairs_records_with_ss(
                    target_seq1=final_a,
                    target_seq2=final_b,
                    attempt=retry_count,
                    window=final_window_index,
                    s1_pct=f_s1_pct,
                    s2_pct=f_s2_pct,
                    ss_pred_a=f_ss_a,
                    ss_pred_b=f_ss_b,
                    avg_pct=f_avg_pct
                )
                print(f"[EVAL SS AT_ATTEMPT_END] Updated {updated} entry(ies) in all_pairs_records with attempt-final SS.")
            except Exception as _e:
                print(f"[WARN] Deferred SS computation at attempt end failed: {_e}")

        # Update best overall using appropriate metric
        if DO_SS:
            score_for_compare = summaries[-1]["SS_score1"] + summaries[-1]["SS_score2"]
            if score_for_compare > sum(best_overall_scores):
                best_overall_scores = (summaries[-1]["SS_score1"], summaries[-1]["SS_score2"])
                best_overall_seq1, best_overall_seq2 = start_seq_aa_1, start_seq_aa_2
                best_overall_summaries = summaries.copy()
        else:
            if (final_esm1 + final_esm2) > sum(best_overall_scores):
                best_overall_scores = (final_esm1, final_esm2)
                best_overall_seq1, best_overall_seq2 = start_seq_aa_1, start_seq_aa_2
                best_overall_summaries = summaries.copy()

        # Stopping decision
        if DO_SS:
            if final_ss1 >= SUCCESS_SS_THRESHOLD and final_ss2 >= SUCCESS_SS_THRESHOLD:
                print(f"Success: Both sequences exceed {SUCCESS_SS_THRESHOLD}% SS. Halting.")
                success = True
                break
            else:
                print(f"Secondary structure below {SUCCESS_SS_THRESHOLD}% — retrying…")
        else:
            if (final_esm1 >= ESM_SUCCESS_THRESHOLD) and (final_esm2 >= ESM_SUCCESS_THRESHOLD):
                print(f"Success: Both sequences exceed {ESM_SUCCESS_THRESHOLD} ESM SSIM. Halting.")
                success = True
                break
            else:
                print(f"ESM SSIM below {ESM_SUCCESS_THRESHOLD} for final candidate — retrying…")

    # End attempts loop

    # ---------------------------------------------------------
    # Deferred SS: evaluate top-N by ESM and write back to records
    # (Only when DO_SS is False and we deferred SS during run)
    # ---------------------------------------------------------
    def _evaluate_top_n_by_esm_and_compute_ss(top_n: int = 20):
        """
        When SS computations were deferred (DO_SS==False), this function:
          - selects the top-N candidates by esm_avg (descending),
          - computes SS predictions for those candidates,
          - writes the SS metrics back into matching all_pairs_records entries.

        Matching strategy:
          - Prefer exact match by translated_integrated_seq pair + attempt + window.
          - If exact match not found, performs a best-effort fallback to any matching seq pair.

        Returns:
          number of records updated
        """
        nonlocal ori_1_predicted_structure, ori_2_predicted_structure, ori_seq_1, ori_seq_2, model_id
        global all_pairs_records
        if DO_SS:
            print("[INFO] DO_SS True -> no deferred top-N SS computation required.")
            return 0

        if not all_pairs_records:
            print("[INFO] No pairs recorded; skipping deferred top-N SS computation.")
            return 0

        df_all = pd.DataFrame(all_pairs_records)

        # require esm_avg present and numeric
        if "esm_avg" not in df_all.columns:
            print("[INFO] esm_avg column missing; skipping deferred top-N SS computation.")
            return 0

        df_candidates = df_all[~df_all["esm_avg"].isna()].copy()
        if df_candidates.empty:
            print("[INFO] No candidates with esm_avg available; skipping deferred top-N SS computation.")
            return 0

        # sort descending and take top_n unique (by integrated seq pair + attempt + window)
        df_candidates.sort_values("esm_avg", ascending=False, inplace=True)
        # keep top_n rows (preserve attempt/window so updates target exact entries)
        df_top = df_candidates.head(top_n)

        # ensure original SS preds exist (we need these for compare_sequences)
        try:
            if ori_1_predicted_structure is None:
                ori_1_predicted_structure = predict_secondary_structure(ori_seq_1)
            if ori_2_predicted_structure is None:
                ori_2_predicted_structure = predict_secondary_structure(ori_seq_2)
        except Exception as _e:
            print(f"[WARN] Could not compute original SS predictions required for deferred scoring: {_e}")
            # Continue: _compute_ss_for_pair_and_scores will attempt per-sequence calls via ss_predict_cached.

        updated_count = 0
        attempted_count = 0
        for r in df_top.itertuples(index=False):
            attempted_count += 1
            try:
                # prefer translated_integrated_seq if present (consistent with earlier updates)
                seq1_t = getattr(r, "translated_integrated_seq_1", None) or getattr(r, "translated_aa_seq_1", None)
                seq2_t = getattr(r, "translated_integrated_seq_2", None) or getattr(r, "translated_aa_seq_2", None)
                if seq1_t is None or seq2_t is None:
                    print(f"  [SKIP] Missing sequence in top candidate row: {r}")
                    continue

                attempt_val = int(getattr(r, "attempt", -1))
                window_val = int(getattr(r, "window", -999))

                # compute SS and similarity scores for this pair
                try:
                    s1_pct, s2_pct, avg_pct, ss_a, ss_b = _compute_ss_for_pair_and_scores(
                        ori_1_predicted_structure, ori_2_predicted_structure,
                        str(seq1_t), str(seq2_t), model_id
                    )
                except Exception as _e:
                    print(f"  [WARN] SS compute failed for candidate (attempt={attempt_val},window={window_val}): {_e}")
                    continue

                # write back into all_pairs_records for matching entries
                n_updated = update_all_pairs_records_with_ss(
                    target_seq1=str(seq1_t),
                    target_seq2=str(seq2_t),
                    attempt=attempt_val,
                    window=window_val,
                    s1_pct=s1_pct,
                    s2_pct=s2_pct,
                    ss_pred_a=ss_a,
                    ss_pred_b=ss_b,
                    avg_pct=avg_pct
                )

                if n_updated > 0:
                    updated_count += n_updated
                    print(f"  [TOP-ESM SS] Updated {n_updated} record(s): attempt={attempt_val}, window={window_val}, SS_avg={avg_pct:.3f}")
                else:
                    # best-effort fallback: try to update any matching seq pair regardless of attempt/window
                    fallback_updated = 0
                    for rec in all_pairs_records:
                        if (str(rec.get("translated_integrated_seq_1","")).strip().rstrip("*") == str(seq1_t).strip().rstrip("*")
                            and str(rec.get("translated_integrated_seq_2","")).strip().rstrip("*") == str(seq2_t).strip().rstrip("*")):
                            rec["ss_score_1"] = float(s1_pct)
                            rec["ss_score_2"] = float(s2_pct)
                            rec["ss_score_avg"] = float(avg_pct)
                            rec["ss_pred_1"] = ss_a
                            rec["ss_pred_2"] = ss_b
                            fallback_updated += 1
                    if fallback_updated:
                        updated_count += fallback_updated
                        print(f"  [TOP-ESM SS fallback] Updated {fallback_updated} matching record(s) without exact attempt/window.")
                    else:
                        print(f"  [TOP-ESM SS] No record matched for seq pair (attempt={attempt_val}, window={window_val}); no update performed.")
            except Exception as _e:
                print(f"  [ERROR] Unexpected error evaluating top candidate: {_e}")

        print(f"[TOP-ESM SS] Attempted {attempted_count} top candidates; updated {updated_count} record(s) in all_pairs_records.")
        return updated_count

    # Only run the top-N deferred SS if we deferred SS (DO_SS==False)
    try:
        if (not DO_SS):
            _evaluate_top_n_by_esm_and_compute_ss(top_n=20)
    except Exception as _e:
        print(f"[WARN] Deferred top-N SS pass failed: {_e}")

    # Final reporting & save
    print("\n===== CACHE / EFFICIENCY REPORT =====")
    for k, v in METRICS.items():
        print(f"{k}: {v}")

    if not success:
        print(f"\nWARNING: Did not achieve success according to configured stopping criteria after all retries.")
        print(f"Best achieved (stored metric): seq1={best_overall_scores[0]}, seq2={best_overall_scores[1]}")
        print("SEQ1:", best_overall_seq1)
        print("SEQ2:", best_overall_seq2)
        if best_overall_summaries:
            print(pd.DataFrame(best_overall_summaries).to_string(index=False))

    date_str = datetime.now().strftime("%Y%m%d")
    model_tag = "-".join(str(m) for m in model_numbers)
    out_name = f"{date_str}_length_{model_tag}_row_{row_index_1based}.xlsx"
    out_path = os.path.join(working_directory, out_name)

    # Ensure columns include ss_pred_1/2 even for empty scaffold
    if all_pairs_records:
        pd.DataFrame(all_pairs_records).to_excel(out_path, index=False)
        print(f"\nSaved {len(all_pairs_records)} predicted candidate pairs to {out_path}")
    else:
        pd.DataFrame(columns=[
            "aa_seq_1_original", "aa_seq_2_original",
            "translated_aa_seq_1","translated_aa_seq_2",
            "translated_integrated_seq_1","translated_integrated_seq_2",
            "ss_score_1","ss_score_2","ss_score_avg","ss_pred_1","ss_pred_2",
            "align1","align2","sub1","sub2",
            "esm1","esm2","esm_avg",
            "esm_scan_score_1","esm_scan_score_2","esm_scan_avg",
            "combined_score","attempt","window","attempt_window"
        ]).to_excel(out_path, index=False)
        print(f"\nNo candidates produced; wrote empty scaffold to {out_path}")

    return out_path

In [ ]:
import os

# ----------------------------
# Entry point: run over input table
# ----------------------------

working_directory = "/mnt/e/RStuff/codon_overlap/aa_change_predictions/results/20250827_batch_run_27/"
os.makedirs(working_directory, exist_ok=True)
os.chdir(working_directory)

if __name__ == "__main__":
    # Directly use working_directory ahead of file name
    src_path = os.path.join(working_directory, "aa_1_aa_2.xlsx")
    print(f"Loading pairs from: {src_path}")

    pairs_df = load_pairs_table(src_path)

    # Optional: clean whitespace/newlines in sequences
    pairs_df["aa_seq_1"] = pairs_df["aa_seq_1"].astype(str).str.replace(r"\s+", "", regex=True)
    pairs_df["aa_seq_2"] = pairs_df["aa_seq_2"].astype(str).str.replace(r"\s+", "", regex=True)
    pairs_df["aa_seq_1_brackets"] = pairs_df["aa_seq_1_brackets"].astype(str).str.replace(r"\s+", "", regex=True)
    pairs_df["aa_seq_2_brackets"] = pairs_df["aa_seq_2_brackets"].astype(str).str.replace(r"\s+", "", regex=True)

    # ----------------------------
    # Row range control
    # ----------------------------
    excel_start = 0   # Excel-style: row number to start on (1-based)
    excel_end   = None  # or e.g. 120 to stop after row 120; set to None to run to end

    start_row = excel_start - 1  # convert to 0-based index
    if excel_end is not None:
        end_row = excel_end      # iloc slicing is exclusive on end
        df_slice = pairs_df.iloc[start_row:end_row]
    else:
        df_slice = pairs_df.iloc[start_row:]

    out_files = []
    for i, row in df_slice.iterrows():
        seq1 = row["aa_seq_1"]
        seq2 = row["aa_seq_2"]
        seq1_bracket = row["aa_seq_1_brackets"]
        seq2_bracket = row["aa_seq_2_brackets"]

        if not isinstance(seq1, str) or not isinstance(seq2, str) or not seq1 or not seq2:
            print(f"Row {i+1}: missing/invalid sequences, skipping.")
            continue

        print(f"\n===== Processing row {i+1} =====")

        # --- Reset per-row state ---
        all_pairs_records = []  # fresh record list for this row
        setattr(optimize_pair_and_save, "_had_success_flag", False)  # clear dropout state
        had_successful_prediction = False  # reset for this row
        if RESET_CACHES_PER_ROW:
            SSCACHE = {}
            ESM_EMB_CACHE = {}
            ESM_CONT_CACHE = {}

        out_path = optimize_pair_and_save(seq1, seq2, seq1_bracket, seq2_bracket, i+1)
        out_files.append(out_path)

    print("\nDone. Wrote:")
    for p in out_files:
        print("  -", p)


In [ ]:
#!/usr/bin/env python3
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.patches import Patch
import os

# -------------------------
# Config
# -------------------------
SUB_MATRIX = "BLOSUM62"
RESULTS_FILE = os.path.join(working_directory, "20250824_length_312_row_1.xlsx")

# -------------------------
# Load Data
# -------------------------
if not os.path.exists(RESULTS_FILE):
    raise FileNotFoundError(f"Results file not found: {RESULTS_FILE}")

df_plot = pd.read_excel(RESULTS_FILE)
if df_plot.empty:
    raise ValueError("Results file is empty; nothing to plot.")

df_plot.sort_values(["attempt", "window"], inplace=True)
max_window = df_plot["window"].max() + 1
df_plot["window_cont"] = df_plot["window"] + (df_plot["attempt"] - 1) * max_window
attempts = sorted(df_plot["attempt"].unique())

# Seq-specific colors
seq_colors = {"seq1": "red", "seq2": "blue"}

# Background colors per attempt
pass_bg_colors = {
    1: "#cfcfcf",
    2: "#e7e7b0",
    3: "#b0d4e7",
    4: "#e7b0d4"
}

# -------------------------
# Helper: plot group of metrics
# -------------------------
def plot_metric_group(ax, metrics, labels, styles):
    """Plot multiple metrics in one panel, with shaded ranges per pass."""
    for attempt in attempts:
        subdf = df_plot[df_plot["attempt"] == attempt]
        if subdf.empty:
            continue
        min_x = subdf["window_cont"].min()
        max_x = subdf["window_cont"].max()
        bg_color = pass_bg_colors.get(attempt, "#f9f9f9")

        ax.axvspan(min_x, max_x, color=bg_color, alpha=0.15, zorder=0)
        ax.text((min_x + max_x) / 2, 5, f"Pass {attempt}",
                ha="center", va="bottom",
                fontsize=9, weight="bold",
                color="black", alpha=0.7, zorder=1)

    for metric, label, style in zip(metrics, labels, styles):
        if metric not in df_plot.columns:
            continue
        seq_key = "seq1" if metric.endswith("1") else "seq2"
        color = seq_colors[seq_key]

        for attempt in attempts:
            subdf = df_plot[df_plot["attempt"] == attempt]
            grouped = subdf.groupby("window_cont")[metric]
            x = grouped.mean().index.to_numpy()
            y = grouped.mean().to_numpy()
            y_min = grouped.min().to_numpy()
            y_max = grouped.max().to_numpy()

            ax.fill_between(x, y_min, y_max, color=color, alpha=0.2)
            ax.plot(
                x, y,
                linestyle=style,
                color=color,
                linewidth=1.5,
                marker="o",
                markersize=3,
                label=f"{label} (Pass {attempt})"
            )

    ax.set_ylim(0, 100)
    ax.set_xlabel("Window (continuous)", fontsize=11)
    ax.set_ylabel("Score (%)", fontsize=11)
    ax.tick_params(axis='both', labelsize=9)
    ax.grid(True, alpha=0.3, linewidth=0.6)

# -------------------------
# Create Plots
# -------------------------
fig, axes = plt.subplots(3, 2, figsize=(12, 10))
axes = axes.flatten()

# Row 1: Combined | SS
plot_metric_group(axes[0], ["combined_score"], ["Combined Score"], ["-"])
axes[0].set_title("Combined Score", fontsize=12, weight="bold")

plot_metric_group(
    axes[1],
    ["ss_score_1", "ss_score_2"],
    ["SS Score 1", "SS Score 2"],
    ["-", "--"]
)
axes[1].set_title("Secondary Structure Scores", fontsize=12, weight="bold")

# Row 2: ESM contact | ESM scan
if {"esm1", "esm2"} <= set(df_plot.columns):
    plot_metric_group(
        axes[2],
        ["esm1", "esm2"],
        ["ESM Contact 1", "ESM Contact 2"],
        ["-", "--"]
    )
    axes[2].set_title("ESM Contact-Map Scores", fontsize=12, weight="bold")

if {"esm_scan_score_1", "esm_scan_score_2"} <= set(df_plot.columns):
    plot_metric_group(
        axes[3],
        ["esm_scan_score_1", "esm_scan_score_2"],
        ["ESM Scan 1", "ESM Scan 2"],
        ["-", "--"]
    )
    axes[3].set_title("ESM Scan Scores", fontsize=12, weight="bold")

# Row 3: Alignment | Substitution
if {"align1", "align2"} <= set(df_plot.columns):
    plot_metric_group(
        axes[4],
        ["align1", "align2"],
        ["Alignment 1", "Alignment 2"],
        ["-", "--"]
    )
    axes[4].set_title("Alignment Scores", fontsize=12, weight="bold")

if {"sub1", "sub2"} <= set(df_plot.columns):
    plot_metric_group(
        axes[5],
        ["sub1", "sub2"],
        [f"{SUB_MATRIX.upper()} 1", f"{SUB_MATRIX.upper()} 2"],
        ["-", "--"]
    )
    axes[5].set_title(f"{SUB_MATRIX.upper()} Scores", fontsize=12, weight="bold")

# Legend
legend_patches = [
    Patch(facecolor=seq_colors["seq1"], alpha=0.4, label="Seq 1 range/line"),
    Patch(facecolor=seq_colors["seq2"], alpha=0.4, label="Seq 2 range/line")
]
fig.legend(
    handles=legend_patches,
    loc="upper center",
    ncol=2,
    fontsize=9,
    frameon=False
)

plt.tight_layout(rect=[0, 0, 1, 0.94], pad=1.0)
plt.show()


In [ ]:
#!/usr/bin/env python3
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.patches import Patch
import glob
import os
import re

# -------------------------
# Config (edit if needed)
# -------------------------
try:
    working_directory  # noqa: F821
except NameError:
    # set this to your folder (WSL: /mnt/e/..., Windows: r"E:\...")
    working_directory = "/mnt/e/RStuff/codon_overlap/aa_change_predictions/results/20250823_batch_run_26"

# Pattern to match files. If None the script will try to auto-detect from RESULTS_FILE.
RESULTS_FILE = os.path.join(working_directory, "20250824_length_310_row_1.xlsx")
RESULTS_GLOB = os.path.join(working_directory, "20250824_length_310_row_*.xlsx")

# If True each figure will also be saved as PNG to working_directory/plots
SAVE_PLOTS = True
PLOT_DPI = 150

SUB_MATRIX = "BLOSUM62"

# Plot colors
seq_colors = {"seq1": "red", "seq2": "blue"}
pass_bg_colors = {
    1: "#cfcfcf",
    2: "#e7e7b0",
    3: "#b0d4e7",
    4: "#e7b0d4"
}

# -------------------------
# Helpers: file discovery
# -------------------------
def find_versioned_files(results_file, results_glob=None):
    if results_glob:
        files = sorted(glob.glob(results_glob))
        return files

    m = re.match(r"(.+)_\d+(\.xls[xm]?)$", os.path.basename(results_file))
    if not m:
        base = os.path.splitext(os.path.basename(results_file))[0]
        prefix_guess = base.rsplit("_", 1)[0] if "_" in base else base
        glob_pattern = os.path.join(working_directory, f"{prefix_guess}_*.xlsx")
        files = sorted(glob.glob(glob_pattern))
        if not files:
            alt = os.path.join(working_directory, os.path.basename(results_file))
            if os.path.exists(alt):
                return [alt]
        return files

    prefix = m.group(1)
    glob_pattern = os.path.join(working_directory, f"{prefix}_*.xlsx")
    files = sorted(glob.glob(glob_pattern))
    if not files and os.path.exists(results_file):
        return [results_file]
    return files

files = find_versioned_files(RESULTS_FILE, RESULTS_GLOB)
if not files:
    raise FileNotFoundError(f"No versioned files found (tried: {RESULTS_FILE} and glob={RESULTS_GLOB}).")

# Optional: prepare output dir for saved plots
plots_dir = os.path.join(working_directory, "plots")
if SAVE_PLOTS and not os.path.exists(plots_dir):
    os.makedirs(plots_dir, exist_ok=True)

# -------------------------
# Plotting function for single file
# -------------------------
def plot_file(df, title_prefix=""):
    # compute window_cont (local to this file)
    if "attempt" not in df.columns or "window" not in df.columns:
        raise ValueError("Input dataframe must contain 'attempt' and 'window' columns")

    df = df.copy()
    df["attempt"] = df["attempt"].astype(int)
    max_window = int(df["window"].max()) + 1
    df["window_cont"] = df["window"] + (df["attempt"] - 1) * max_window

    attempt_pairs = sorted(df["attempt"].unique())

    def plot_metric_group(ax, metrics, labels, styles):
        # Shade and annotate per attempt
        ylim_top = 100
        for attempt in attempt_pairs:
            subdf = df[df["attempt"] == attempt]
            if subdf.empty:
                continue
            min_x = subdf["window_cont"].min()
            max_x = subdf["window_cont"].max()
            bg_color = pass_bg_colors.get(attempt, "#f9f9f9")
            ax.axvspan(min_x, max_x, color=bg_color, alpha=0.12, zorder=0)
            ax.text((min_x + max_x) / 2, ylim_top * 0.98, f"P{attempt}",
                    ha="center", va="top", fontsize=7, weight="bold",
                    color="black", alpha=0.7, zorder=1,
                    bbox=dict(facecolor='white', edgecolor='none', pad=0.3, alpha=0.6))

        for metric, label, style in zip(metrics, labels, styles):
            if metric not in df.columns:
                continue
            seq_key = "seq1" if metric.endswith("1") else "seq2"
            color = seq_colors.get(seq_key, "black")
            for attempt in attempt_pairs:
                subdf = df[df["attempt"] == attempt]
                if subdf.empty:
                    continue
                grouped = subdf.groupby("window_cont")[metric]
                x = grouped.mean().index.to_numpy()
                y = grouped.mean().to_numpy()
                y_min = grouped.min().to_numpy()
                y_max = grouped.max().to_numpy()

                ax.fill_between(x, y_min, y_max, color=color, alpha=0.12)
                ax.plot(x, y, linestyle=style, color=color, linewidth=1.2,
                        marker="o", markersize=3, label=f"{label} (P{attempt})")

        ax.set_ylim(0, 100)
        ax.set_xlabel("Window (continuous)", fontsize=10)
        ax.set_ylabel("Score (%)", fontsize=10)
        ax.tick_params(axis='both', labelsize=9)
        ax.grid(True, alpha=0.3, linewidth=0.6)

    # create the 3x2 grid
    fig, axes = plt.subplots(3, 2, figsize=(12, 9))
    axes = axes.flatten()

    plot_metric_group(axes[0], ["combined_score"], ["Combined Score"], ["-"])
    axes[0].set_title("Combined Score", fontsize=11, weight="bold")

    plot_metric_group(axes[1], ["ss_score_1", "ss_score_2"], ["SS Score 1", "SS Score 2"], ["-", "--"])
    axes[1].set_title("Secondary Structure Scores", fontsize=11, weight="bold")

    if {"esm1", "esm2"} <= set(df.columns):
        plot_metric_group(axes[2], ["esm1", "esm2"], ["ESM Contact 1", "ESM Contact 2"], ["-", "--"])
        axes[2].set_title("ESM Contact-Map Scores", fontsize=11, weight="bold")
    else:
        axes[2].set_visible(False)

    if {"esm_scan_score_1", "esm_scan_score_2"} <= set(df.columns):
        plot_metric_group(axes[3], ["esm_scan_score_1", "esm_scan_score_2"], ["ESM Scan 1", "ESM Scan 2"], ["-", "--"])
        axes[3].set_title("ESM Scan Scores", fontsize=11, weight="bold")
    else:
        axes[3].set_visible(False)

    if {"align1", "align2"} <= set(df.columns):
        plot_metric_group(axes[4], ["align1", "align2"], ["Alignment 1", "Alignment 2"], ["-", "--"])
        axes[4].set_title("Alignment Scores", fontsize=11, weight="bold")
    else:
        axes[4].set_visible(False)

    if {"sub1", "sub2"} <= set(df.columns):
        plot_metric_group(axes[5], ["sub1", "sub2"], [f"{SUB_MATRIX.upper()} 1", f"{SUB_MATRIX.upper()} 2"], ["-", "--"])
        axes[5].set_title(f"{SUB_MATRIX.upper()} Scores", fontsize=11, weight="bold")
    else:
        axes[5].set_visible(False)

    # shared legend
    legend_patches = [
        Patch(facecolor=seq_colors["seq1"], alpha=0.4, label="Seq 1 range/line"),
        Patch(facecolor=seq_colors["seq2"], alpha=0.4, label="Seq 2 range/line")
    ]
    fig.legend(handles=legend_patches, loc="upper center", ncol=2, fontsize=9, frameon=False)
    plt.tight_layout(rect=[0, 0, 1, 0.95], pad=1.0)
    if title_prefix:
        plt.suptitle(title_prefix, fontsize=12, weight="bold")
    return fig

# -------------------------
# Main loop: one figure per file
# -------------------------
for idx, fpath in enumerate(files, start=1):
    ext = os.path.splitext(fpath)[1].lower()
    if ext in (".xlsx", ".xls"):
        df = pd.read_excel(fpath)
    elif ext == ".csv":
        df = pd.read_csv(fpath)
    else:
        print(f"[WARN] skipping unsupported file type: {fpath}")
        continue

    if df.empty:
        print(f"[WARN] '{os.path.basename(fpath)}' empty; skipping.")
        continue

    if not {"window", "attempt"}.issubset(df.columns):
        print(f"[WARN] '{os.path.basename(fpath)}' missing 'window'/'attempt'; skipping.")
        continue

    title = f"{os.path.basename(fpath)} (file {idx}/{len(files)})"
    fig = plot_file(df, title_prefix=title)

    if SAVE_PLOTS:
        outpath = os.path.join(plots_dir, f"{os.path.splitext(os.path.basename(fpath))[0]}.png")
        fig.savefig(outpath, dpi=PLOT_DPI, bbox_inches="tight")
        print(f"[SAVED] {outpath}")

    plt.show()
    plt.close(fig)


In [ ]:
#!/usr/bin/env python3
import matplotlib.pyplot as plt
import pandas as pd
import glob
import os

# -------------------------
# Config
# -------------------------
try:
    working_directory  # noqa: F821
except NameError:
    working_directory = "/mnt/e/RStuff/codon_overlap/aa_change_predictions/results/20250823_batch_run_26"

RESULTS_GLOB = os.path.join(working_directory, "*.*")

SAVE_PLOTS = True
SAVE_SUMMARY = True
PLOT_DPI = 150

plots_dir = os.path.join(working_directory, "plots")
if SAVE_PLOTS and not os.path.exists(plots_dir):
    os.makedirs(plots_dir, exist_ok=True)

# -------------------------
# Collect summary per file
# -------------------------
summary = []

for fpath in sorted(glob.glob(RESULTS_GLOB)):
    ext = os.path.splitext(fpath)[1].lower()
    if ext not in (".xlsx", ".xlsm", ".xls", ".csv"):
        continue

    # Load with explicit engine
    try:
        if ext == ".csv":
            df = pd.read_csv(fpath)
        elif ext in (".xlsx", ".xlsm"):
            df = pd.read_excel(fpath, engine="openpyxl")
        elif ext == ".xls":
            df = pd.read_excel(fpath, engine="xlrd")
        else:
            print(f"[WARN] skipping unsupported file type: {fpath}")
            continue
    except Exception as e:
        print(f"[ERROR] Failed to load {fpath}: {e}")
        continue

    if df.empty or not {"ss_score_1", "ss_score_2"} <= set(df.columns):
        print(f"[WARN] '{os.path.basename(fpath)}' missing SS score columns or empty; skipping.")
        continue

    df = df.copy()
    df["ss_avg"] = (df["ss_score_1"] + df["ss_score_2"]) / 2.0

    # Pick row with max avg
    best_row = df.loc[df["ss_avg"].idxmax()]
    summary.append({
        "file": os.path.basename(fpath),
        "ss1": float(best_row["ss_score_1"]),
        "ss2": float(best_row["ss_score_2"]),
        "ss_avg": float(best_row["ss_avg"])
    })

# Convert to DataFrame
summary_df = pd.DataFrame(summary)

# -------------------------
# Save summary table
# -------------------------
if SAVE_SUMMARY and not summary_df.empty:
    summary_outpath = os.path.join(working_directory, "summary_highest_avg_ss.xlsx")
    summary_df.to_excel(summary_outpath, index=False)
    print(f"[SAVED] summary table: {summary_outpath}")

# -------------------------
# Plot summary
# -------------------------
if not summary_df.empty:
    fig, ax = plt.subplots(figsize=(10, 6))

    x = range(len(summary_df))
    ax.plot(x, summary_df["ss_avg"], color="black", marker="s", linestyle="-", label="Avg SS (max per file)")
    ax.plot(x, summary_df["ss1"], color="red", marker="o", linestyle="--", label="SS1 at max avg")
    ax.plot(x, summary_df["ss2"], color="blue", marker="o", linestyle="--", label="SS2 at max avg")

    ax.set_xticks(x)
    ax.set_xticklabels(summary_df["file"], rotation=60, ha="right", fontsize=8)
    ax.set_ylabel("SS Score (%)")
    ax.set_title("Highest Average SS Score (with underlying SS1 and SS2) per File", fontsize=12, weight="bold")
    ax.grid(True, alpha=0.3)
    ax.legend()

    plt.tight_layout()
    plt.ylim(0, 100)
    if SAVE_PLOTS:
        outpath = os.path.join(plots_dir, "summary_highest_avg_ss.png")
        fig.savefig(outpath, dpi=PLOT_DPI, bbox_inches="tight")
        print(f"[SAVED] plot: {outpath}")

    plt.show()


In [ ]:
import seaborn as sns

# -------------------------
# Violin plot (SS1, SS2, Avg SS)
# -------------------------
if not summary_df.empty:
    # Reshape data into long format
    melted = summary_df.melt(
        value_vars=["ss1", "ss2", "ss_avg"],
        var_name="Metric",
        value_name="Score"
    )

    fig, ax = plt.subplots(figsize=(4, 6))
    sns.violinplot(
        data=melted,
        x="Metric", y="Score",
        palette={"ss1": "red", "ss2": "blue", "ss_avg": "gray"},
        inner="box",  # show boxplot inside violins
        cut=0
    )

    ax.set_ylim(0, 100)
    ax.set_title("Distribution of Max-Row SS Scores Across Files", fontsize=12, weight="bold")
    ax.set_ylabel("SS Score (%)")
    ax.set_xlabel("")

    plt.tight_layout()

    if SAVE_PLOTS:
        violin_outpath = os.path.join(plots_dir, "violin_ss_scores.png")
        fig.savefig(violin_outpath, dpi=PLOT_DPI, bbox_inches="tight")
        print(f"[SAVED] violin plot: {violin_outpath}")

    plt.show()


In [ ]:
def impact_analysis_per_file(df: pd.DataFrame, outdir: str, metric_priority=None, max_features=8, random_state=42):
    os.makedirs(outdir, exist_ok=True)
    if metric_priority is None:
        metric_priority = [
            "combined_score",
            "ss_score_1", "ss_score_2",
            "esm1", "esm2",
            "esm_scan_score_1", "esm_scan_score_2",
            "align1", "align2",
            "sub1", "sub2"
        ]

    # pick available metrics in priority order
    available = [m for m in metric_priority if m in df.columns]
    if not available:
        print(f"[IMPACT] no prioritized columns present. available columns: {list(df.columns)}")
        return

    # choose target
    if "combined_score" in df.columns:
        target_col = "combined_score"
    else:
        target_col = available[0]
    print(f"[IMPACT] selected target: {target_col}")

    # choose feature set but EXCLUDE the target to avoid duplicate-column errors
    features = [m for m in available if m != target_col][:max_features]
    if not features:
        print(f"[IMPACT] No features available after removing target '{target_col}'. Nothing to analyze.")
        return

    print(f"[IMPACT] features used: {features}")

    # Prepare dataset (drop rows with NaNs in features or target)
    df_clean = df[features + [target_col]].copy()
    n_before = len(df)
    df_clean = df_clean.dropna()
    n_after = len(df_clean)
    print(f"[IMPACT] rows in original df: {n_before}; rows after dropna on selected cols: {n_after}")

    if df_clean.empty:
        print("[IMPACT] No complete rows after dropna; skipping impact analysis.")
        return

    X = df_clean[features].astype(float)
    y = df_clean[target_col].astype(float)

    # 1) Correlation matrices (Pearson + Spearman)
    try:
        pear = X.join(y).corr(method="pearson")
        spear = X.join(y).corr(method="spearman")
        pear.to_csv(os.path.join(outdir, "correlation_pearson.csv"))
        spear.to_csv(os.path.join(outdir, "correlation_spearman.csv"))
        print(f"[IMPACT] Saved correlation CSVs to {outdir}")
    except Exception as e:
        print(f"[IMPACT] Correlation step failed: {e}")
        return

    # heatmap (Pearson)
    try:
        fig, ax = plt.subplots(figsize=(8, 6))
        im = ax.imshow(pear, interpolation='nearest', cmap='RdBu_r', vmin=-1, vmax=1)
        ax.set_xticks(range(len(pear.columns))); ax.set_xticklabels(pear.columns, rotation=45, ha="right")
        ax.set_yticks(range(len(pear.index))); ax.set_yticklabels(pear.index)
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02)
        ax.set_title("Pearson correlation")
        plt.tight_layout()
        heatpath = os.path.join(outdir, "corr_pearson_heatmap.png")
        fig.savefig(heatpath, dpi=PLOT_DPI, bbox_inches="tight")
        plt.close(fig)
        print(f"[IMPACT] Saved heatmap: {heatpath}")
    except Exception as e:
        print(f"[IMPACT] Heatmap failed: {e}")

    # 2) VIF
    try:
        vif_df = compute_vif(X)
        vif_df.to_csv(os.path.join(outdir, "vif.csv"), index=False)
        print(f"[IMPACT] Saved VIF CSV: {os.path.join(outdir, 'vif.csv')}")
    except Exception as e:
        print(f"[IMPACT] VIF computation failed: {e}")

    # 3) Random Forest Importances
    try:
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=random_state)
        rf = RandomForestRegressor(n_estimators=200, random_state=random_state, n_jobs=-1)
        rf.fit(X_train, y_train)
        importances = pd.DataFrame({"feature": features, "importance": rf.feature_importances_}).sort_values("importance", ascending=False)
        importances.to_csv(os.path.join(outdir, "rf_importances.csv"), index=False)
        print(f"[IMPACT] Saved RF importances: {os.path.join(outdir, 'rf_importances.csv')}")
        # plot
        fig, ax = plt.subplots(figsize=(6.5, 3.5))
        ax.bar(importances["feature"], importances["importance"])
        ax.set_xticklabels(importances["feature"], rotation=45, ha="right")
        ax.set_title(f"RF importances -> predicting {target_col}")
        plt.tight_layout()
        imp_path = os.path.join(outdir, "rf_importances.png")
        fig.savefig(imp_path, dpi=PLOT_DPI, bbox_inches="tight")
        plt.close(fig)
        print(f"[IMPACT] Saved RF importance plot: {imp_path}")
    except Exception as e:
        print(f"[IMPACT] RF importance failed: {e}")

    # 4) Partial Dependence for top features
    try:
        top_feats = importances["feature"].tolist()[:3]
        if top_feats:
            fig, ax = plt.subplots(figsize=(6.5, 3.5 * len(top_feats)))
            PartialDependenceDisplay.from_estimator(rf, X, top_feats, ax=ax)
            plt.suptitle("Partial dependence (RF)")
            plt.tight_layout()
            pdp_path = os.path.join(outdir, "pdp_top_features.png")
            fig.savefig(pdp_path, dpi=PLOT_DPI, bbox_inches="tight")
            plt.close(fig)
            print(f"[IMPACT] Saved PDP: {pdp_path}")
    except Exception as e:
        print(f"[IMPACT] PDP failed: {e}")

    # 5) PCA biplot
    try:
        scaler = StandardScaler()
        Xs = scaler.fit_transform(X.fillna(X.mean()))
        pca = PCA(n_components=min(len(features), 4))
        comps = pca.fit_transform(Xs)
        loadings = pca.components_.T * np.sqrt(pca.explained_variance_)
        load_df = pd.DataFrame(loadings, index=features, columns=[f"PC{i+1}" for i in range(loadings.shape[1])])
        load_df.to_csv(os.path.join(outdir, "pca_loadings.csv"))
        fig, ax = plt.subplots(figsize=(6.5, 5))
        ax.scatter(comps[:, 0], comps[:, 1], s=18, alpha=0.5)
        for i, feat in enumerate(features):
            ax.arrow(0, 0, loadings[i, 0]*3, loadings[i, 1]*3, color='r', head_width=0.05)
            ax.text(loadings[i, 0]*3.2, loadings[i, 1]*3.2, feat, color='r', fontsize=9)
        ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
        ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
        ax.set_title("PCA biplot")
        plt.tight_layout()
        pca_path = os.path.join(outdir, "pca_biplot.png")
        fig.savefig(pca_path, dpi=PLOT_DPI, bbox_inches="tight")
        plt.close(fig)
        print(f"[IMPACT] Saved PCA biplot: {pca_path}")
    except Exception as e:
        print(f"[IMPACT] PCA failed: {e}")

    print(f"[IMPACT] Completed impact analysis outputs in: {outdir}")


In [ ]:
import os
import glob
import pandas as pd

# --- safety defaults in case globals weren't set earlier ---
PLOT_DPI = globals().get("PLOT_DPI", 150)
SAVE_PLOTS = globals().get("SAVE_PLOTS", True)
RESULTS_GLOB = globals().get("RESULTS_GLOB", os.path.join(os.getcwd(), "20250824_length_312_row_*.xlsx"))
plots_dir = globals().get("plots_dir", os.path.join(os.getcwd(), "plots"))
os.makedirs(plots_dir, exist_ok=True)

# Find files
files = sorted(glob.glob(RESULTS_GLOB))
print("TEST: RESULTS_GLOB =", RESULTS_GLOB)
print("TEST: found files =", len(files))
if not files:
    raise RuntimeError("No files found for quick test. Fix RESULTS_GLOB or working_directory first.")

# Read first file
fpath = files[0]
print("TEST: using file:", fpath)
df_test = pd.read_excel(fpath)
print("TEST: df rows,cols =", df_test.shape)
print("TEST: sample head:\n", df_test.head(2).to_string(index=False))

# Ensure helper exists
if "compute_vif" not in globals():
    print("WARNING: compute_vif not found in globals — please define compute_vif before calling impact_analysis_per_file.")
else:
    print("compute_vif found.")

# call the function and catch any exceptions
outdir = os.path.join(plots_dir, "impact", os.path.splitext(os.path.basename(fpath))[0])
try:
    print("TEST: calling impact_analysis_per_file...")
    impact_analysis_per_file(df_test, outdir, max_features=8)
    print("TEST: function returned normally.")
except Exception as e:
    import traceback
    print("TEST: impact_analysis_per_file raised an exception:")
    traceback.print_exc()

# show what was created (if any)
if os.path.exists(outdir):
    print("TEST: files in outdir:", outdir)
    for fn in sorted(os.listdir(outdir))[:50]:
        print("  ", fn)
else:
    print("TEST: outdir does not exist:", outdir)


In [ ]:
working_directory = "/mnt/e/RStuff/codon_overlap/aa_change_predictions/results/20250816_batch_run_7/"
os.makedirs(working_directory, exist_ok=True)
os.chdir(working_directory)

#Option A: use in-memory `all_pairs_records` (if you ran your script in this same session)
#fasta_paths = export_top_n_fastas(top_n=20, out_dir="./folding_top20")

#Option B: read the Excel your script produced
excel = "/mnt/e/RStuff/codon_overlap/aa_change_predictions/results/20250816_batch_run_7/20250816_length_311_row_1.xlsx"
fasta_paths = export_top_n_fastas(top_n=20, excel_path=excel, out_dir="./folding_top20")

#Print results
print("\n".join(str(p) for p in fasta_paths))

In [ ]:
#Code to start the ESM Fold server and fold sequences
#The function prints the pLDDT score

import subprocess, json

# Start the server process (runs in background)
esmfold_proc = subprocess.Popen(
    ["/home/jason/outputdir/esmfold/.venv/bin/python",
     "/home/jason/outputdir/esmfold/run_esmfold_server.py"],
    stdin=subprocess.PIPE,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
    bufsize=1
)

def fold_sequence(seq, num_recycles=None):
    payload = {"sequence": seq}
    if num_recycles is not None:
        payload["num_recycles"] = num_recycles
    esmfold_proc.stdin.write(json.dumps(payload) + "\n")
    esmfold_proc.stdin.flush()
    line = esmfold_proc.stdout.readline()
    return json.loads(line)


In [ ]:
def esm_fold_plddt_output(seq, num_recycles=3):
    """
    Fold a sequence using ESM-Fold and return the mean pLDDT score.
    """
    result = fold_sequence(seq, num_recycles)
    if "plddt_mean" in result:
        return result["plddt_mean"]
    else:
        raise ValueError(f"ESM-Fold did not return pLDDT scores. Got: {result}")

In [ ]:
from Bio.PDB.MMCIFParser import MMCIFParser
import numpy as np

In [ ]:


# def plddt_vector_cif(cif_path):
#     parser = MMCIFParser(QUIET=True)
#     structure = parser.get_structure("model", cif_path)
#     scores = []
#     for res in structure.get_residues():
#         if res.id[0] != " ":
#             continue
#         atom = next(res.get_atoms())
#         scores.append(atom.bfactor)   # pLDDT is stored here
#     return np.array(scores, dtype=float)

# # File paths (use Unix-style /mnt/e/ for WSL or Linux subsystems)
# cif_A = "/mnt/e/RStuff/codon_overlap/aa_change_predictions/egfp_ampr/egfp_native_fold/fold_egfp_native_fold_model_0.cif"
# cif_B = "/mnt/e/RStuff/codon_overlap/aa_change_predictions/egfp_ampr/test_folds/egfp/fold_egfp_model_0.cif"

# # Extract pLDDT vectors
# plddt_A = plddt_vector_cif(cif_A)
# plddt_B = plddt_vector_cif(cif_B)

# # Truncate to shortest length for per-residue comparison
# N = min(len(plddt_A), len(plddt_B))
# plddt_A, plddt_B = plddt_A[:N], plddt_B[:N]
# delta = plddt_B - plddt_A

# # Print summary statistics
# print(f"Native Fold Mean pLDDT: {plddt_A.mean():.2f} (min: {plddt_A.min():.1f}, max: {plddt_A.max():.1f})")
# print(f"Test Fold Mean pLDDT:   {plddt_B.mean():.2f} (min: {plddt_B.min():.1f}, max: {plddt_B.max():.1f})")
# print(f"Mean ΔpLDDT (Test – Native): {delta.mean():+.2f}")

# # Plot comparison if you want visualization
# plt.plot(plddt_A, label="Native Fold")
# plt.plot(plddt_B, label="Test Fold")
# plt.ylabel("pLDDT")
# plt.xlabel("Residue")
# plt.legend()
# plt.title("AlphaFold pLDDT Comparison")
# plt.show()

In [ ]:
from Bio.PDB import PDBParser, MMCIFParser, Superimposer
import numpy as np
import os

def calculate_rmsd_with_matching(
    structure1_path, structure2_path, threshold=3.0, terminal_residues=None
):
    """
    Match residues between two structures, align them, and calculate RMSD.
    Parameters:
        structure1_path (str): Path to the first structure file (PDB or CIF).
        structure2_path (str): Path to the second structure file (PDB or CIF).
        threshold (float): Pruning distance threshold (default: 3.0 Å).
        terminal_residues (int): Number of terminal residues to include in the RMSD calculation (default: None, include all).
    Returns:
        dict: {
            "terminal": (rmsd_pruned, rmsd_all, n_pruned, n_total),
            "full": (rmsd_pruned_full, rmsd_all_full, n_pruned_full, n_total_full)
        }
    """

    def parse_structure(file_path, structure_id):
        file_ext = os.path.splitext(file_path)[-1].lower()
        if file_ext == ".pdb":
            parser = PDBParser(QUIET=True)
        elif file_ext == ".cif":
            parser = MMCIFParser(QUIET=True)
        else:
            raise ValueError(f"Unsupported file format: {file_ext}")
        return parser.get_structure(structure_id, file_path)

    structure1 = parse_structure(structure1_path, "structure1")
    structure2 = parse_structure(structure2_path, "structure2")

    residues1 = [residue for residue in structure1.get_residues() if residue.has_id("CA")]
    residues2 = [residue for residue in structure2.get_residues() if residue.has_id("CA")]

    if len(residues1) != len(residues2):
        raise ValueError("Mismatch in the number of residues between the two structures.")

    # Superimpose using all residues
    atoms1_full = [residue["CA"] for residue in residues1]
    atoms2_full = [residue["CA"] for residue in residues2]

    sup = Superimposer()
    sup.set_atoms(atoms1_full, atoms2_full)
    sup.apply(structure2.get_atoms())

    # ----- Terminal region -----
    if terminal_residues is not None and terminal_residues > 0:
        eval_residues1 = residues1[-terminal_residues:]
        eval_residues2 = residues2[-terminal_residues:]
    else:
        eval_residues1 = residues1
        eval_residues2 = residues2

    eval_atoms1 = [residue["CA"] for residue in eval_residues1]
    eval_atoms2 = [residue["CA"] for residue in eval_residues2]

    coords1 = np.array([atom.get_coord() for atom in eval_atoms1])
    coords2 = np.array([atom.get_coord() for atom in eval_atoms2])

    distances = np.sqrt(np.sum((coords1 - coords2) ** 2, axis=1))
    rmsd_all = np.sqrt(np.mean(distances ** 2))

    pruned_indices = distances <= threshold
    coords1_pruned = coords1[pruned_indices]
    coords2_pruned = coords2[pruned_indices]

    if len(coords1_pruned) == 0:
        raise ValueError("No atom pairs remain after pruning (terminal region). Increase the threshold.")

    rmsd_pruned = np.sqrt(np.mean(np.sum((coords1_pruned - coords2_pruned) ** 2, axis=1)))

    # ----- Whole structure -----
    coords1_full = np.array([atom.get_coord() for atom in atoms1_full])
    coords2_full = np.array([atom.get_coord() for atom in atoms2_full])
    distances_full = np.sqrt(np.sum((coords1_full - coords2_full) ** 2, axis=1))
    rmsd_all_full = np.sqrt(np.mean(distances_full ** 2))

    pruned_indices_full = distances_full <= threshold
    coords1_pruned_full = coords1_full[pruned_indices_full]
    coords2_pruned_full = coords2_full[pruned_indices_full]

    if len(coords1_pruned_full) == 0:
        raise ValueError("No atom pairs remain after pruning (full structure). Increase the threshold.")

    rmsd_pruned_full = np.sqrt(np.mean(np.sum((coords1_pruned_full - coords2_pruned_full) ** 2, axis=1)))

    return {
        "terminal": (rmsd_pruned, rmsd_all, len(coords1_pruned), len(coords1)),
        "full": (rmsd_pruned_full, rmsd_all_full, len(coords1_pruned_full), len(coords1_full))
    }

# EGFP
structure1_path = "/mnt/e/RStuff/codon_overlap/aa_change_predictions/egfp_ampr/egfp_native_fold/fold_egfp_native_fold_model_0.cif"
structure2_path = "/mnt/e/RStuff/codon_overlap/aa_change_predictions/egfp_ampr/test_folds/alphafold_job_egfp_3/fold_alphafold_job_egfp_3_model_0.cif"

results = calculate_rmsd_with_matching(structure1_path, structure2_path, terminal_residues=95)

print("[EGFP | C-terminal region]:")
print(f"  RMSD (pruned pairs): {results['terminal'][0]:.3f} Å")
print(f"  RMSD (all pairs): {results['terminal'][1]:.3f} Å")
print(f"  Pruned pairs: {results['terminal'][2]}/{results['terminal'][3]}")
print("[EGFP | Whole structure]:")
print(f"  RMSD (pruned pairs): {results['full'][0]:.3f} Å")
print(f"  RMSD (all pairs): {results['full'][1]:.3f} Å")
print(f"  Pruned pairs: {results['full'][2]}/{results['full'][3]}")

# AmpR
structure1_path = "/mnt/e/RStuff/codon_overlap/aa_change_predictions/egfp_ampr/ampr_native_fold/fold_ampr_native_fold_model_0.cif"
structure2_path = "/mnt/e/RStuff/codon_overlap/aa_change_predictions/egfp_ampr/test_folds/alphafold_job_ampr_3/fold_alphafold_job_ampr_3_model_0.cif"

results = calculate_rmsd_with_matching(structure1_path, structure2_path, terminal_residues=95)

print("\n[AmpR | C-terminal region]:")
print(f"  RMSD (pruned pairs): {results['terminal'][0]:.3f} Å")
print(f"  RMSD (all pairs): {results['terminal'][1]:.3f} Å")
print(f"  Pruned pairs: {results['terminal'][2]}/{results['terminal'][3]}")
print("[AmpR | Whole structure]:")
print(f"  RMSD (pruned pairs): {results['full'][0]:.3f} Å")
print(f"  RMSD (all pairs): {results['full'][1]:.3f} Å")
print(f"  Pruned pairs: {results['full'][2]}/{results['full'][3]}")
